# Notebook 07: System Evaluation

**Author:** Anthony Amit Biswas

## What this notebook does

Evaluates the medSpaCy complication-extraction system against the reviewed annotation corpus using exact-span, relaxed-overlap, category, assertion, and temporality metrics.


In [ ]:
# 1. Environment Setup and Evaluation Configuration

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import warnings

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

warnings.filterwarnings("ignore")

print("-" * 80)
print("NOTEBOOK 07 — CLINICAL NLP SYSTEM EVALUATION")
print("-" * 80)


# 1.1 Mount Google Drive


drive.mount("/content/drive")



# 1.2 Project directories


PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Dissertation"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "outputs"
)

ANNOTATION_DIRECTORY = (
    OUTPUT_DIRECTORY
    / "gold_standard_annotation"
)

MEDSPACY_DIRECTORY = (
    OUTPUT_DIRECTORY
    / "medspacy"
)

EVALUATION_DIRECTORY = (
    OUTPUT_DIRECTORY
    / "evaluation_lexicon_v2"
)

EVALUATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


# 1.3 Input files

GOLD_STANDARD_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_notebook06_complete_70_model_.xlsx"
)

CANDIDATE_MENTIONS_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_70_note_medspacy_candidates_lexicon_v2_aligned.csv.gz"
)

SAMPLE_WITH_TEXT_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_70_note_sample_with_text_lexicon_v2_aligned.csv.gz"
)

NOTEBOOK_06_REPORT_FILE = (
    ANNOTATION_DIRECTORY
    / "notebook_06_full_corpus_completion_report.json"
)


# 1.4 Notebook 07 output files

EVALUATION_CONFIGURATION_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_evaluation_configuration.json"
)

GOLD_STANDARD_MENTIONS_FILE = (
    EVALUATION_DIRECTORY
    / "gold_standard_mentions.csv"
)

SYSTEM_PREDICTIONS_FILE = (
    EVALUATION_DIRECTORY
    / "system_predictions.csv"
)

MATCHED_MENTIONS_FILE = (
    EVALUATION_DIRECTORY
    / "matched_mentions.csv"
)

FALSE_POSITIVES_FILE = (
    EVALUATION_DIRECTORY
    / "false_positive_mentions.csv"
)

FALSE_NEGATIVES_FILE = (
    EVALUATION_DIRECTORY
    / "false_negative_mentions.csv"
)

OVERALL_METRICS_FILE = (
    EVALUATION_DIRECTORY
    / "overall_evaluation_metrics.csv"
)

CATEGORY_METRICS_FILE = (
    EVALUATION_DIRECTORY
    / "category_evaluation_metrics.csv"
)

EVALUATION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_evaluation_report.json"
)


# 1.5 Matching configuration


EVALUATION_CONFIGURATION = {
    "notebook": "07",
    "notebook_title":
        "Clinical NLP System Evaluation",

    "gold_standard_workbook":
        str(GOLD_STANDARD_WORKBOOK_FILE),

    "candidate_mentions_file":
        str(CANDIDATE_MENTIONS_FILE),

    "sample_with_text_file":
        str(SAMPLE_WITH_TEXT_FILE),

    "expected_notes": 70,

    "matching_strategies": {
        "exact_span": {
            "description":
                "Prediction and gold-standard mention must have "
                "identical note ID, start character and end character."
        },

        "relaxed_span": {
            "description":
                "Prediction and gold-standard mention must overlap "
                "within the same note."
        },

        "category_aware": {
            "description":
                "The predicted complication category must match "
                "the gold-standard category."
        },
    },

    "primary_evaluation": {
        "span_matching": "relaxed_span",
        "category_matching": True,
    },

    "secondary_evaluation": {
        "span_matching": "exact_span",
        "category_matching": True,
    },

    "context_attributes": [
        "is_negated",
        "is_historical",
        "is_family",
        "is_uncertain",
        "is_hypothetical",
        "is_current_affirmed",
    ],

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
}



# 1.6 Validate required inputs


required_input_files = {
    "Gold-standard workbook":
        GOLD_STANDARD_WORKBOOK_FILE,

    "Candidate mentions":
        CANDIDATE_MENTIONS_FILE,

    "Sample notes with text":
        SAMPLE_WITH_TEXT_FILE,
}

input_validation_rows = []

for description, file_path in required_input_files.items():

    exists = file_path.is_file()

    input_validation_rows.append(
        {
            "Input": description,
            "Path": str(file_path),
            "Exists": exists,
            "Size_MB": (
                round(
                    file_path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if exists
                else np.nan
            ),
        }
    )

input_validation_df = pd.DataFrame(
    input_validation_rows
)

display(input_validation_df)

missing_input_files = [
    description
    for description, file_path
    in required_input_files.items()
    if not file_path.is_file()
]

if missing_input_files:
    raise FileNotFoundError(
        "\nNotebook 07 cannot start because these required "
        "input files were not found:\n- "
        + "\n- ".join(missing_input_files)
        + "\n\nCheck that the filenames and Google Drive "
        "locations exactly match the paths above."
    )



# 1.7 Record file hashes

def calculate_sha256(file_path):
    """Return the SHA-256 checksum for a file."""

    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as file_object:

        for block in iter(
            lambda: file_object.read(1024 * 1024),
            b"",
        ):
            sha256_hash.update(block)

    return sha256_hash.hexdigest()


input_hashes = {
    description: calculate_sha256(file_path)
    for description, file_path
    in required_input_files.items()
}


# 1.8 Save configuration


EVALUATION_CONFIGURATION["input_sha256"] = (
    input_hashes
)

EVALUATION_CONFIGURATION["software"] = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
}

with open(
    EVALUATION_CONFIGURATION_FILE,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        EVALUATION_CONFIGURATION,
        configuration_file,
        indent=2,
        ensure_ascii=False,
    )


# 1.9 Final setup status

print("\n" + "-" * 80)
print("ENVIRONMENT SETUP COMPLETE")
print("-" * 80)

print(
    f"\nGold-standard workbook:\n"
    f"{GOLD_STANDARD_WORKBOOK_FILE}"
)

print(
    f"\nCandidate mentions:\n"
    f"{CANDIDATE_MENTIONS_FILE}"
)

print(
    f"\nEvaluation outputs:\n"
    f"{EVALUATION_DIRECTORY}"
)

print(
    f"\nConfiguration saved to:\n"
    f"{EVALUATION_CONFIGURATION_FILE}"
)

print("\nAll required inputs were found.")
print("Ready to load and validate the annotation workbook.")

## 2. Loading and inspecting the completed annotation workbook

This section loads the completed Notebook 06 workbook, inspects its worksheets and validates the workbook structure before constructing the final gold-standard mention table.

In [ ]:
# 2. Load and Inspect the Completed Annotation Workbook


print("-" * 80)
print("STEP 2 — LOAD AND INSPECT ANNOTATION WORKBOOK")
print("-" * 80)


# 2.1 Load workbook metadata


excel_file = pd.ExcelFile(
    GOLD_STANDARD_WORKBOOK_FILE
)

workbook_sheet_names = excel_file.sheet_names

print("\nWorkbook sheets:")

for index, sheet_name in enumerate(
    workbook_sheet_names,
    start=1,
):
    print(f"{index}. {sheet_name}")


# 2.2 Load every worksheet

workbook_sheets = {}

for sheet_name in workbook_sheet_names:

    sheet_df = pd.read_excel(
        GOLD_STANDARD_WORKBOOK_FILE,
        sheet_name=sheet_name,
    )

    workbook_sheets[sheet_name] = sheet_df



# 2.3 Creating workbook inventory

workbook_inventory_rows = []

for sheet_name, sheet_df in workbook_sheets.items():

    workbook_inventory_rows.append(
        {
            "Sheet": sheet_name,
            "Rows": len(sheet_df),
            "Columns": len(sheet_df.columns),
            "Column_names": " | ".join(
                str(column)
                for column in sheet_df.columns
            ),
        }
    )

workbook_inventory_df = pd.DataFrame(
    workbook_inventory_rows
)

print("\nWorkbook inventory:")
display(workbook_inventory_df)


# 2.4 Displaying the first rows of each worksheet


for sheet_name, sheet_df in workbook_sheets.items():

    print("\n" + "-" * 80)
    print(f"SHEET: {sheet_name}")
    print("-" * 80)

    print(
        f"Shape: {sheet_df.shape[0]} rows × "
        f"{sheet_df.shape[1]} columns"
    )

    display(
        sheet_df.head(5)
    )


# 2.5 Identifying likely annotation worksheet


annotation_keywords = {
    "note_id",
    "candidate",
    "mention",
    "span",
    "start",
    "end",
    "label",
    "decision",
    "annotation",
    "complication",
    "category",
}

annotation_sheet_scores = []

for sheet_name, sheet_df in workbook_sheets.items():

    normalised_columns = {
        str(column).strip().lower()
        for column in sheet_df.columns
    }

    matched_keywords = sorted(
        keyword
        for keyword in annotation_keywords
        if any(
            keyword in column_name
            for column_name in normalised_columns
        )
    )

    annotation_sheet_scores.append(
        {
            "Sheet": sheet_name,
            "Rows": len(sheet_df),
            "Matched_keywords":
                ", ".join(matched_keywords),
            "Keyword_score":
                len(matched_keywords),
        }
    )

annotation_sheet_scores_df = (
    pd.DataFrame(annotation_sheet_scores)
    .sort_values(
        by=[
            "Keyword_score",
            "Rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print("\nLikely annotation worksheet ranking:")
display(annotation_sheet_scores_df)


# 2.6 Selecting likely annotation worksheet

if annotation_sheet_scores_df.empty:
    raise ValueError(
        "No worksheets were found in the workbook."
    )

LIKELY_ANNOTATION_SHEET = "Mention_Annotations"

annotation_df = workbook_sheets[
    LIKELY_ANNOTATION_SHEET
].copy()

print(
    "\nGold-standard annotation worksheet selected:"
)
print(LIKELY_ANNOTATION_SHEET)

print(
    f"\nSelected worksheet shape: "
    f"{annotation_df.shape[0]} rows × "
    f"{annotation_df.shape[1]} columns"
)

annotation_df = (
    workbook_sheets[
        LIKELY_ANNOTATION_SHEET
    ]
    .copy()
)

print(
    "\nLikely annotation worksheet selected:"
)
print(LIKELY_ANNOTATION_SHEET)

print(
    f"\nSelected worksheet shape: "
    f"{annotation_df.shape[0]} rows × "
    f"{annotation_df.shape[1]} columns"
)


# 2.7 Normalising column names for inspection


def normalise_column_name(column_name):
    """
    Convert a column name to a consistent snake_case format.
    """

    normalised = (
        str(column_name)
        .strip()
        .lower()
    )

    replacements = {
        " ": "_",
        "-": "_",
        "/": "_",
        "\\": "_",
        "(": "",
        ")": "",
        "[": "",
        "]": "",
        ":": "",
        ".": "_",
    }

    for old_value, new_value in replacements.items():
        normalised = normalised.replace(
            old_value,
            new_value,
        )

    while "__" in normalised:
        normalised = normalised.replace(
            "__",
            "_",
        )

    return normalised.strip("_")


original_to_normalised_columns = {
    column: normalise_column_name(column)
    for column in annotation_df.columns
}

column_mapping_df = pd.DataFrame(
    {
        "Original_column":
            list(
                original_to_normalised_columns.keys()
            ),

        "Normalised_column":
            list(
                original_to_normalised_columns.values()
            ),
    }
)

print("\nColumn-name mapping:")
display(column_mapping_df)


# 2.8 Checking for duplicate normalised column names


normalised_column_values = list(
    original_to_normalised_columns.values()
)

duplicate_normalised_columns = sorted(
    {
        column
        for column in normalised_column_values
        if normalised_column_values.count(column) > 1
    }
)

if duplicate_normalised_columns:

    print(
        "\nWarning: duplicate normalised column names "
        "were detected:"
    )

    for column in duplicate_normalised_columns:
        print(f"- {column}")

else:
    print(
        "\nNo duplicate normalised column names "
        "were detected."
    )



# 2.9 Loading supporting CSV files


candidate_mentions_df = pd.read_csv(
    CANDIDATE_MENTIONS_FILE
)

sample_with_text_df = pd.read_csv(
    SAMPLE_WITH_TEXT_FILE
)

print("\nSupporting input shapes:")

supporting_inputs_df = pd.DataFrame(
    [
        {
            "Dataset": "Candidate mentions",
            "Rows": len(candidate_mentions_df),
            "Columns": len(
                candidate_mentions_df.columns
            ),
        },
        {
            "Dataset": "Sample notes with text",
            "Rows": len(sample_with_text_df),
            "Columns": len(
                sample_with_text_df.columns
            ),
        },
    ]
)

display(supporting_inputs_df)



# 2.10 Displaying supporting columns


print("\nCandidate mention columns:")
print(
    candidate_mentions_df.columns.tolist()
)

print("\nSample-with-text columns:")
print(
    sample_with_text_df.columns.tolist()
)


# 2.11 Basic note-count checks


def identify_first_matching_column(
    dataframe,
    possible_names,
):
    """
    Return the first column matching one of the possible
    normalised names.
    """

    normalised_lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for possible_name in possible_names:

        if possible_name in normalised_lookup:
            return normalised_lookup[possible_name]

    return None


possible_note_id_columns = [
    "note_id",
    "noteid",
    "row_id",
    "document_id",
    "doc_id",
]

annotation_note_id_column = (
    identify_first_matching_column(
        annotation_df,
        possible_note_id_columns,
    )
)

candidate_note_id_column = (
    identify_first_matching_column(
        candidate_mentions_df,
        possible_note_id_columns,
    )
)

sample_note_id_column = (
    identify_first_matching_column(
        sample_with_text_df,
        possible_note_id_columns,
    )
)


note_count_rows = []

for dataset_name, dataframe, note_id_column in [
    (
        "Annotation worksheet",
        annotation_df,
        annotation_note_id_column,
    ),
    (
        "Candidate mentions",
        candidate_mentions_df,
        candidate_note_id_column,
    ),
    (
        "Sample notes with text",
        sample_with_text_df,
        sample_note_id_column,
    ),
]:

    note_count_rows.append(
        {
            "Dataset": dataset_name,
            "Detected_note_ID_column":
                note_id_column,
            "Unique_note_count": (
                dataframe[
                    note_id_column
                ].nunique(dropna=True)
                if note_id_column is not None
                else np.nan
            ),
        }
    )

note_count_df = pd.DataFrame(
    note_count_rows
)

print("\nDetected note-ID columns and counts:")
display(note_count_df)



# 2.12 Saving workbook inspection report


WORKBOOK_INSPECTION_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_workbook_inspection.json"
)

workbook_inspection_report = {
    "workbook_path":
        str(GOLD_STANDARD_WORKBOOK_FILE),

    "sheet_names":
        workbook_sheet_names,

    "selected_annotation_sheet":
        LIKELY_ANNOTATION_SHEET,

    "selected_annotation_shape": {
        "rows": int(
            annotation_df.shape[0]
        ),
        "columns": int(
            annotation_df.shape[1]
        ),
    },

    "selected_annotation_columns": [
        str(column)
        for column in annotation_df.columns
    ],

    "normalised_column_mapping": {
        str(original): normalised
        for original, normalised
        in original_to_normalised_columns.items()
    },

    "detected_note_id_columns": {
        "annotation":
            annotation_note_id_column,

        "candidate_mentions":
            candidate_note_id_column,

        "sample_with_text":
            sample_note_id_column,
    },

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    WORKBOOK_INSPECTION_FILE,
    "w",
    encoding="utf-8",
) as inspection_file:

    json.dump(
        workbook_inspection_report,
        inspection_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# 2.13 Completion status


print("\n" + "-" * 80)
print("STEP 2 COMPLETE")
print("-" * 80)

print(
    f"\nSelected annotation worksheet: "
    f"{LIKELY_ANNOTATION_SHEET}"
)

print(
    f"Inspection report saved to:\n"
    f"{WORKBOOK_INSPECTION_FILE}"
)

print(
    "\nThe workbook has been loaded successfully."
)

print(
    "Review the displayed sheet names, worksheet ranking, "
    "column mapping and note counts before proceeding."
)

## 3. Constructing the gold-standard and system-prediction datasets

This section converts the completed `Mention_Annotations` worksheet into a validated gold-standard mention table and converts the original medSpaCy candidate output into a standardised prediction table.

Only annotations marked as clinically relevant complications are retained as positive gold-standard mentions. Character offsets are checked against the original discharge-summary text before evaluation.

In [ ]:
# 3. Construct Gold-Standard and System-Prediction Tables

print("-" * 80)
print("STEP 3 — CONSTRUCT GOLD STANDARD AND SYSTEM PREDICTIONS")
print("-" * 80)



# 3.1 Retrieving the required workbook worksheets explicitly


gold_source_df = workbook_sheets[
    "Mention_Annotations"
].copy()

note_register_df = workbook_sheets[
    "Note_Register"
].copy()

full_text_review_df = workbook_sheets[
    "Full_Text_Review"
].copy()

candidate_review_df = workbook_sheets[
    "Candidate_Review"
].copy()



# 3.2 Standardising text and categorical values


def clean_string_series(series):
    """
    Convert a pandas Series to stripped nullable strings.
    Empty strings are converted to pandas missing values.
    """

    cleaned = (
        series
        .astype("string")
        .str.strip()
    )

    return cleaned.replace("", pd.NA)


def upper_string_series(series):
    """
    Standardise a categorical text Series to uppercase.
    """

    return (
        clean_string_series(series)
        .str.upper()
    )


def safe_integer_series(series):
    """
    Convert a Series to pandas nullable integer values.
    """

    return pd.to_numeric(
        series,
        errors="coerce",
    ).astype("Int64")



# 3.3 Validating the 70-note corpus register


required_note_register_columns = {
    "annotation_note_number",
    "note_id",
    "sampling_group",
    "annotation_status",
    "original_note_text",
}

missing_note_register_columns = (
    required_note_register_columns
    - set(note_register_df.columns)
)

if missing_note_register_columns:
    raise ValueError(
        "Note_Register is missing required columns: "
        + ", ".join(
            sorted(missing_note_register_columns)
        )
    )


note_register_df["note_id"] = clean_string_series(
    note_register_df["note_id"]
)

note_register_df["annotation_status"] = upper_string_series(
    note_register_df["annotation_status"]
)

note_register_df["sampling_group"] = clean_string_series(
    note_register_df["sampling_group"]
)

note_register_df["original_note_text"] = (
    note_register_df["original_note_text"]
    .fillna("")
    .astype(str)
)

registered_note_count = (
    note_register_df["note_id"]
    .nunique(dropna=True)
)

if registered_note_count != 70:
    raise ValueError(
        f"Expected 70 registered notes, but found "
        f"{registered_note_count}."
    )

duplicate_registered_note_ids = (
    note_register_df.loc[
        note_register_df["note_id"].duplicated(
            keep=False
        ),
        "note_id",
    ]
    .dropna()
    .unique()
    .tolist()
)

if duplicate_registered_note_ids:
    raise ValueError(
        "Duplicate note IDs were found in Note_Register: "
        + ", ".join(
            map(str, duplicate_registered_note_ids)
        )
    )

incomplete_registered_notes = note_register_df.loc[
    note_register_df["annotation_status"]
    != "COMPLETED"
].copy()

if not incomplete_registered_notes.empty:
    raise ValueError(
        f"{len(incomplete_registered_notes)} notes are not "
        "marked COMPLETED in Note_Register."
    )



# 3.4 Validating completion of full-text review


required_full_review_columns = {
    "note_id",
    "candidate_review_complete",
    "full_original_text_reviewed",
    "note_annotation_complete",
}

missing_full_review_columns = (
    required_full_review_columns
    - set(full_text_review_df.columns)
)

if missing_full_review_columns:
    raise ValueError(
        "Full_Text_Review is missing required columns: "
        + ", ".join(
            sorted(missing_full_review_columns)
        )
    )

for column in [
    "candidate_review_complete",
    "full_original_text_reviewed",
    "note_annotation_complete",
]:
    full_text_review_df[column] = upper_string_series(
        full_text_review_df[column]
    )

full_text_review_df["note_id"] = clean_string_series(
    full_text_review_df["note_id"]
)

full_review_note_count = (
    full_text_review_df["note_id"]
    .nunique(dropna=True)
)

if full_review_note_count != 70:
    raise ValueError(
        f"Expected 70 notes in Full_Text_Review, but found "
        f"{full_review_note_count}."
    )

incomplete_full_reviews = full_text_review_df.loc[
    (
        full_text_review_df[
            "candidate_review_complete"
        ] != "YES"
    )
    |
    (
        full_text_review_df[
            "full_original_text_reviewed"
        ] != "YES"
    )
    |
    (
        full_text_review_df[
            "note_annotation_complete"
        ] != "YES"
    )
].copy()

if not incomplete_full_reviews.empty:
    raise ValueError(
        f"{len(incomplete_full_reviews)} notes have incomplete "
        "review status in Full_Text_Review."
    )



# 3.5 Validating gold-standard source columns


required_gold_columns = {
    "annotation_id",
    "annotation_note_number",
    "note_id",
    "mention_text",
    "complication_category",
    "relevance",
    "assertion",
    "temporality",
    "start_char",
    "end_char",
    "supporting_sentence",
    "annotator_confidence",
    "review_required",
}

missing_gold_columns = (
    required_gold_columns
    - set(gold_source_df.columns)
)

if missing_gold_columns:
    raise ValueError(
        "Mention_Annotations is missing required columns: "
        + ", ".join(
            sorted(missing_gold_columns)
        )
    )



# 3.6 Standardising the gold-standard source table


gold_source_df["annotation_id"] = clean_string_series(
    gold_source_df["annotation_id"]
)

gold_source_df["note_id"] = clean_string_series(
    gold_source_df["note_id"]
)

gold_source_df["mention_text"] = clean_string_series(
    gold_source_df["mention_text"]
)

gold_source_df["complication_category"] = upper_string_series(
    gold_source_df["complication_category"]
)

gold_source_df["relevance"] = upper_string_series(
    gold_source_df["relevance"]
)

gold_source_df["assertion"] = upper_string_series(
    gold_source_df["assertion"]
)

gold_source_df["temporality"] = upper_string_series(
    gold_source_df["temporality"]
)

gold_source_df["annotator_confidence"] = upper_string_series(
    gold_source_df["annotator_confidence"]
)

gold_source_df["review_required"] = upper_string_series(
    gold_source_df["review_required"]
)

gold_source_df["start_char"] = safe_integer_series(
    gold_source_df["start_char"]
)

gold_source_df["end_char"] = safe_integer_series(
    gold_source_df["end_char"]
)


# 3.7 Retaining positive clinical-complication annotations


gold_standard_df = gold_source_df.loc[
    gold_source_df["relevance"]
    == "CLINICAL_COMPLICATION"
].copy()

excluded_gold_rows_df = gold_source_df.loc[
    gold_source_df["relevance"]
    != "CLINICAL_COMPLICATION"
].copy()

gold_standard_df = gold_standard_df[
    [
        "annotation_id",
        "annotation_note_number",
        "note_id",
        "mention_text",
        "complication_category",
        "relevance",
        "assertion",
        "temporality",
        "start_char",
        "end_char",
        "supporting_sentence",
        "annotator_confidence",
        "review_required",
        "annotator_notes",
    ]
].copy()



# 3.8 Attaching note-level metadata and original text


note_metadata_columns = [
    "annotation_note_number",
    "note_id",
    "subject_id",
    "hadm_id",
    "sampling_group",
    "enrichment_category",
    "word_count",
    "original_note_text",
]

gold_standard_df = gold_standard_df.merge(
    note_register_df[note_metadata_columns],
    on=[
        "annotation_note_number",
        "note_id",
    ],
    how="left",
    validate="many_to_one",
)



# 3.9 Validating gold-standard identifiers and offsets


gold_validation_rows = []

for row_index, row in gold_standard_df.iterrows():

    note_text = (
        row["original_note_text"]
        if isinstance(
            row["original_note_text"],
            str,
        )
        else ""
    )

    start_char = row["start_char"]
    end_char = row["end_char"]
    mention_text = row["mention_text"]

    valid_offsets = (
        pd.notna(start_char)
        and pd.notna(end_char)
        and int(start_char) >= 0
        and int(end_char) > int(start_char)
        and int(end_char) <= len(note_text)
    )

    extracted_text = None
    exact_text_match = False
    case_insensitive_match = False

    if valid_offsets:

        extracted_text = note_text[
            int(start_char):int(end_char)
        ]

        exact_text_match = (
            extracted_text == str(mention_text)
        )

        case_insensitive_match = (
            extracted_text.strip().casefold()
            == str(mention_text).strip().casefold()
        )

    gold_validation_rows.append(
        {
            "row_index": row_index,
            "annotation_id":
                row["annotation_id"],
            "note_id":
                row["note_id"],
            "valid_offsets":
                valid_offsets,
            "offset_text":
                extracted_text,
            "annotation_text":
                mention_text,
            "exact_text_match":
                exact_text_match,
            "case_insensitive_match":
                case_insensitive_match,
        }
    )

gold_span_validation_df = pd.DataFrame(
    gold_validation_rows
)

gold_standard_df = gold_standard_df.merge(
    gold_span_validation_df[
        [
            "annotation_id",
            "valid_offsets",
            "offset_text",
            "exact_text_match",
            "case_insensitive_match",
        ]
    ],
    on="annotation_id",
    how="left",
    validate="one_to_one",
)



# 3.10 Checking gold-standard integrity


gold_missing_required_values = (
    gold_standard_df[
        [
            "annotation_id",
            "note_id",
            "mention_text",
            "complication_category",
            "assertion",
            "temporality",
            "start_char",
            "end_char",
        ]
    ]
    .isna()
    .sum()
)

duplicate_annotation_ids = (
    gold_standard_df.loc[
        gold_standard_df[
            "annotation_id"
        ].duplicated(
            keep=False
        ),
        "annotation_id",
    ]
    .dropna()
    .unique()
    .tolist()
)

invalid_gold_offset_count = int(
    (~gold_standard_df["valid_offsets"]).sum()
)

gold_exact_text_mismatch_count = int(
    (
        gold_standard_df["valid_offsets"]
        &
        ~gold_standard_df["exact_text_match"]
    ).sum()
)

gold_case_insensitive_mismatch_count = int(
    (
        gold_standard_df["valid_offsets"]
        &
        ~gold_standard_df[
            "case_insensitive_match"
        ]
    ).sum()
)



# 3.11 Standardising system-prediction table


required_prediction_columns = {
    "candidate_id",
    "note_id",
    "entity_text",
    "category",
    "start_char",
    "end_char",
    "sentence_text",
    "is_negated",
    "is_historical",
    "is_family",
    "is_uncertain",
    "is_hypothetical",
    "is_current_affirmed",
    "annotation_note_number",
    "sampling_group",
    "predicted_assertion",
    "predicted_temporality",
}

missing_prediction_columns = (
    required_prediction_columns
    - set(candidate_mentions_df.columns)
)

if missing_prediction_columns:
    raise ValueError(
        "Candidate prediction file is missing required columns: "
        + ", ".join(
            sorted(missing_prediction_columns)
        )
    )


system_predictions_df = candidate_mentions_df.copy()

system_predictions_df["candidate_id"] = clean_string_series(
    system_predictions_df["candidate_id"]
)

system_predictions_df["note_id"] = clean_string_series(
    system_predictions_df["note_id"]
)

system_predictions_df["mention_text"] = clean_string_series(
    system_predictions_df["entity_text"]
)

system_predictions_df["complication_category"] = (
    upper_string_series(
        system_predictions_df["category"]
    )
)

system_predictions_df["predicted_assertion"] = (
    upper_string_series(
        system_predictions_df["predicted_assertion"]
    )
)

system_predictions_df["predicted_temporality"] = (
    upper_string_series(
        system_predictions_df["predicted_temporality"]
    )
)

system_predictions_df["start_char"] = safe_integer_series(
    system_predictions_df["start_char"]
)

system_predictions_df["end_char"] = safe_integer_series(
    system_predictions_df["end_char"]
)



# 3.12 Standardising Boolean context flags


boolean_prediction_columns = [
    "is_negated",
    "is_historical",
    "is_family",
    "is_uncertain",
    "is_hypothetical",
    "is_current_affirmed",
]

TRUE_VALUES = {
    True,
    1,
    "1",
    "TRUE",
    "T",
    "YES",
    "Y",
}

FALSE_VALUES = {
    False,
    0,
    "0",
    "FALSE",
    "F",
    "NO",
    "N",
}


def normalise_boolean_value(value):
    """
    Convert common Boolean representations to True, False
    or pandas missing value.
    """

    if pd.isna(value):
        return pd.NA

    if isinstance(value, str):
        value = value.strip().upper()

    if value in TRUE_VALUES:
        return True

    if value in FALSE_VALUES:
        return False

    return pd.NA


for column in boolean_prediction_columns:

    system_predictions_df[column] = (
        system_predictions_df[column]
        .apply(normalise_boolean_value)
        .astype("boolean")
    )



# 3.13 Attaching original note text to predictions


prediction_note_metadata_columns = [
    "annotation_note_number",
    "note_id",
    "subject_id",
    "hadm_id",
    "sampling_group",
    "enrichment_category",
    "original_note_text",
]

# Removeing overlapping metadata columns before merging to avoid
# suffix confusion.
prediction_merge_df = system_predictions_df.drop(
    columns=[
        column
        for column in [
            "subject_id",
            "hadm_id",
            "sampling_group",
            "enrichment_category",
        ]
        if column in system_predictions_df.columns
    ]
)

system_predictions_df = prediction_merge_df.merge(
    note_register_df[
        prediction_note_metadata_columns
    ],
    on=[
        "annotation_note_number",
        "note_id",
    ],
    how="left",
    validate="many_to_one",
)


# 3.14 Validating system-prediction offsets


prediction_validation_rows = []

for row_index, row in system_predictions_df.iterrows():

    note_text = (
        row["original_note_text"]
        if isinstance(
            row["original_note_text"],
            str,
        )
        else ""
    )

    start_char = row["start_char"]
    end_char = row["end_char"]
    mention_text = row["mention_text"]

    valid_offsets = (
        pd.notna(start_char)
        and pd.notna(end_char)
        and int(start_char) >= 0
        and int(end_char) > int(start_char)
        and int(end_char) <= len(note_text)
    )

    extracted_text = None
    exact_text_match = False
    case_insensitive_match = False

    if valid_offsets:

        extracted_text = note_text[
            int(start_char):int(end_char)
        ]

        exact_text_match = (
            extracted_text == str(mention_text)
        )

        case_insensitive_match = (
            extracted_text.strip().casefold()
            == str(mention_text).strip().casefold()
        )

    prediction_validation_rows.append(
        {
            "row_index": row_index,
            "candidate_id":
                row["candidate_id"],
            "note_id":
                row["note_id"],
            "valid_offsets":
                valid_offsets,
            "offset_text":
                extracted_text,
            "prediction_text":
                mention_text,
            "exact_text_match":
                exact_text_match,
            "case_insensitive_match":
                case_insensitive_match,
        }
    )

prediction_span_validation_df = pd.DataFrame(
    prediction_validation_rows
)

system_predictions_df = system_predictions_df.merge(
    prediction_span_validation_df[
        [
            "candidate_id",
            "valid_offsets",
            "offset_text",
            "exact_text_match",
            "case_insensitive_match",
        ]
    ],
    on="candidate_id",
    how="left",
    validate="one_to_one",
)



# 3.15 Selecting final prediction columns


system_predictions_df = system_predictions_df[
    [
        "candidate_id",
        "annotation_note_number",
        "note_id",
        "subject_id",
        "hadm_id",
        "sampling_group",
        "enrichment_category",
        "mention_text",
        "normalized_text",
        "complication_category",
        "start_char",
        "end_char",
        "sentence_text",
        "predicted_assertion",
        "predicted_temporality",
        "is_negated",
        "is_historical",
        "is_family",
        "is_uncertain",
        "is_hypothetical",
        "is_current_affirmed",
        "valid_offsets",
        "offset_text",
        "exact_text_match",
        "case_insensitive_match",
        "original_note_text",
    ]
].copy()



# 3.16 Checking prediction integrity


duplicate_candidate_ids = (
    system_predictions_df.loc[
        system_predictions_df[
            "candidate_id"
        ].duplicated(
            keep=False
        ),
        "candidate_id",
    ]
    .dropna()
    .unique()
    .tolist()
)

invalid_prediction_offset_count = int(
    (
        ~system_predictions_df[
            "valid_offsets"
        ]
    ).sum()
)

prediction_exact_text_mismatch_count = int(
    (
        system_predictions_df["valid_offsets"]
        &
        ~system_predictions_df[
            "exact_text_match"
        ]
    ).sum()
)

prediction_case_insensitive_mismatch_count = int(
    (
        system_predictions_df["valid_offsets"]
        &
        ~system_predictions_df[
            "case_insensitive_match"
        ]
    ).sum()
)


# 3.17 Constructing summary statistics


construction_summary_df = pd.DataFrame(
    [
        {
            "Measure":
                "Registered corpus notes",
            "Value":
                registered_note_count,
        },
        {
            "Measure":
                "Fully reviewed notes",
            "Value":
                full_review_note_count,
        },
        {
            "Measure":
                "All Mention_Annotations rows",
            "Value":
                len(gold_source_df),
        },
        {
            "Measure":
                "Positive gold-standard mentions",
            "Value":
                len(gold_standard_df),
        },
        {
            "Measure":
                "Excluded non-positive annotation rows",
            "Value":
                len(excluded_gold_rows_df),
        },
        {
            "Measure":
                "Notes containing gold mentions",
            "Value":
                gold_standard_df[
                    "note_id"
                ].nunique(),
        },
        {
            "Measure":
                "Notes containing predictions",
            "Value":
                system_predictions_df[
                    "note_id"
                ].nunique(),
        },
        {
            "Measure":
                "System predictions",
            "Value":
                len(system_predictions_df),
        },
        {
            "Measure":
                "Gold annotations with invalid offsets",
            "Value":
                invalid_gold_offset_count,
        },
        {
            "Measure":
                "Gold exact-text mismatches",
            "Value":
                gold_exact_text_mismatch_count,
        },
        {
            "Measure":
                "Prediction offsets invalid",
            "Value":
                invalid_prediction_offset_count,
        },
        {
            "Measure":
                "Prediction exact-text mismatches",
            "Value":
                prediction_exact_text_mismatch_count,
        },
    ]
)

print("\nDataset construction summary:")
display(construction_summary_df)


# 3.18 Displaying missing-value audit


print("\nMissing required values in the gold standard:")
display(
    gold_missing_required_values
    .rename("Missing_count")
    .to_frame()
)


# 3.19 Displaying gold-standard distributions

print("\nGold-standard complication-category distribution:")

gold_category_distribution_df = (
    gold_standard_df[
        "complication_category"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "complication_category"
    )
    .reset_index(
        name="gold_mentions"
    )
)

display(gold_category_distribution_df)


print("\nGold-standard assertion distribution:")

gold_assertion_distribution_df = (
    gold_standard_df["assertion"]
    .value_counts(dropna=False)
    .rename_axis("assertion")
    .reset_index(name="gold_mentions")
)

display(gold_assertion_distribution_df)


print("\nGold-standard temporality distribution:")

gold_temporality_distribution_df = (
    gold_standard_df["temporality"]
    .value_counts(dropna=False)
    .rename_axis("temporality")
    .reset_index(name="gold_mentions")
)

display(gold_temporality_distribution_df)


# 3.20 Displaying system-prediction distributions

print("\nSystem-prediction category distribution:")

prediction_category_distribution_df = (
    system_predictions_df[
        "complication_category"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "complication_category"
    )
    .reset_index(
        name="predicted_mentions"
    )
)

display(prediction_category_distribution_df)


# 3.21 Saving standardised datasets

gold_standard_df.to_csv(
    GOLD_STANDARD_MENTIONS_FILE,
    index=False,
)

system_predictions_df.to_csv(
    SYSTEM_PREDICTIONS_FILE,
    index=False,
)

GOLD_SPAN_VALIDATION_FILE = (
    EVALUATION_DIRECTORY
    / "gold_standard_span_validation.csv"
)

PREDICTION_SPAN_VALIDATION_FILE = (
    EVALUATION_DIRECTORY
    / "system_prediction_span_validation.csv"
)

gold_span_validation_df.to_csv(
    GOLD_SPAN_VALIDATION_FILE,
    index=False,
)

prediction_span_validation_df.to_csv(
    PREDICTION_SPAN_VALIDATION_FILE,
    index=False,
)


# 3.22 Saving dataset construction report


DATASET_CONSTRUCTION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_dataset_construction_report.json"
)

dataset_construction_report = {
    "corpus": {
        "registered_notes":
            int(registered_note_count),

        "fully_reviewed_notes":
            int(full_review_note_count),
    },

    "gold_standard": {
        "source_rows":
            int(len(gold_source_df)),

        "positive_mentions":
            int(len(gold_standard_df)),

        "excluded_non_positive_rows":
            int(len(excluded_gold_rows_df)),

        "notes_with_mentions":
            int(
                gold_standard_df[
                    "note_id"
                ].nunique()
            ),

        "invalid_offset_count":
            invalid_gold_offset_count,

        "exact_text_mismatch_count":
            gold_exact_text_mismatch_count,

        "case_insensitive_mismatch_count":
            gold_case_insensitive_mismatch_count,

        "duplicate_annotation_ids":
            [
                str(value)
                for value
                in duplicate_annotation_ids
            ],

        "missing_required_values":
            {
                str(key): int(value)
                for key, value
                in gold_missing_required_values.items()
            },
    },

    "system_predictions": {
        "prediction_count":
            int(len(system_predictions_df)),

        "notes_with_predictions":
            int(
                system_predictions_df[
                    "note_id"
                ].nunique()
            ),

        "invalid_offset_count":
            invalid_prediction_offset_count,

        "exact_text_mismatch_count":
            prediction_exact_text_mismatch_count,

        "case_insensitive_mismatch_count":
            prediction_case_insensitive_mismatch_count,

        "duplicate_candidate_ids":
            [
                str(value)
                for value
                in duplicate_candidate_ids
            ],
    },

    "output_files": {
        "gold_standard_mentions":
            str(GOLD_STANDARD_MENTIONS_FILE),

        "system_predictions":
            str(SYSTEM_PREDICTIONS_FILE),

        "gold_span_validation":
            str(GOLD_SPAN_VALIDATION_FILE),

        "prediction_span_validation":
            str(PREDICTION_SPAN_VALIDATION_FILE),
    },

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    DATASET_CONSTRUCTION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        dataset_construction_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# 3.23 Critical validation failures


critical_errors = []

if duplicate_annotation_ids:
    critical_errors.append(
        "Duplicate annotation IDs were detected."
    )

if duplicate_candidate_ids:
    critical_errors.append(
        "Duplicate candidate IDs were detected."
    )

if gold_missing_required_values.sum() > 0:
    critical_errors.append(
        "The gold standard contains missing required values."
    )

if invalid_gold_offset_count > 0:
    critical_errors.append(
        "One or more gold-standard offsets are invalid."
    )

if invalid_prediction_offset_count > 0:
    critical_errors.append(
        "One or more system-prediction offsets are invalid."
    )

if gold_case_insensitive_mismatch_count > 0:
    critical_errors.append(
        "One or more gold-standard mention texts do not "
        "match the note text at their recorded offsets."
    )

if prediction_case_insensitive_mismatch_count > 0:
    critical_errors.append(
        "One or more prediction texts do not match the note "
        "text at their recorded offsets."
    )



# 3.24 Completion status

print("\n" + "-" * 80)

if critical_errors:

    print("STEP 3 COMPLETED WITH VALIDATION ERRORS")
    print("-" * 80)

    for error_number, error_message in enumerate(
        critical_errors,
        start=1,
    ):
        print(
            f"{error_number}. {error_message}"
        )

    print(
        "\nDo not begin mention matching until these "
        "validation problems have been reviewed."
    )

else:

    print("STEP 3 COMPLETE — DATASETS VALIDATED")
    print("-" * 80)

    print(
        f"\nGold-standard mentions: "
        f"{len(gold_standard_df):,}"
    )

    print(
        f"System predictions: "
        f"{len(system_predictions_df):,}"
    )

    print(
        f"Corpus notes: "
        f"{registered_note_count:,}"
    )

    print(
        "\nAll required identifiers, note links and "
        "character offsets passed validation."
    )

    print(
        "\nThe datasets are ready for exact-span and "
        "relaxed-span mention matching."
    )

print(
    f"\nGold-standard table saved to:\n"
    f"{GOLD_STANDARD_MENTIONS_FILE}"
)

print(
    f"\nSystem-prediction table saved to:\n"
    f"{SYSTEM_PREDICTIONS_FILE}"
)

print(
    f"\nConstruction report saved to:\n"
    f"{DATASET_CONSTRUCTION_REPORT_FILE}"
)

In [ ]:
# 3A. Diagnose Prediction Text and Offset Source — robust and efficient version

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display


print("-" * 80)
print("STEP 3A — DIAGNOSE PREDICTION OFFSET SOURCE")
print("-" * 80)


# 3A.1 Helper functions

def clean_text_value(value):
    """
    Convert a value to a safe string without turning NaN into 'nan'.
    """
    if pd.isna(value):
        return ""
    return str(value)


def normalise_for_comparison(value):
    """
    Normalise text for case-insensitive comparison while preserving
    internal punctuation and spacing.
    """
    return clean_text_value(value).strip().casefold()


def safe_prefix(value, length=150):
    """
    Return a printable prefix even when the source value is missing.
    """
    return repr(clean_text_value(value)[:length])


def safe_span(text, start_char, end_char):
    """
    Extract a character span only when the text and offsets are valid.
    """
    text = clean_text_value(text)

    if pd.isna(start_char) or pd.isna(end_char):
        return pd.NA

    try:
        start_char = int(start_char)
        end_char = int(end_char)
    except (TypeError, ValueError, OverflowError):
        return pd.NA

    if (
        start_char < 0
        or end_char <= start_char
        or end_char > len(text)
    ):
        return pd.NA

    return text[start_char:end_char]


def find_all_occurrences(text, entity_text):
    """
    Return all case-insensitive literal occurrences of entity_text.
    """
    text = clean_text_value(text)
    entity_text = clean_text_value(entity_text).strip()

    if not text or not entity_text:
        return []

    pattern = re.compile(
        re.escape(entity_text),
        flags=re.IGNORECASE,
    )

    return [
        (match.start(), match.end())
        for match in pattern.finditer(text)
    ]


def nearest_occurrence(text, entity_text, expected_start):
    """
    Find the entity occurrence nearest to the stored start offset.

    Returns:
        found_start, found_end, offset_delta, occurrence_count
    """
    occurrences = find_all_occurrences(
        text=text,
        entity_text=entity_text,
    )

    if not occurrences:
        return pd.NA, pd.NA, pd.NA, 0

    if pd.isna(expected_start):
        selected_start, selected_end = occurrences[0]
        return (
            selected_start,
            selected_end,
            pd.NA,
            len(occurrences),
        )

    try:
        expected_start = int(expected_start)
    except (TypeError, ValueError, OverflowError):
        selected_start, selected_end = occurrences[0]
        return (
            selected_start,
            selected_end,
            pd.NA,
            len(occurrences),
        )

    selected_start, selected_end = min(
        occurrences,
        key=lambda span: abs(span[0] - expected_start),
    )

    return (
        selected_start,
        selected_end,
        selected_start - expected_start,
        len(occurrences),
    )


def safe_mode(series):
    """
    Return the first mode or NaN when no mode exists.
    """
    cleaned = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()

    if cleaned.empty:
        return np.nan

    modes = cleaned.mode()

    if modes.empty:
        return np.nan

    return modes.iloc[0]


# 3A.2 Preparing workbook note text


required_workbook_columns = {
    "note_id",
    "annotation_note_number",
    "original_note_text",
}

missing_workbook_columns = (
    required_workbook_columns
    - set(note_register_df.columns)
)

if missing_workbook_columns:
    raise KeyError(
        "note_register_df is missing required columns: "
        + ", ".join(sorted(missing_workbook_columns))
    )


workbook_note_text_df = (
    note_register_df[
        [
            "note_id",
            "annotation_note_number",
            "original_note_text",
        ]
    ]
    .copy()
)

workbook_note_text_df["note_id"] = (
    clean_string_series(
        workbook_note_text_df["note_id"]
    )
)

workbook_note_text_df["workbook_annotation_note_number"] = (
    safe_integer_series(
        workbook_note_text_df["annotation_note_number"]
    )
)

workbook_note_text_df["workbook_text"] = (
    workbook_note_text_df["original_note_text"]
    .map(clean_text_value)
)

workbook_note_text_df = (
    workbook_note_text_df[
        [
            "note_id",
            "workbook_annotation_note_number",
            "workbook_text",
        ]
    ]
    .drop_duplicates(
        subset=["note_id"],
        keep="first",
    )
    .reset_index(drop=True)
)


# 3A.3 Preparing sample note text

required_sample_columns = {
    "note_id",
    "annotation_note_number",
    "text",
}

missing_sample_columns = (
    required_sample_columns
    - set(sample_with_text_df.columns)
)

if missing_sample_columns:
    raise KeyError(
        "sample_with_text_df is missing required columns: "
        + ", ".join(sorted(missing_sample_columns))
    )


sample_note_text_df = (
    sample_with_text_df[
        [
            "note_id",
            "annotation_note_number",
            "text",
        ]
    ]
    .copy()
)

sample_note_text_df["note_id"] = (
    clean_string_series(
        sample_note_text_df["note_id"]
    )
)

sample_note_text_df["sample_annotation_note_number"] = (
    safe_integer_series(
        sample_note_text_df["annotation_note_number"]
    )
)

sample_note_text_df["sample_text"] = (
    sample_note_text_df["text"]
    .map(clean_text_value)
)

sample_note_text_df = (
    sample_note_text_df[
        [
            "note_id",
            "sample_annotation_note_number",
            "sample_text",
        ]
    ]
    .drop_duplicates(
        subset=["note_id"],
        keep="first",
    )
    .reset_index(drop=True)
)


# 3A.4 Comparing note availability and note numbering

text_comparison_df = workbook_note_text_df.merge(
    sample_note_text_df,
    on="note_id",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

text_comparison_df["present_in_workbook"] = (
    text_comparison_df["_merge"].isin(
        ["both", "left_only"]
    )
)

text_comparison_df["present_in_sample"] = (
    text_comparison_df["_merge"].isin(
        ["both", "right_only"]
    )
)

text_comparison_df["annotation_number_matches"] = (
    text_comparison_df[
        "workbook_annotation_note_number"
    ].eq(
        text_comparison_df[
            "sample_annotation_note_number"
        ]
    )
)

text_comparison_df["workbook_text"] = (
    text_comparison_df["workbook_text"]
    .map(clean_text_value)
)

text_comparison_df["sample_text"] = (
    text_comparison_df["sample_text"]
    .map(clean_text_value)
)

text_comparison_df["workbook_length"] = (
    text_comparison_df["workbook_text"].str.len()
)

text_comparison_df["sample_length"] = (
    text_comparison_df["sample_text"].str.len()
)

text_comparison_df["length_difference"] = (
    text_comparison_df["sample_length"]
    - text_comparison_df["workbook_length"]
)

text_comparison_df["texts_exactly_equal"] = (
    text_comparison_df["workbook_text"]
    .eq(text_comparison_df["sample_text"])
    & text_comparison_df["present_in_workbook"]
    & text_comparison_df["present_in_sample"]
)

text_comparison_df["texts_equal_after_strip"] = (
    text_comparison_df["workbook_text"]
    .str.strip()
    .eq(
        text_comparison_df["sample_text"]
        .str.strip()
    )
    & text_comparison_df["present_in_workbook"]
    & text_comparison_df["present_in_sample"]
)


note_source_summary_df = pd.DataFrame(
    [
        {
            "Measure": "Unique workbook notes",
            "Value": int(
                workbook_note_text_df["note_id"].nunique()
            ),
        },
        {
            "Measure": "Unique sample notes",
            "Value": int(
                sample_note_text_df["note_id"].nunique()
            ),
        },
        {
            "Measure": "Union of note IDs",
            "Value": int(len(text_comparison_df)),
        },
        {
            "Measure": "Notes present in both sources",
            "Value": int(
                (
                    text_comparison_df["_merge"]
                    == "both"
                ).sum()
            ),
        },
        {
            "Measure": "Notes only in workbook",
            "Value": int(
                (
                    text_comparison_df["_merge"]
                    == "left_only"
                ).sum()
            ),
        },
        {
            "Measure": "Notes only in sample",
            "Value": int(
                (
                    text_comparison_df["_merge"]
                    == "right_only"
                ).sum()
            ),
        },
        {
            "Measure": "Matching annotation numbers",
            "Value": int(
                text_comparison_df[
                    "annotation_number_matches"
                ].sum()
            ),
        },
        {
            "Measure": "Exactly identical note texts",
            "Value": int(
                text_comparison_df[
                    "texts_exactly_equal"
                ].sum()
            ),
        },
        {
            "Measure": "Equal after outer-whitespace removal",
            "Value": int(
                text_comparison_df[
                    "texts_equal_after_strip"
                ].sum()
            ),
        },
        {
            "Measure": "Notes with different text lengths",
            "Value": int(
                (
                    text_comparison_df[
                        "length_difference"
                    ] != 0
                ).sum()
            ),
        },
    ]
)

print("\nNote-text source comparison:")
display(note_source_summary_df)


print("\nLength-difference summary:")
display(
    text_comparison_df[
        "length_difference"
    ]
    .describe()
    .rename("Value")
    .to_frame()
)


# 3A.5 Preparing predictions and join note texts

required_prediction_columns = {
    "candidate_id",
    "note_id",
    "entity_text",
    "start_char",
    "end_char",
}

missing_prediction_columns = (
    required_prediction_columns
    - set(candidate_mentions_df.columns)
)

if missing_prediction_columns:
    raise KeyError(
        "candidate_mentions_df is missing required columns: "
        + ", ".join(sorted(missing_prediction_columns))
    )


prediction_source_test_df = (
    candidate_mentions_df.copy()
)

prediction_source_test_df["note_id"] = (
    clean_string_series(
        prediction_source_test_df["note_id"]
    )
)

prediction_source_test_df["entity_text"] = (
    prediction_source_test_df["entity_text"]
    .map(clean_text_value)
)

prediction_source_test_df["start_char"] = (
    safe_integer_series(
        prediction_source_test_df["start_char"]
    )
)

prediction_source_test_df["end_char"] = (
    safe_integer_series(
        prediction_source_test_df["end_char"]
    )
)


# Merge by note_id only.
# This prevents annotation-number differences from dropping valid notes.

prediction_source_test_df = (
    prediction_source_test_df.merge(
        text_comparison_df[
            [
                "note_id",
                "workbook_annotation_note_number",
                "sample_annotation_note_number",
                "workbook_text",
                "sample_text",
                "workbook_length",
                "sample_length",
                "length_difference",
                "texts_exactly_equal",
                "_merge",
            ]
        ],
        on="note_id",
        how="left",
        validate="many_to_one",
    )
)


# 3A.6 Direct span reconstruction

prediction_source_test_df[
    "workbook_offset_text"
] = [
    safe_span(text, start, end)
    for text, start, end in zip(
        prediction_source_test_df["workbook_text"],
        prediction_source_test_df["start_char"],
        prediction_source_test_df["end_char"],
    )
]

prediction_source_test_df[
    "sample_offset_text"
] = [
    safe_span(text, start, end)
    for text, start, end in zip(
        prediction_source_test_df["sample_text"],
        prediction_source_test_df["start_char"],
        prediction_source_test_df["end_char"],
    )
]


entity_normalised = (
    prediction_source_test_df["entity_text"]
    .map(normalise_for_comparison)
)

workbook_span_normalised = (
    prediction_source_test_df[
        "workbook_offset_text"
    ]
    .map(normalise_for_comparison)
)

sample_span_normalised = (
    prediction_source_test_df[
        "sample_offset_text"
    ]
    .map(normalise_for_comparison)
)

prediction_source_test_df[
    "matches_workbook_text"
] = (
    entity_normalised.ne("")
    & entity_normalised.eq(
        workbook_span_normalised
    )
)

prediction_source_test_df[
    "matches_sample_text"
] = (
    entity_normalised.ne("")
    & entity_normalised.eq(
        sample_span_normalised
    )
)


# 3A.7 Searching for nearest literal entity occurrence

workbook_nearest_results = [
    nearest_occurrence(
        text=text,
        entity_text=entity,
        expected_start=start,
    )
    for text, entity, start in zip(
        prediction_source_test_df["workbook_text"],
        prediction_source_test_df["entity_text"],
        prediction_source_test_df["start_char"],
    )
]

sample_nearest_results = [
    nearest_occurrence(
        text=text,
        entity_text=entity,
        expected_start=start,
    )
    for text, entity, start in zip(
        prediction_source_test_df["sample_text"],
        prediction_source_test_df["entity_text"],
        prediction_source_test_df["start_char"],
    )
]


prediction_source_test_df[
    [
        "workbook_nearest_start",
        "workbook_nearest_end",
        "workbook_offset_delta",
        "workbook_occurrence_count",
    ]
] = pd.DataFrame(
    workbook_nearest_results,
    index=prediction_source_test_df.index,
)

prediction_source_test_df[
    [
        "sample_nearest_start",
        "sample_nearest_end",
        "sample_offset_delta",
        "sample_occurrence_count",
    ]
] = pd.DataFrame(
    sample_nearest_results,
    index=prediction_source_test_df.index,
)


prediction_source_test_df[
    "entity_found_in_workbook"
] = (
    prediction_source_test_df[
        "workbook_occurrence_count"
    ] > 0
)

prediction_source_test_df[
    "entity_found_in_sample"
] = (
    prediction_source_test_df[
        "sample_occurrence_count"
    ] > 0
)


# 3A.8 Summaries

total_predictions = len(
    prediction_source_test_df
)

workbook_direct_matches = int(
    prediction_source_test_df[
        "matches_workbook_text"
    ].sum()
)

sample_direct_matches = int(
    prediction_source_test_df[
        "matches_sample_text"
    ].sum()
)

workbook_entities_found = int(
    prediction_source_test_df[
        "entity_found_in_workbook"
    ].sum()
)

sample_entities_found = int(
    prediction_source_test_df[
        "entity_found_in_sample"
    ].sum()
)


def percentage(numerator, denominator):
    if denominator == 0:
        return 0.0
    return round(
        100 * numerator / denominator,
        2,
    )


source_test_summary_df = pd.DataFrame(
    [
        {
            "Text source":
                "Note_Register.original_note_text",
            "Direct offset matches":
                workbook_direct_matches,
            "Entity found anywhere":
                workbook_entities_found,
            "Total predictions":
                total_predictions,
            "Direct_match_percentage":
                percentage(
                    workbook_direct_matches,
                    total_predictions,
                ),
            "Entity_found_percentage":
                percentage(
                    workbook_entities_found,
                    total_predictions,
                ),
            "Median_offset_delta":
                pd.to_numeric(
                    prediction_source_test_df[
                        "workbook_offset_delta"
                    ],
                    errors="coerce",
                ).median(),
            "Most_common_offset_delta":
                safe_mode(
                    prediction_source_test_df[
                        "workbook_offset_delta"
                    ]
                ),
        },
        {
            "Text source":
                "sample_with_text.text",
            "Direct offset matches":
                sample_direct_matches,
            "Entity found anywhere":
                sample_entities_found,
            "Total predictions":
                total_predictions,
            "Direct_match_percentage":
                percentage(
                    sample_direct_matches,
                    total_predictions,
                ),
            "Entity_found_percentage":
                percentage(
                    sample_entities_found,
                    total_predictions,
                ),
            "Median_offset_delta":
                pd.to_numeric(
                    prediction_source_test_df[
                        "sample_offset_delta"
                    ],
                    errors="coerce",
                ).median(),
            "Most_common_offset_delta":
                safe_mode(
                    prediction_source_test_df[
                        "sample_offset_delta"
                    ]
                ),
        },
    ]
)

print("\nPrediction offset-source test:")
display(source_test_summary_df)


# 3A.9 Representative examples

diagnostic_example_columns = [
    "candidate_id",
    "note_id",
    "entity_text",
    "start_char",
    "end_char",
    "workbook_offset_text",
    "sample_offset_text",
    "matches_workbook_text",
    "matches_sample_text",
    "workbook_nearest_start",
    "sample_nearest_start",
    "workbook_offset_delta",
    "sample_offset_delta",
    "workbook_occurrence_count",
    "sample_occurrence_count",
    "length_difference",
]

print("\nFirst 15 prediction source comparisons:")

display(
    prediction_source_test_df[
        diagnostic_example_columns
    ].head(15)
)


print("\nFirst 15 predictions with failed direct offsets:")

failed_direct_offsets_df = (
    prediction_source_test_df.loc[
        ~(
            prediction_source_test_df[
                "matches_workbook_text"
            ]
            |
            prediction_source_test_df[
                "matches_sample_text"
            ]
        ),
        diagnostic_example_columns,
    ]
    .head(15)
)

display(failed_direct_offsets_df)


# 3A.10 Show the missing and mismatched notes

unmatched_notes_df = (
    text_comparison_df.loc[
        text_comparison_df["_merge"] != "both",
        [
            "note_id",
            "workbook_annotation_note_number",
            "sample_annotation_note_number",
            "_merge",
            "workbook_length",
            "sample_length",
        ],
    ]
    .copy()
)

if not unmatched_notes_df.empty:

    print(
        "\nNotes present in only one text source:"
    )

    display(unmatched_notes_df.head(30))


annotation_number_mismatch_df = (
    text_comparison_df.loc[
        (text_comparison_df["_merge"] == "both")
        & (
            ~text_comparison_df[
                "annotation_number_matches"
            ]
        ),
        [
            "note_id",
            "workbook_annotation_note_number",
            "sample_annotation_note_number",
        ],
    ]
    .copy()
)

if not annotation_number_mismatch_df.empty:

    print(
        "\nNotes with different annotation-note numbers:"
    )

    display(
        annotation_number_mismatch_df.head(30)
    )


# 3A.11 Inspecting differing text prefixes safely

different_text_notes_df = (
    text_comparison_df.loc[
        (
            text_comparison_df["_merge"]
            != "both"
        )
        |
        (
            ~text_comparison_df[
                "texts_exactly_equal"
            ]
        )
    ]
    .copy()
)

if not different_text_notes_df.empty:

    prefix_comparison_df = (
        different_text_notes_df[
            [
                "note_id",
                "_merge",
                "workbook_length",
                "sample_length",
                "length_difference",
                "workbook_text",
                "sample_text",
            ]
        ]
        .head(10)
        .copy()
    )

    prefix_comparison_df[
        "workbook_prefix_repr"
    ] = (
        prefix_comparison_df[
            "workbook_text"
        ]
        .map(safe_prefix)
    )

    prefix_comparison_df[
        "sample_prefix_repr"
    ] = (
        prefix_comparison_df[
            "sample_text"
        ]
        .map(safe_prefix)
    )

    print(
        "\nPrefixes from the first notes whose text "
        "representations differ:"
    )

    display(
        prefix_comparison_df[
            [
                "note_id",
                "_merge",
                "workbook_length",
                "sample_length",
                "length_difference",
                "workbook_prefix_repr",
                "sample_prefix_repr",
            ]
        ]
    )


# 3A.12 Saving diagnostic outputs

TEXT_SOURCE_COMPARISON_FILE = (
    EVALUATION_DIRECTORY
    / "note_text_source_comparison.csv"
)

PREDICTION_SOURCE_DIAGNOSTIC_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_source_diagnostic.csv"
)

OFFSET_DELTA_SUMMARY_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_delta_summary.csv"
)


text_comparison_df.drop(
    columns=[
        "workbook_text",
        "sample_text",
    ],
    errors="ignore",
).to_csv(
    TEXT_SOURCE_COMPARISON_FILE,
    index=False,
)


prediction_source_test_df.drop(
    columns=[
        "workbook_text",
        "sample_text",
    ],
    errors="ignore",
).to_csv(
    PREDICTION_SOURCE_DIAGNOSTIC_FILE,
    index=False,
)


source_test_summary_df.to_csv(
    OFFSET_DELTA_SUMMARY_FILE,
    index=False,
)


# 3A.13 Diagnostic conclusion

print("\n" + "-" * 80)
print("STEP 3A — DIAGNOSTIC CONCLUSION")
print("-" * 80)


if sample_direct_matches == total_predictions:

    print(
        "\nConfirmed: all prediction offsets correspond "
        "to sample_with_text.text."
    )

    print(
        "Use sample_with_text.text for prediction-offset "
        "validation."
    )

elif workbook_direct_matches == total_predictions:

    print(
        "\nConfirmed: all prediction offsets correspond "
        "to Note_Register.original_note_text."
    )

    print(
        "Use Note_Register.original_note_text for "
        "prediction-offset validation."
    )

elif (
    sample_direct_matches > workbook_direct_matches
    and sample_direct_matches > 0
):

    print(
        "\nThe sample text is the better direct-offset "
        "source, but not every prediction aligns."
    )

    print(
        "Inspect the offset-delta distribution before "
        "continuing with evaluation."
    )

elif (
    workbook_direct_matches > sample_direct_matches
    and workbook_direct_matches > 0
):

    print(
        "\nThe workbook text is the better direct-offset "
        "source, but not every prediction aligns."
    )

    print(
        "Inspect the offset-delta distribution before "
        "continuing with evaluation."
    )

elif (
    sample_entities_found > 0
    or workbook_entities_found > 0
):

    print(
        "\nThe entities exist in at least one note-text "
        "source, but the stored character offsets do not align."
    )

    print(
        "Use the nearest-occurrence and offset-delta columns "
        "to identify whether the offsets are section-relative, "
        "normalised-text-relative, or systematically shifted."
    )

else:

    print(
        "\nNeither note-text source contains the predicted "
        "entity strings at the recorded note IDs."
    )

    print(
        "This suggests the candidate file and note-text files "
        "were generated from different data versions."
    )


print(
    f"\nText-source comparison saved to:\n"
    f"{TEXT_SOURCE_COMPARISON_FILE}"
)

print(
    f"\nPrediction diagnostic saved to:\n"
    f"{PREDICTION_SOURCE_DIAGNOSTIC_FILE}"
)

print(
    f"\nOffset summary saved to:\n"
    f"{OFFSET_DELTA_SUMMARY_FILE}"
)

### 3B. Reconstructing document-level prediction offsets

The original candidate character offsets were generated relative to an intermediate NLP processing unit rather than the complete discharge summary. This section locates each candidate's supporting sentence within the original note and reconstructs document-level mention offsets.

Reconstruction is accepted automatically only when the supporting sentence and mention can be located unambiguously. Ambiguous or unresolved candidates are retained for separate review rather than assigned fabricated offsets.

In [ ]:
# 3B. Reconstructing Document-Level Prediction Offsets

print("-" * 80)
print("STEP 3B — RECONSTRUCT DOCUMENT-LEVEL PREDICTION OFFSETS")
print("-" * 80)


# 3B.1 Preparing candidate data and full note text


offset_reconstruction_df = candidate_mentions_df.copy()

offset_reconstruction_df["candidate_id"] = clean_string_series(
    offset_reconstruction_df["candidate_id"]
)

offset_reconstruction_df["note_id"] = clean_string_series(
    offset_reconstruction_df["note_id"]
)

offset_reconstruction_df["entity_text"] = clean_string_series(
    offset_reconstruction_df["entity_text"]
)

offset_reconstruction_df["sentence_text"] = (
    offset_reconstruction_df["sentence_text"]
    .fillna("")
    .astype(str)
)

offset_reconstruction_df["original_start_char"] = safe_integer_series(
    offset_reconstruction_df["start_char"]
)

offset_reconstruction_df["original_end_char"] = safe_integer_series(
    offset_reconstruction_df["end_char"]
)


note_text_lookup_df = note_register_df[
    [
        "annotation_note_number",
        "note_id",
        "original_note_text",
    ]
].copy()

note_text_lookup_df["note_id"] = clean_string_series(
    note_text_lookup_df["note_id"]
)

note_text_lookup_df["original_note_text"] = (
    note_text_lookup_df["original_note_text"]
    .fillna("")
    .astype(str)
)


offset_reconstruction_df = offset_reconstruction_df.merge(
    note_text_lookup_df,
    on=[
        "annotation_note_number",
        "note_id",
    ],
    how="left",
    validate="many_to_one",
)


# 3B.2 Utility functions

import re
import unicodedata


def find_all_exact_occurrences(text, substring):
    """
    Return every exact occurrence of substring in text.
    Overlapping matches are included.
    """

    if (
        not isinstance(text, str)
        or not isinstance(substring, str)
        or substring == ""
    ):
        return []

    positions = []
    search_start = 0

    while True:

        position = text.find(
            substring,
            search_start,
        )

        if position == -1:
            break

        positions.append(position)

        search_start = position + 1

    return positions


def normalise_with_index_map(text):
    """
    Create a whitespace-normalised, case-folded representation
    while retaining a map back to original character positions.

    Consecutive whitespace characters are collapsed to one space.
    Unicode characters are normalised using NFKC.
    """

    if not isinstance(text, str):
        return "", []

    normalised_characters = []
    original_index_map = []

    previous_was_space = False

    for original_index, character in enumerate(text):

        expanded = unicodedata.normalize(
            "NFKC",
            character,
        ).casefold()

        for expanded_character in expanded:

            if expanded_character.isspace():

                if not previous_was_space:
                    normalised_characters.append(" ")
                    original_index_map.append(original_index)

                previous_was_space = True

            else:

                normalised_characters.append(
                    expanded_character
                )

                original_index_map.append(
                    original_index
                )

                previous_was_space = False

    return (
        "".join(normalised_characters),
        original_index_map,
    )


def find_normalised_occurrences(
    original_text,
    query_text,
):
    """
    Locate query_text in original_text after NFKC,
    case and whitespace normalisation.

    Returns document-level start and end offsets in the
    original text.
    """

    if (
        not isinstance(original_text, str)
        or not isinstance(query_text, str)
        or query_text.strip() == ""
    ):
        return []

    normalised_document, document_index_map = (
        normalise_with_index_map(
            original_text
        )
    )

    normalised_query, _ = normalise_with_index_map(
        query_text
    )

    normalised_query = normalised_query.strip()

    if not normalised_query:
        return []

    matches = []

    for match in re.finditer(
        re.escape(normalised_query),
        normalised_document,
    ):

        normalised_start = match.start()
        normalised_end = match.end()

        if (
            normalised_start >= len(document_index_map)
            or normalised_end <= 0
        ):
            continue

        original_start = document_index_map[
            normalised_start
        ]

        original_end = (
            document_index_map[
                normalised_end - 1
            ]
            + 1
        )

        matches.append(
            (
                original_start,
                original_end,
            )
        )

    return matches


def find_entity_inside_sentence(
    sentence_text,
    entity_text,
):
    """
    Return entity offsets relative to the supporting sentence.

    Exact matching is attempted first, followed by
    case/whitespace-normalised matching.
    """

    exact_positions = find_all_exact_occurrences(
        sentence_text,
        entity_text,
    )

    if exact_positions:

        return [
            (
                position,
                position + len(entity_text),
                "EXACT",
            )
            for position in exact_positions
        ]

    normalised_matches = find_normalised_occurrences(
        sentence_text,
        entity_text,
    )

    return [
        (
            start,
            end,
            "NORMALISED",
        )
        for start, end in normalised_matches
    ]


def extract_context(
    text,
    start_char,
    end_char,
    context_width=60,
):
    """
    Extract a short context window around a reconstructed span.
    """

    if (
        not isinstance(text, str)
        or pd.isna(start_char)
        or pd.isna(end_char)
    ):
        return pd.NA

    start_char = int(start_char)
    end_char = int(end_char)

    context_start = max(
        0,
        start_char - context_width,
    )

    context_end = min(
        len(text),
        end_char + context_width,
    )

    return text[
        context_start:context_end
    ]


# 3B.3 Reconstructing offsets candidate by candidate

reconstruction_rows = []

for _, row in offset_reconstruction_df.iterrows():

    candidate_id = row["candidate_id"]
    note_id = row["note_id"]

    entity_text = (
        str(row["entity_text"])
        if pd.notna(row["entity_text"])
        else ""
    )

    sentence_text = (
        row["sentence_text"]
        if isinstance(
            row["sentence_text"],
            str,
        )
        else ""
    )

    note_text = (
        row["original_note_text"]
        if isinstance(
            row["original_note_text"],
            str,
        )
        else ""
    )

    reconstructed_start = pd.NA
    reconstructed_end = pd.NA

    reconstruction_method = "UNRESOLVED"
    reconstruction_status = "UNRESOLVED"

    sentence_occurrence_count = 0
    entity_occurrence_count_in_sentence = 0
    direct_entity_occurrence_count = 0

    selected_sentence_start = pd.NA
    selected_sentence_end = pd.NA


    # Strategy 1:
    # Locating the complete supporting sentence in the note,
    # then locating the entity inside that sentence.


    sentence_locations = find_normalised_occurrences(
        note_text,
        sentence_text,
    )

    sentence_occurrence_count = len(
        sentence_locations
    )

    entity_locations_in_sentence = (
        find_entity_inside_sentence(
            sentence_text,
            entity_text,
        )
    )

    entity_occurrence_count_in_sentence = len(
        entity_locations_in_sentence
    )

    candidate_span_options = []

    if (
        sentence_occurrence_count > 0
        and entity_occurrence_count_in_sentence > 0
    ):

        for sentence_start, sentence_end in sentence_locations:

            located_sentence_text = note_text[
                sentence_start:sentence_end
            ]

            located_entity_matches = (
                find_normalised_occurrences(
                    located_sentence_text,
                    entity_text,
                )
            )

            for local_start, local_end in located_entity_matches:

                candidate_span_options.append(
                    (
                        sentence_start + local_start,
                        sentence_start + local_end,
                        sentence_start,
                        sentence_end,
                    )
                )

    # Removing duplicate candidate spans.
    candidate_span_options = list(
        dict.fromkeys(candidate_span_options)
    )

    if len(candidate_span_options) == 1:

        (
            reconstructed_start,
            reconstructed_end,
            selected_sentence_start,
            selected_sentence_end,
        ) = candidate_span_options[0]

        reconstruction_method = (
            "SUPPORTING_SENTENCE"
        )

        reconstruction_status = (
            "UNIQUE_SENTENCE_ENTITY_MATCH"
        )

    elif len(candidate_span_options) > 1:

        reconstruction_method = (
            "SUPPORTING_SENTENCE"
        )

        reconstruction_status = (
            "AMBIGUOUS_SENTENCE_ENTITY_MATCH"
        )

    else:


        # Strategy 2:
        # Locating the entity directly in the complete note.
        # Accepting only a unique occurrence.


        direct_entity_locations = (
            find_normalised_occurrences(
                note_text,
                entity_text,
            )
        )

        direct_entity_occurrence_count = len(
            direct_entity_locations
        )

        if direct_entity_occurrence_count == 1:

            (
                reconstructed_start,
                reconstructed_end,
            ) = direct_entity_locations[0]

            reconstruction_method = (
                "UNIQUE_ENTITY_IN_NOTE"
            )

            reconstruction_status = (
                "UNIQUE_DIRECT_ENTITY_MATCH"
            )

        elif direct_entity_occurrence_count > 1:

            reconstruction_method = (
                "DIRECT_ENTITY_SEARCH"
            )

            reconstruction_status = (
                "AMBIGUOUS_DIRECT_ENTITY_MATCH"
            )

        else:

            reconstruction_method = (
                "NO_MATCH"
            )

            reconstruction_status = (
                "ENTITY_NOT_FOUND"
            )

    # Validating reconstructed span


    reconstructed_text = pd.NA
    reconstructed_text_matches = False

    if (
        pd.notna(reconstructed_start)
        and pd.notna(reconstructed_end)
    ):

        reconstructed_start = int(
            reconstructed_start
        )

        reconstructed_end = int(
            reconstructed_end
        )

        reconstructed_text = note_text[
            reconstructed_start:
            reconstructed_end
        ]

        reconstructed_text_matches = (
            reconstructed_text
            .strip()
            .casefold()
            ==
            entity_text
            .strip()
            .casefold()
        )

        # A whitespace-normalised comparison is also valid.
        if not reconstructed_text_matches:

            reconstructed_normalised, _ = (
                normalise_with_index_map(
                    reconstructed_text
                )
            )

            entity_normalised, _ = (
                normalise_with_index_map(
                    entity_text
                )
            )

            reconstructed_text_matches = (
                reconstructed_normalised.strip()
                ==
                entity_normalised.strip()
            )

    reconstruction_rows.append(
        {
            "candidate_id":
                candidate_id,

            "note_id":
                note_id,

            "entity_text":
                entity_text,

            "original_start_char":
                row["original_start_char"],

            "original_end_char":
                row["original_end_char"],

            "reconstructed_start_char":
                reconstructed_start,

            "reconstructed_end_char":
                reconstructed_end,

            "reconstructed_text":
                reconstructed_text,

            "reconstructed_text_matches":
                reconstructed_text_matches,

            "reconstruction_method":
                reconstruction_method,

            "reconstruction_status":
                reconstruction_status,

            "sentence_occurrence_count":
                sentence_occurrence_count,

            "entity_occurrence_count_in_sentence":
                entity_occurrence_count_in_sentence,

            "direct_entity_occurrence_count":
                direct_entity_occurrence_count,

            "selected_sentence_start":
                selected_sentence_start,

            "selected_sentence_end":
                selected_sentence_end,

            "reconstructed_context":
                extract_context(
                    note_text,
                    reconstructed_start,
                    reconstructed_end,
                ),
        }
    )


prediction_offset_reconstruction_df = pd.DataFrame(
    reconstruction_rows
)

prediction_offset_reconstruction_df[
    "reconstructed_start_char"
] = safe_integer_series(
    prediction_offset_reconstruction_df[
        "reconstructed_start_char"
    ]
)

prediction_offset_reconstruction_df[
    "reconstructed_end_char"
] = safe_integer_series(
    prediction_offset_reconstruction_df[
        "reconstructed_end_char"
    ]
)


# 3B.4 Summarising reconstruction outcomes


reconstruction_status_summary_df = (
    prediction_offset_reconstruction_df[
        "reconstruction_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("reconstruction_status")
    .reset_index(name="candidate_count")
)

reconstruction_status_summary_df[
    "percentage"
] = (
    100
    * reconstruction_status_summary_df[
        "candidate_count"
    ]
    / len(prediction_offset_reconstruction_df)
).round(2)

print("\nOffset reconstruction status:")

display(
    reconstruction_status_summary_df
)


reconstruction_method_summary_df = (
    prediction_offset_reconstruction_df[
        "reconstruction_method"
    ]
    .value_counts(dropna=False)
    .rename_axis("reconstruction_method")
    .reset_index(name="candidate_count")
)

print("\nOffset reconstruction methods:")

display(
    reconstruction_method_summary_df
)


resolved_mask = (
    prediction_offset_reconstruction_df[
        "reconstructed_start_char"
    ].notna()
    &
    prediction_offset_reconstruction_df[
        "reconstructed_end_char"
    ].notna()
)

resolved_count = int(
    resolved_mask.sum()
)

unresolved_count = int(
    (~resolved_mask).sum()
)

resolved_text_match_count = int(
    prediction_offset_reconstruction_df.loc[
        resolved_mask,
        "reconstructed_text_matches",
    ].sum()
)


reconstruction_summary_df = pd.DataFrame(
    [
        {
            "Measure":
                "Total predictions",
            "Value":
                len(
                    prediction_offset_reconstruction_df
                ),
        },
        {
            "Measure":
                "Offsets reconstructed",
            "Value":
                resolved_count,
        },
        {
            "Measure":
                "Offsets unresolved or ambiguous",
            "Value":
                unresolved_count,
        },
        {
            "Measure":
                "Resolved spans matching entity text",
            "Value":
                resolved_text_match_count,
        },
    ]
)

print("\nOverall reconstruction summary:")

display(
    reconstruction_summary_df
)


# 3B.5 Displaying representative reconstructed candidates

print("\nFirst 20 reconstruction results:")

display(
    prediction_offset_reconstruction_df[
        [
            "candidate_id",
            "note_id",
            "entity_text",
            "original_start_char",
            "original_end_char",
            "reconstructed_start_char",
            "reconstructed_end_char",
            "reconstructed_text",
            "reconstruction_method",
            "reconstruction_status",
            "sentence_occurrence_count",
            "direct_entity_occurrence_count",
        ]
    ].head(20)
)


# 3B.6 Displaying unresolved or ambiguous candidates


unresolved_reconstruction_df = (
    prediction_offset_reconstruction_df.loc[
        ~resolved_mask
    ]
    .copy()
)

print(
    "\nUnresolved or ambiguous candidates:"
)

if unresolved_reconstruction_df.empty:

    print(
        "None — all candidate offsets were reconstructed."
    )

else:

    display(
        unresolved_reconstruction_df[
            [
                "candidate_id",
                "note_id",
                "entity_text",
                "reconstruction_status",
                "sentence_occurrence_count",
                "entity_occurrence_count_in_sentence",
                "direct_entity_occurrence_count",
            ]
        ].head(50)
    )



# 3B.7 Checking whether reconstructed predictions correspond
#      to Candidate_Review rows

candidate_review_offset_check_df = (
    candidate_review_df[
        [
            "candidate_id",
            "note_id",
            "mention_text",
            "start_char",
            "end_char",
            "reviewer_decision",
        ]
    ]
    .copy()
)

candidate_review_offset_check_df = (
    candidate_review_offset_check_df.rename(
        columns={
            "mention_text":
                "review_mention_text",

            "start_char":
                "review_start_char",

            "end_char":
                "review_end_char",
        }
    )
)

candidate_review_offset_check_df[
    "candidate_id"
] = clean_string_series(
    candidate_review_offset_check_df[
        "candidate_id"
    ]
)

candidate_review_offset_check_df[
    "review_start_char"
] = safe_integer_series(
    candidate_review_offset_check_df[
        "review_start_char"
    ]
)

candidate_review_offset_check_df[
    "review_end_char"
] = safe_integer_series(
    candidate_review_offset_check_df[
        "review_end_char"
    ]
)


reconstruction_review_comparison_df = (
    prediction_offset_reconstruction_df.merge(
        candidate_review_offset_check_df,
        on=[
            "candidate_id",
            "note_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

reconstruction_review_comparison_df[
    "reconstructed_equals_review_offsets"
] = (
    reconstruction_review_comparison_df[
        "reconstructed_start_char"
    ]
    ==
    reconstruction_review_comparison_df[
        "review_start_char"
    ]
) & (
    reconstruction_review_comparison_df[
        "reconstructed_end_char"
    ]
    ==
    reconstruction_review_comparison_df[
        "review_end_char"
    ]
)

print(
    "\nComparison with Candidate_Review offsets:"
)

display(
    pd.DataFrame(
        [
            {
                "Measure":
                    "Candidate Review rows linked",

                "Value":
                    int(
                        reconstruction_review_comparison_df[
                            "reviewer_decision"
                        ].notna().sum()
                    ),
            },
            {
                "Measure":
                    "Reconstructed offsets equal review offsets",

                "Value":
                    int(
                        reconstruction_review_comparison_df[
                            "reconstructed_equals_review_offsets"
                        ].sum()
                    ),
            },
        ]
    )
)


# 3B.8 Saving reconstruction outputs

PREDICTION_OFFSET_RECONSTRUCTION_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_document_offset_reconstruction.csv"
)

UNRESOLVED_PREDICTION_OFFSETS_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offsets_unresolved.csv"
)

prediction_offset_reconstruction_df.to_csv(
    PREDICTION_OFFSET_RECONSTRUCTION_FILE,
    index=False,
)

unresolved_reconstruction_df.to_csv(
    UNRESOLVED_PREDICTION_OFFSETS_FILE,
    index=False,
)


PREDICTION_OFFSET_RECONSTRUCTION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_reconstruction_report.json"
)

prediction_offset_reconstruction_report = {
    "total_predictions":
        int(
            len(
                prediction_offset_reconstruction_df
            )
        ),

    "resolved_predictions":
        resolved_count,

    "unresolved_or_ambiguous_predictions":
        unresolved_count,

    "resolved_text_match_count":
        resolved_text_match_count,

    "status_counts": {
        str(key): int(value)
        for key, value in
        prediction_offset_reconstruction_df[
            "reconstruction_status"
        ]
        .value_counts(dropna=False)
        .items()
    },

    "method_counts": {
        str(key): int(value)
        for key, value in
        prediction_offset_reconstruction_df[
            "reconstruction_method"
        ]
        .value_counts(dropna=False)
        .items()
    },

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    PREDICTION_OFFSET_RECONSTRUCTION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        prediction_offset_reconstruction_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# 3B.9 Diagnostic conclusion

print("\n" + "-" * 80)
print("STEP 3B — OFFSET RECONSTRUCTION CONCLUSION")
print("-" * 80)

print(
    f"\nResolved candidates: "
    f"{resolved_count:,} / "
    f"{len(prediction_offset_reconstruction_df):,}"
)

print(
    f"Unresolved or ambiguous candidates: "
    f"{unresolved_count:,}"
)

print(
    f"Resolved spans matching entity text: "
    f"{resolved_text_match_count:,} / "
    f"{resolved_count:,}"
)

if (
    resolved_count
    == len(
        prediction_offset_reconstruction_df
    )
    and resolved_text_match_count
    == resolved_count
):

    print(
        "\nAll document-level prediction offsets were "
        "reconstructed successfully."
    )

    print(
        "The reconstructed offsets can replace the original "
        "local offsets for mention-level evaluation."
    )

elif resolved_count > 0:

    print(
        "\nA proportion of prediction offsets was reconstructed."
    )

    print(
        "Do not discard unresolved candidates. They require "
        "additional deterministic disambiguation or review."
    )

else:

    print(
        "\nNo prediction offsets could be reconstructed."
    )

    print(
        "The extraction-stage offset-generation logic must be "
        "inspected before mention-level evaluation."
    )

print(
    f"\nFull reconstruction table saved to:\n"
    f"{PREDICTION_OFFSET_RECONSTRUCTION_FILE}"
)

print(
    f"\nUnresolved candidates saved to:\n"
    f"{UNRESOLVED_PREDICTION_OFFSETS_FILE}"
)

print(
    f"\nReconstruction report saved to:\n"
    f"{PREDICTION_OFFSET_RECONSTRUCTION_REPORT_FILE}"
)

### 3C. Resolving ambiguous prediction offsets using anchored sequence alignment

The first reconstruction stage recovered document-level offsets for candidates with unique textual locations. Remaining candidates were ambiguous because their mention text or supporting sentence occurred multiple times in the same discharge summary.

To resolve these candidates without consulting gold-standard labels or reviewer decisions, confirmed prediction spans are used as positional anchors. For each note, the mapping between the original extraction offsets and reconstructed document offsets is estimated from uniquely resolved candidates. Ambiguous occurrences are then ranked according to:

1. proximity to the estimated document position;
2. consistency with neighbouring resolved candidates;
3. preservation of original candidate order; and
4. supporting-sentence compatibility.

Candidates remain unresolved when no defensible deterministic location can be selected.

In [ ]:
# 3C. Resolving Ambiguous Prediction Offsets with Anchors


print("-" * 80)
print("STEP 3C — RESOLVE AMBIGUOUS OFFSETS USING POSITIONAL ANCHORS")
print("-" * 80)


# 3C.1 Preparing reconstruction table


anchored_resolution_df = (
    prediction_offset_reconstruction_df.copy()
)

anchored_resolution_df[
    "original_start_char"
] = safe_integer_series(
    anchored_resolution_df[
        "original_start_char"
    ]
)

anchored_resolution_df[
    "original_end_char"
] = safe_integer_series(
    anchored_resolution_df[
        "original_end_char"
    ]
)

anchored_resolution_df[
    "reconstructed_start_char"
] = safe_integer_series(
    anchored_resolution_df[
        "reconstructed_start_char"
    ]
)

anchored_resolution_df[
    "reconstructed_end_char"
] = safe_integer_series(
    anchored_resolution_df[
        "reconstructed_end_char"
    ]
)


# Attaching candidate sequencing and source metadata.
candidate_sequence_columns = [
    "candidate_id",
    "note_id",
    "annotation_note_number",
    "window_number",
    "sentence_text",
]

candidate_sequence_df = candidate_mentions_df[
    candidate_sequence_columns
].copy()

candidate_sequence_df["candidate_id"] = (
    clean_string_series(
        candidate_sequence_df["candidate_id"]
    )
)

candidate_sequence_df["note_id"] = (
    clean_string_series(
        candidate_sequence_df["note_id"]
    )
)

candidate_sequence_df["window_number"] = (
    safe_integer_series(
        candidate_sequence_df["window_number"]
    )
)

candidate_sequence_df["sentence_text"] = (
    candidate_sequence_df["sentence_text"]
    .fillna("")
    .astype(str)
)


anchored_resolution_df = (
    anchored_resolution_df.merge(
        candidate_sequence_df,
        on=[
            "candidate_id",
            "note_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Attaching full note text.
anchored_resolution_df = (
    anchored_resolution_df.merge(
        note_register_df[
            [
                "note_id",
                "original_note_text",
            ]
        ],
        on="note_id",
        how="left",
        validate="many_to_one",
    )
)

anchored_resolution_df[
    "original_note_text"
] = (
    anchored_resolution_df[
        "original_note_text"
    ]
    .fillna("")
    .astype(str)
)


# Preserving the result of the first reconstruction stage.
anchored_resolution_df[
    "stage_3b_resolved"
] = (
    anchored_resolution_df[
        "reconstructed_start_char"
    ].notna()
    &
    anchored_resolution_df[
        "reconstructed_end_char"
    ].notna()
)


# 3C.2 Generating all plausible document-level occurrences


def generate_candidate_occurrences(
    note_text,
    sentence_text,
    entity_text,
):
    """
    Generate plausible document-level spans for one candidate.

    Sentence-supported occurrences are preferred, but direct
    entity occurrences are retained when the sentence cannot
    be aligned.
    """

    occurrence_records = []

    # Sentence-supported occurrences


    sentence_locations = find_normalised_occurrences(
        note_text,
        sentence_text,
    )

    for sentence_start, sentence_end in sentence_locations:

        located_sentence = note_text[
            sentence_start:sentence_end
        ]

        entity_locations = find_normalised_occurrences(
            located_sentence,
            entity_text,
        )

        for local_start, local_end in entity_locations:

            document_start = (
                sentence_start + local_start
            )

            document_end = (
                sentence_start + local_end
            )

            occurrence_records.append(
                {
                    "candidate_start":
                        document_start,

                    "candidate_end":
                        document_end,

                    "candidate_text":
                        note_text[
                            document_start:
                            document_end
                        ],

                    "source":
                        "SUPPORTING_SENTENCE",

                    "sentence_start":
                        sentence_start,

                    "sentence_end":
                        sentence_end,
                }
            )

    # Direct note-level occurrences


    direct_locations = find_normalised_occurrences(
        note_text,
        entity_text,
    )

    for document_start, document_end in direct_locations:

        occurrence_records.append(
            {
                "candidate_start":
                    document_start,

                "candidate_end":
                    document_end,

                "candidate_text":
                    note_text[
                        document_start:
                        document_end
                    ],

                "source":
                    "DIRECT_ENTITY",

                "sentence_start":
                    pd.NA,

                "sentence_end":
                    pd.NA,
            }
        )

    if not occurrence_records:
        return []

    occurrence_df = pd.DataFrame(
        occurrence_records
    )

    # Deduplicate identical spans. When the same span is found
    # both through its sentence and directly, retain the
    # sentence-supported source.
    occurrence_df["source_priority"] = (
        occurrence_df["source"]
        .map(
            {
                "SUPPORTING_SENTENCE": 0,
                "DIRECT_ENTITY": 1,
            }
        )
    )

    occurrence_df = (
        occurrence_df.sort_values(
            [
                "candidate_start",
                "candidate_end",
                "source_priority",
            ]
        )
        .drop_duplicates(
            subset=[
                "candidate_start",
                "candidate_end",
            ],
            keep="first",
        )
        .drop(
            columns=["source_priority"]
        )
        .reset_index(drop=True)
    )

    return occurrence_df.to_dict(
        orient="records"
    )


all_occurrence_options = {}

for _, row in anchored_resolution_df.iterrows():

    all_occurrence_options[
        row["candidate_id"]
    ] = generate_candidate_occurrences(
        note_text=row["original_note_text"],
        sentence_text=row["sentence_text"],
        entity_text=row["entity_text"],
    )


anchored_resolution_df[
    "plausible_occurrence_count"
] = anchored_resolution_df[
    "candidate_id"
].map(
    lambda candidate_id: len(
        all_occurrence_options.get(
            candidate_id,
            [],
        )
    )
)


# 3C.3 Estimating document position from confirmed anchors

def estimate_document_position(
    target_original_offset,
    anchor_original_offsets,
    anchor_document_offsets,
):
    """
    Estimate a document-level position using piecewise-linear
    interpolation or nearest-anchor translation.

    No gold annotations or reviewer decisions are used.
    """

    if (
        pd.isna(target_original_offset)
        or len(anchor_original_offsets) == 0
    ):
        return pd.NA, "NO_ANCHOR"

    target_original_offset = float(
        target_original_offset
    )

    anchor_pairs = sorted(
        zip(
            anchor_original_offsets,
            anchor_document_offsets,
        ),
        key=lambda pair: pair[0],
    )

    # Removing missing values.
    anchor_pairs = [
        (
            float(original_offset),
            float(document_offset),
        )
        for original_offset, document_offset
        in anchor_pairs
        if (
            pd.notna(original_offset)
            and pd.notna(document_offset)
        )
    ]

    if not anchor_pairs:
        return pd.NA, "NO_ANCHOR"

    # One anchor: use translated displacement.
    if len(anchor_pairs) == 1:

        original_anchor, document_anchor = (
            anchor_pairs[0]
        )

        estimated = (
            target_original_offset
            + (
                document_anchor
                - original_anchor
            )
        )

        return estimated, "SINGLE_ANCHOR_TRANSLATION"

    # Identifying anchors immediately before and after target.
    previous_anchor = None
    next_anchor = None

    for original_anchor, document_anchor in anchor_pairs:

        if original_anchor <= target_original_offset:
            previous_anchor = (
                original_anchor,
                document_anchor,
            )

        if (
            original_anchor
            >= target_original_offset
            and next_anchor is None
        ):
            next_anchor = (
                original_anchor,
                document_anchor,
            )

    # Interpolating between two surrounding anchors.
    if (
        previous_anchor is not None
        and next_anchor is not None
        and previous_anchor[0] != next_anchor[0]
    ):

        previous_original, previous_document = (
            previous_anchor
        )

        next_original, next_document = next_anchor

        interpolation_fraction = (
            (
                target_original_offset
                - previous_original
            )
            /
            (
                next_original
                - previous_original
            )
        )

        estimated = (
            previous_document
            + interpolation_fraction
            * (
                next_document
                - previous_document
            )
        )

        return estimated, "BETWEEN_ANCHORS_INTERPOLATION"

    # Before all anchors.
    if previous_anchor is None:

        original_anchor, document_anchor = (
            anchor_pairs[0]
        )

        estimated = (
            target_original_offset
            + (
                document_anchor
                - original_anchor
            )
        )

        return estimated, "FIRST_ANCHOR_TRANSLATION"

    # After all anchors.
    original_anchor, document_anchor = (
        anchor_pairs[-1]
    )

    estimated = (
        target_original_offset
        + (
            document_anchor
            - original_anchor
        )
    )

    return estimated, "LAST_ANCHOR_TRANSLATION"


# 3C.4 Determining neighbouring confirmed spans

def get_neighbouring_resolved_positions(
    note_candidates_df,
    target_index,
):
    """
    Return document positions of nearest resolved candidates
    before and after a target in original extraction order.
    """

    previous_position = pd.NA
    next_position = pd.NA

    for index in range(
        target_index - 1,
        -1,
        -1,
    ):

        value = note_candidates_df.iloc[
            index
        ]["reconstructed_start_char"]

        if pd.notna(value):
            previous_position = int(value)
            break

    for index in range(
        target_index + 1,
        len(note_candidates_df),
    ):

        value = note_candidates_df.iloc[
            index
        ]["reconstructed_start_char"]

        if pd.notna(value):
            next_position = int(value)
            break

    return previous_position, next_position


# 3C.5 Score ambiguous occurrence options


def score_occurrence_option(
    option,
    estimated_position,
    previous_position,
    next_position,
):
    """
    Lower scores are better.

    The score prioritises proximity to the anchor-based
    estimate and penalises violations of neighbouring order.
    """

    candidate_start = int(
        option["candidate_start"]
    )

    score = 0.0
    order_violation = False

    # Distance from estimated document position.
    if pd.notna(estimated_position):

        score += abs(
            candidate_start
            - float(estimated_position)
        )

    else:

        # Explicit high uncertainty when no estimate exists.
        score += 1_000_000

    # Prefer sentence-supported occurrences.
    if option["source"] == "DIRECT_ENTITY":
        score += 25.0

    # Preserving order relative to the previous resolved item.
    if (
        pd.notna(previous_position)
        and candidate_start < int(previous_position)
    ):

        order_violation = True

        score += (
            100_000
            + abs(
                candidate_start
                - int(previous_position)
            )
        )

    # Preserving order relative to the next resolved item.
    if (
        pd.notna(next_position)
        and candidate_start > int(next_position)
    ):

        order_violation = True

        score += (
            100_000
            + abs(
                candidate_start
                - int(next_position)
            )
        )

    return score, order_violation


# 3C.6 Resolving note by note


anchored_resolution_df[
    "anchored_estimated_start"
] = pd.NA

anchored_resolution_df[
    "anchor_estimation_method"
] = pd.NA

anchored_resolution_df[
    "anchored_selected_score"
] = pd.NA

anchored_resolution_df[
    "anchored_second_best_score"
] = pd.NA

anchored_resolution_df[
    "anchored_score_margin"
] = pd.NA

anchored_resolution_df[
    "anchored_resolution_status"
] = pd.NA

anchored_resolution_df[
    "anchored_resolution_source"
] = pd.NA


for note_id, note_indices in (
    anchored_resolution_df.groupby(
        "note_id"
    ).groups.items()
):

    note_df = anchored_resolution_df.loc[
        note_indices
    ].copy()

    # Candidate extraction order is determined primarily from
    # the original local offset, with candidate ID as a stable
    # tie-breaker.
    note_df = note_df.sort_values(
        [
            "original_start_char",
            "original_end_char",
            "candidate_id",
        ]
    ).reset_index()

    confirmed_anchor_mask = (
        note_df["stage_3b_resolved"]
    )

    anchor_original_offsets = (
        note_df.loc[
            confirmed_anchor_mask,
            "original_start_char",
        ]
        .dropna()
        .astype(float)
        .tolist()
    )

    anchor_document_offsets = (
        note_df.loc[
            confirmed_anchor_mask,
            "reconstructed_start_char",
        ]
        .dropna()
        .astype(float)
        .tolist()
    )

    for local_position, note_row in note_df.iterrows():

        global_index = note_row["index"]

        # Preserving already confirmed stage-3B resolutions.
        if note_row["stage_3b_resolved"]:

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_status",
            ] = "PRESERVED_STAGE_3B"

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_source",
            ] = note_row[
                "reconstruction_method"
            ]

            continue

        estimated_position, estimation_method = (
            estimate_document_position(
                target_original_offset=note_row[
                    "original_start_char"
                ],
                anchor_original_offsets=(
                    anchor_original_offsets
                ),
                anchor_document_offsets=(
                    anchor_document_offsets
                ),
            )
        )

        anchored_resolution_df.loc[
            global_index,
            "anchored_estimated_start",
        ] = estimated_position

        anchored_resolution_df.loc[
            global_index,
            "anchor_estimation_method",
        ] = estimation_method

        occurrence_options = all_occurrence_options.get(
            note_row["candidate_id"],
            [],
        )

        if not occurrence_options:

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_status",
            ] = "NO_TEXT_OCCURRENCE"

            continue

        previous_position, next_position = (
            get_neighbouring_resolved_positions(
                note_candidates_df=note_df,
                target_index=local_position,
            )
        )

        scored_options = []

        for option in occurrence_options:

            option_score, order_violation = (
                score_occurrence_option(
                    option=option,
                    estimated_position=estimated_position,
                    previous_position=previous_position,
                    next_position=next_position,
                )
            )

            scored_options.append(
                {
                    **option,
                    "score": option_score,
                    "order_violation":
                        order_violation,
                }
            )

        scored_options = sorted(
            scored_options,
            key=lambda option: (
                option["score"],
                option["candidate_start"],
                option["candidate_end"],
            ),
        )

        best_option = scored_options[0]

        second_best_score = (
            scored_options[1]["score"]
            if len(scored_options) > 1
            else pd.NA
        )

        score_margin = (
            second_best_score
            - best_option["score"]
            if pd.notna(second_best_score)
            else pd.NA
        )

        anchored_resolution_df.loc[
            global_index,
            "anchored_selected_score",
        ] = best_option["score"]

        anchored_resolution_df.loc[
            global_index,
            "anchored_second_best_score",
        ] = second_best_score

        anchored_resolution_df.loc[
            global_index,
            "anchored_score_margin",
        ] = score_margin

        # Accepting only when:
        # - there is an anchor-based positional estimate;
        # - the best candidate preserves neighbour order; and
        # - it is uniquely best or sufficiently separated.
        has_position_estimate = pd.notna(
            estimated_position
        )

        preserves_order = not best_option[
            "order_violation"
        ]

        uniquely_best = (
            len(scored_options) == 1
            or (
                pd.notna(score_margin)
                and float(score_margin) > 0
            )
        )

        if (
            has_position_estimate
            and preserves_order
            and uniquely_best
        ):

            reconstructed_start = int(
                best_option["candidate_start"]
            )

            reconstructed_end = int(
                best_option["candidate_end"]
            )

            anchored_resolution_df.loc[
                global_index,
                "reconstructed_start_char",
            ] = reconstructed_start

            anchored_resolution_df.loc[
                global_index,
                "reconstructed_end_char",
            ] = reconstructed_end

            anchored_resolution_df.loc[
                global_index,
                "reconstructed_text",
            ] = best_option["candidate_text"]

            anchored_resolution_df.loc[
                global_index,
                "reconstructed_text_matches",
            ] = True

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_status",
            ] = "RESOLVED_WITH_POSITIONAL_ANCHOR"

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_source",
            ] = best_option["source"]

        else:

            if not has_position_estimate:

                unresolved_reason = (
                    "NO_POSITIONAL_ANCHOR"
                )

            elif not preserves_order:

                unresolved_reason = (
                    "ORDER_CONSTRAINT_VIOLATION"
                )

            else:

                unresolved_reason = (
                    "NON_UNIQUE_BEST_OPTION"
                )

            anchored_resolution_df.loc[
                global_index,
                "anchored_resolution_status",
            ] = unresolved_reason



# 3C.7 Converting reconstructed offsets to nullable integers


anchored_resolution_df[
    "reconstructed_start_char"
] = safe_integer_series(
    anchored_resolution_df[
        "reconstructed_start_char"
    ]
)

anchored_resolution_df[
    "reconstructed_end_char"
] = safe_integer_series(
    anchored_resolution_df[
        "reconstructed_end_char"
    ]
)


# 3C.8 Revalidation of every resolved span


def validate_final_reconstructed_span(row):
    """
    Confirm that the final reconstructed span matches the
    extracted entity after case and whitespace normalisation.
    """

    start_char = row[
        "reconstructed_start_char"
    ]

    end_char = row[
        "reconstructed_end_char"
    ]

    note_text = row[
        "original_note_text"
    ]

    entity_text = row[
        "entity_text"
    ]

    if (
        pd.isna(start_char)
        or pd.isna(end_char)
    ):
        return False

    extracted_text = note_text[
        int(start_char):int(end_char)
    ]

    extracted_normalised, _ = (
        normalise_with_index_map(
            extracted_text
        )
    )

    entity_normalised, _ = (
        normalise_with_index_map(
            entity_text
        )
    )

    return (
        extracted_normalised.strip()
        ==
        entity_normalised.strip()
    )


anchored_resolution_df[
    "final_span_text_match"
] = anchored_resolution_df.apply(
    validate_final_reconstructed_span,
    axis=1,
)


final_resolved_mask = (
    anchored_resolution_df[
        "reconstructed_start_char"
    ].notna()
    &
    anchored_resolution_df[
        "reconstructed_end_char"
    ].notna()
    &
    anchored_resolution_df[
        "final_span_text_match"
    ]
)


# 3C.9 Summary tables


anchored_status_summary_df = (
    anchored_resolution_df[
        "anchored_resolution_status"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "anchored_resolution_status"
    )
    .reset_index(
        name="candidate_count"
    )
)

anchored_status_summary_df[
    "percentage"
] = (
    100
    * anchored_status_summary_df[
        "candidate_count"
    ]
    / len(anchored_resolution_df)
).round(2)

print("\nAnchored resolution status:")

display(
    anchored_status_summary_df
)


stage_3b_resolved_count = int(
    anchored_resolution_df[
        "stage_3b_resolved"
    ].sum()
)

newly_resolved_count = int(
    (
        anchored_resolution_df[
            "anchored_resolution_status"
        ]
        ==
        "RESOLVED_WITH_POSITIONAL_ANCHOR"
    ).sum()
)

final_resolved_count = int(
    final_resolved_mask.sum()
)

final_unresolved_count = int(
    (
        ~final_resolved_mask
    ).sum()
)


anchored_summary_df = pd.DataFrame(
    [
        {
            "Measure":
                "Total predictions",
            "Value":
                len(anchored_resolution_df),
        },
        {
            "Measure":
                "Resolved in Stage 3B",
            "Value":
                stage_3b_resolved_count,
        },
        {
            "Measure":
                "Newly resolved using anchors",
            "Value":
                newly_resolved_count,
        },
        {
            "Measure":
                "Final resolved predictions",
            "Value":
                final_resolved_count,
        },
        {
            "Measure":
                "Final unresolved predictions",
            "Value":
                final_unresolved_count,
        },
        {
            "Measure":
                "Resolved spans passing text validation",
            "Value":
                int(
                    anchored_resolution_df[
                        "final_span_text_match"
                    ].sum()
                ),
        },
    ]
)

print("\nAnchored reconstruction summary:")

display(
    anchored_summary_df
)



# 3C.10 Inspecting newly resolved candidates

newly_resolved_df = anchored_resolution_df.loc[
    anchored_resolution_df[
        "anchored_resolution_status"
    ]
    ==
    "RESOLVED_WITH_POSITIONAL_ANCHOR"
].copy()

print("\nFirst 30 newly resolved candidates:")

display(
    newly_resolved_df[
        [
            "candidate_id",
            "note_id",
            "entity_text",
            "original_start_char",
            "anchored_estimated_start",
            "reconstructed_start_char",
            "reconstructed_end_char",
            "anchored_resolution_source",
            "anchored_selected_score",
            "anchored_second_best_score",
            "anchored_score_margin",
            "final_span_text_match",
        ]
    ].head(30)
)


# 3C.11 Inspecting candidates still unresolved


final_unresolved_df = anchored_resolution_df.loc[
    ~final_resolved_mask
].copy()

print("\nCandidates still unresolved after anchored alignment:")

if final_unresolved_df.empty:

    print(
        "None — all "
    f"{len(anchored_resolution_df):,} predictions now have "
    "validated document-level offsets."
    )

else:

    display(
        final_unresolved_df[
            [
                "candidate_id",
                "note_id",
                "entity_text",
                "original_start_char",
                "plausible_occurrence_count",
                "anchored_estimated_start",
                "anchored_resolution_status",
                "anchor_estimation_method",
                "anchored_selected_score",
                "anchored_second_best_score",
                "anchored_score_margin",
            ]
        ].head(100)
    )


# 3C.12 Creating final prediction-offset table


FINAL_PREDICTION_OFFSETS_FILE = (
    EVALUATION_DIRECTORY
    / "system_predictions_with_document_offsets.csv"
)

ANCHORED_UNRESOLVED_FILE = (
    EVALUATION_DIRECTORY
    / "system_prediction_offsets_still_unresolved.csv"
)

anchored_resolution_df.to_csv(
    FINAL_PREDICTION_OFFSETS_FILE,
    index=False,
)

final_unresolved_df.to_csv(
    ANCHORED_UNRESOLVED_FILE,
    index=False,
)


# 3C.13 Saving anchored-resolution report

ANCHORED_RESOLUTION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_anchored_offset_resolution_report.json"
)

anchored_resolution_report = {
    "total_predictions":
        int(len(anchored_resolution_df)),

    "stage_3b_resolved":
        stage_3b_resolved_count,

    "newly_resolved_with_positional_anchors":
        newly_resolved_count,

    "final_resolved":
        final_resolved_count,

    "final_unresolved":
        final_unresolved_count,

    "final_text_validated":
        int(
            anchored_resolution_df[
                "final_span_text_match"
            ].sum()
        ),

    "status_counts": {
        str(key): int(value)
        for key, value in
        anchored_resolution_df[
            "anchored_resolution_status"
        ]
        .value_counts(dropna=False)
        .items()
    },

    "output_files": {
        "final_prediction_offsets":
            str(FINAL_PREDICTION_OFFSETS_FILE),

        "still_unresolved":
            str(ANCHORED_UNRESOLVED_FILE),
    },

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    ANCHORED_RESOLUTION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        anchored_resolution_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# 3C.14 Final conclusion

print("\n" + "-" * 80)
print("STEP 3C — ANCHORED RESOLUTION CONCLUSION")
print("-" * 80)

print(
    f"\nStage 3B resolved: "
    f"{stage_3b_resolved_count:,}"
)

print(
    f"Newly resolved using positional anchors: "
    f"{newly_resolved_count:,}"
)

print(
    f"Final resolved predictions: "
    f"{final_resolved_count:,} / "
    f"{len(anchored_resolution_df):,}"
)

print(
    f"Final unresolved predictions: "
    f"{final_unresolved_count:,}"
)

if final_unresolved_count == 0:

    print(
        "\nAll system predictions now have validated "
        "document-level offsets."
    )

    print(
        "The prediction table is ready for mention-level "
        "matching against the gold standard."
    )

else:

    print(
        "\nSome predictions remain positionally ambiguous."
    )

    print(
        "Do not delete them or classify them automatically "
        "as false positives."
    )

    print(
        "They should undergo a final context-based "
        "disambiguation stage before metric calculation."
    )

print(
    f"\nFinal prediction-offset table saved to:\n"
    f"{FINAL_PREDICTION_OFFSETS_FILE}"
)

print(
    f"\nRemaining unresolved candidates saved to:\n"
    f"{ANCHORED_UNRESOLVED_FILE}"
)

print(
    f"\nAnchored-resolution report saved to:\n"
    f"{ANCHORED_RESOLUTION_REPORT_FILE}"
)

In [ ]:
# 3D. Audit Reconstructed Prediction Offsets


print("-" * 80)
print("STEP 3D — AUDIT RECONSTRUCTED PREDICTION OFFSETS")
print("-" * 80)



# 3D.1 Preparing final reconstructed predictions


prediction_offset_audit_df = anchored_resolution_df.copy()

prediction_offset_audit_df[
    "reconstructed_start_char"
] = safe_integer_series(
    prediction_offset_audit_df[
        "reconstructed_start_char"
    ]
)

prediction_offset_audit_df[
    "reconstructed_end_char"
] = safe_integer_series(
    prediction_offset_audit_df[
        "reconstructed_end_char"
    ]
)

required_audit_columns = {
    "candidate_id",
    "note_id",
    "entity_text",
    "original_start_char",
    "reconstructed_start_char",
    "reconstructed_end_char",
    "reconstructed_text",
    "anchored_resolution_status",
}

missing_audit_columns = (
    required_audit_columns
    - set(prediction_offset_audit_df.columns)
)

if missing_audit_columns:
    raise ValueError(
        "The reconstructed prediction table is missing: "
        + ", ".join(sorted(missing_audit_columns))
    )


# 3D.2 Checking duplicate candidate IDs


duplicate_candidate_id_rows_df = (
    prediction_offset_audit_df.loc[
        prediction_offset_audit_df[
            "candidate_id"
        ].duplicated(keep=False)
    ]
    .sort_values("candidate_id")
    .copy()
)

duplicate_candidate_id_count = int(
    duplicate_candidate_id_rows_df[
        "candidate_id"
    ].nunique()
)



# 3D.3 Detecting predictions assigned to identical spans


same_span_group_columns = [
    "note_id",
    "reconstructed_start_char",
    "reconstructed_end_char",
]

prediction_offset_audit_df[
    "same_span_prediction_count"
] = (
    prediction_offset_audit_df.groupby(
        same_span_group_columns,
        dropna=False,
    )["candidate_id"]
    .transform("count")
)

same_span_collision_df = (
    prediction_offset_audit_df.loc[
        prediction_offset_audit_df[
            "same_span_prediction_count"
        ] > 1
    ]
    .sort_values(
        same_span_group_columns
        + ["candidate_id"]
    )
    .copy()
)

same_span_group_count = int(
    same_span_collision_df[
        same_span_group_columns
    ]
    .drop_duplicates()
    .shape[0]
)

same_span_candidate_count = int(
    same_span_collision_df["candidate_id"].nunique()
)



# 3D.4 Checking whether same-span candidates are exact duplicates


duplicate_semantic_group_columns = [
    "note_id",
    "reconstructed_start_char",
    "reconstructed_end_char",
    "complication_category",
    "predicted_assertion",
    "predicted_temporality",
]

available_semantic_columns = [
    column
    for column in duplicate_semantic_group_columns
    if column in prediction_offset_audit_df.columns
]

prediction_offset_audit_df[
    "same_span_same_label_count"
] = (
    prediction_offset_audit_df.groupby(
        available_semantic_columns,
        dropna=False,
    )["candidate_id"]
    .transform("count")
)

exact_duplicate_prediction_df = (
    prediction_offset_audit_df.loc[
        prediction_offset_audit_df[
            "same_span_same_label_count"
        ] > 1
    ]
    .sort_values(
        available_semantic_columns
        + ["candidate_id"]
    )
    .copy()
)

exact_duplicate_group_count = int(
    exact_duplicate_prediction_df[
        available_semantic_columns
    ]
    .drop_duplicates()
    .shape[0]
)

exact_duplicate_candidate_count = int(
    exact_duplicate_prediction_df[
        "candidate_id"
    ].nunique()
)



# 3D.5 Checking extraction-order monotonicity within each note


order_audit_rows = []

for note_id, note_df in prediction_offset_audit_df.groupby(
    "note_id"
):

    ordered_note_df = (
        note_df.sort_values(
            [
                "original_start_char",
                "original_end_char",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    previous_document_start = None
    previous_candidate_id = None

    for _, row in ordered_note_df.iterrows():

        current_document_start = int(
            row["reconstructed_start_char"]
        )

        order_violation = False

        if previous_document_start is not None:

            order_violation = (
                current_document_start
                < previous_document_start
            )

        order_audit_rows.append(
            {
                "note_id":
                    note_id,

                "candidate_id":
                    row["candidate_id"],

                "previous_candidate_id":
                    previous_candidate_id,

                "original_start_char":
                    row["original_start_char"],

                "reconstructed_start_char":
                    current_document_start,

                "previous_reconstructed_start":
                    previous_document_start,

                "order_violation":
                    order_violation,
            }
        )

        previous_document_start = (
            current_document_start
        )

        previous_candidate_id = (
            row["candidate_id"]
        )


prediction_order_audit_df = pd.DataFrame(
    order_audit_rows
)

order_violation_df = (
    prediction_order_audit_df.loc[
        prediction_order_audit_df[
            "order_violation"
        ]
    ]
    .copy()
)

order_violation_count = len(
    order_violation_df
)

notes_with_order_violations = int(
    order_violation_df[
        "note_id"
    ].nunique()
)



# 3D.6 Checking whether multiple candidates share one span but
#      have different labels


label_conflict_rows = []

for span_key, span_df in (
    prediction_offset_audit_df.groupby(
        same_span_group_columns,
        dropna=False,
    )
):

    if len(span_df) <= 1:
        continue

    if (
        "complication_category"
        in span_df.columns
    ):
        category_count = int(
            span_df[
                "complication_category"
            ].nunique(
                dropna=False
            )
        )
    else:
        category_count = 0

    if (
        "predicted_assertion"
        in span_df.columns
    ):
        assertion_count = int(
            span_df[
                "predicted_assertion"
            ].nunique(
                dropna=False
            )
        )
    else:
        assertion_count = 0

    if (
        "predicted_temporality"
        in span_df.columns
    ):
        temporality_count = int(
            span_df[
                "predicted_temporality"
            ].nunique(
                dropna=False
            )
        )
    else:
        temporality_count = 0

    has_label_conflict = any(
        count > 1
        for count in [
            category_count,
            assertion_count,
            temporality_count,
        ]
    )

    label_conflict_rows.append(
        {
            "note_id":
                span_key[0],

            "reconstructed_start_char":
                span_key[1],

            "reconstructed_end_char":
                span_key[2],

            "candidate_count":
                len(span_df),

            "candidate_ids":
                " | ".join(
                    span_df[
                        "candidate_id"
                    ].astype(str)
                ),

            "entity_texts":
                " | ".join(
                    span_df[
                        "entity_text"
                    ].astype(str)
                ),

            "category_count":
                category_count,

            "assertion_count":
                assertion_count,

            "temporality_count":
                temporality_count,

            "has_label_conflict":
                has_label_conflict,
        }
    )


same_span_label_audit_df = pd.DataFrame(
    label_conflict_rows
)

if same_span_label_audit_df.empty:

    same_span_label_audit_df = pd.DataFrame(
        columns=[
            "note_id",
            "reconstructed_start_char",
            "reconstructed_end_char",
            "candidate_count",
            "candidate_ids",
            "entity_texts",
            "category_count",
            "assertion_count",
            "temporality_count",
            "has_label_conflict",
        ]
    )

    conflicting_same_span_df = (
        same_span_label_audit_df.copy()
    )

else:

    conflicting_same_span_df = (
        same_span_label_audit_df.loc[
            same_span_label_audit_df[
                "has_label_conflict"
            ]
        ]
        .copy()
    )


# 3D.7 Comparing duplicate spans with Candidate_Review
#      for diagnosis only


candidate_review_diagnostic_columns = [
    "candidate_id",
    "reviewer_decision",
    "gold_mention_text",
    "gold_category",
    "gold_assertion",
    "gold_temporality",
]

candidate_review_diagnostic_df = (
    candidate_review_df[
        [
            column
            for column in candidate_review_diagnostic_columns
            if column in candidate_review_df.columns
        ]
    ]
    .copy()
)

same_span_review_diagnostic_df = (
    same_span_collision_df.merge(
        candidate_review_diagnostic_df,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )
)

# Candidate_Review is used here only to diagnose duplicate
# reconstruction. It must not define system predictions.



# 3D.8 Summary


offset_audit_summary_df = pd.DataFrame(
    [
        {
            "Measure":
                "Total reconstructed predictions",
            "Value":
                len(prediction_offset_audit_df),
        },
        {
            "Measure":
                "Duplicate candidate IDs",
            "Value":
                duplicate_candidate_id_count,
        },
        {
            "Measure":
                "Identical-span collision groups",
            "Value":
                same_span_group_count,
        },
        {
            "Measure":
                "Candidates involved in span collisions",
            "Value":
                same_span_candidate_count,
        },
        {
            "Measure":
                "Exact duplicate prediction groups",
            "Value":
                exact_duplicate_group_count,
        },
        {
            "Measure":
                "Candidates in exact duplicate groups",
            "Value":
                exact_duplicate_candidate_count,
        },
        {
            "Measure":
                "Extraction-order violations",
            "Value":
                order_violation_count,
        },
        {
            "Measure":
                "Notes with extraction-order violations",
            "Value":
                notes_with_order_violations,
        },
        {
            "Measure":
                "Same-span groups with label conflicts",
            "Value":
                len(conflicting_same_span_df),
        },
    ]
)

print("\nPrediction-offset audit summary:")

display(offset_audit_summary_df)



# 3D.9 Displaying identical-span collisions

print("\nPredictions assigned to identical document spans:")

if same_span_collision_df.empty:

    print("None.")

else:

    display(
        same_span_review_diagnostic_df[
            [
                column
                for column in [
                    "note_id",
                    "reconstructed_start_char",
                    "reconstructed_end_char",
                    "candidate_id",
                    "entity_text",
                    "complication_category",
                    "predicted_assertion",
                    "predicted_temporality",
                    "original_start_char",
                    "anchored_estimated_start",
                    "anchored_resolution_status",
                    "reviewer_decision",
                    "gold_mention_text",
                ]
                if column
                in same_span_review_diagnostic_df.columns
            ]
        ].head(100)
    )


# 3D.10 Displaying order violations


print("\nExtraction-order violations:")

if order_violation_df.empty:

    print("None.")

else:

    display(
        order_violation_df.head(100)
    )


# 3D.11 Displaying same-span label conflicts


print("\nSame-span label conflicts:")

if conflicting_same_span_df.empty:

    print("None.")

else:

    display(
        conflicting_same_span_df.head(100)
    )



# 3D.12 Saving audit files


PREDICTION_OFFSET_AUDIT_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_audit.csv"
)

SAME_SPAN_COLLISIONS_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_same_span_collisions.csv"
)

ORDER_VIOLATIONS_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_order_violations.csv"
)

OFFSET_AUDIT_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_offset_audit_report.json"
)

prediction_offset_audit_df.to_csv(
    PREDICTION_OFFSET_AUDIT_FILE,
    index=False,
)

same_span_review_diagnostic_df.to_csv(
    SAME_SPAN_COLLISIONS_FILE,
    index=False,
)

order_violation_df.to_csv(
    ORDER_VIOLATIONS_FILE,
    index=False,
)


offset_audit_report = {
    "total_predictions":
        int(len(prediction_offset_audit_df)),

    "duplicate_candidate_id_count":
        duplicate_candidate_id_count,

    "same_span_collision_group_count":
        same_span_group_count,

    "same_span_candidate_count":
        same_span_candidate_count,

    "exact_duplicate_group_count":
        exact_duplicate_group_count,

    "exact_duplicate_candidate_count":
        exact_duplicate_candidate_count,

    "order_violation_count":
        int(order_violation_count),

    "notes_with_order_violations":
        notes_with_order_violations,

    "same_span_label_conflict_count":
        int(len(conflicting_same_span_df)),

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    OFFSET_AUDIT_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        offset_audit_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )



# 3D.13 Conclusion

print("\n" + "-" * 80)
print("STEP 3D — OFFSET AUDIT CONCLUSION")
print("-" * 80)

critical_offset_audit_issues = []

if duplicate_candidate_id_count > 0:
    critical_offset_audit_issues.append(
        "Duplicate candidate IDs exist."
    )

if same_span_group_count > 0:
    critical_offset_audit_issues.append(
        "Multiple candidates were assigned to identical spans."
    )

if order_violation_count > 0:
    critical_offset_audit_issues.append(
        "Some reconstructed offsets violate extraction order."
    )

if len(conflicting_same_span_df) > 0:
    critical_offset_audit_issues.append(
        "Some identical spans have conflicting predicted labels."
    )

if critical_offset_audit_issues:

    print("\nThe reconstruction requires a collision-resolution stage.")

    for issue_number, issue in enumerate(
        critical_offset_audit_issues,
        start=1,
    ):
        print(f"{issue_number}. {issue}")

    print(
        "\nDo not calculate mention-level metrics yet."
    )

else:

    print(
        "\nNo duplicate-span, label-conflict or ordering "
        "problems were detected."
    )

    print(
        "The reconstructed prediction table is ready for "
        "mention matching."
    )

print(
    f"\nAudit report saved to:\n"
    f"{OFFSET_AUDIT_REPORT_FILE}"
)

In [ ]:
# 3E. Diagnosing Same-Span Collisions and Order Violations

print("-" * 80)
print("STEP 3E — DIAGNOSE COLLISION GROUPS")
print("-" * 80)


# 3E.1 Identifying all candidates requiring collision review

collision_candidate_ids = set(
    same_span_collision_df[
        "candidate_id"
    ].astype(str)
)

order_violation_candidate_ids = set(
    order_violation_df[
        "candidate_id"
    ].astype(str)
)

previous_order_candidate_ids = set(
    order_violation_df[
        "previous_candidate_id"
    ]
    .dropna()
    .astype(str)
)

diagnostic_candidate_ids = (
    collision_candidate_ids
    | order_violation_candidate_ids
    | previous_order_candidate_ids
)

print(
    f"\nCandidates directly involved in collisions: "
    f"{len(collision_candidate_ids)}"
)

print(
    f"Candidates included in extended diagnosis: "
    f"{len(diagnostic_candidate_ids)}"
)


# 3E.2 Building candidate-level diagnostic table

diagnostic_columns = [
    "candidate_id",
    "note_id",
    "annotation_note_number",
    "window_number",
    "entity_text",
    "sentence_text",
    "complication_category",
    "predicted_assertion",
    "predicted_temporality",
    "original_start_char",
    "original_end_char",
    "reconstructed_start_char",
    "reconstructed_end_char",
    "anchored_estimated_start",
    "anchored_resolution_status",
    "anchored_resolution_source",
    "anchored_selected_score",
    "anchored_second_best_score",
    "anchored_score_margin",
    "plausible_occurrence_count",
    "original_note_text",
]

available_diagnostic_columns = [
    column
    for column in diagnostic_columns
    if column in anchored_resolution_df.columns
]

collision_diagnostic_df = (
    anchored_resolution_df.loc[
        anchored_resolution_df[
            "candidate_id"
        ].astype(str).isin(
            diagnostic_candidate_ids
        ),
        available_diagnostic_columns,
    ]
    .copy()
)


# 3E.3 Adding source-window and reviewer information

candidate_review_columns = [
    "candidate_id",
    "reviewer_decision",
    "gold_mention_text",
    "gold_category",
    "gold_assertion",
    "gold_temporality",
]

available_candidate_review_columns = [
    column
    for column in candidate_review_columns
    if column in candidate_review_df.columns
]

collision_diagnostic_df = (
    collision_diagnostic_df.merge(
        candidate_review_df[
            available_candidate_review_columns
        ],
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )
)


# 3E.4 Utility: candidate context

def extract_numbered_context(
    text,
    start_char,
    end_char,
    width=180,
):
    """
    Return context around a span with the mention marked.
    """

    if (
        not isinstance(text, str)
        or pd.isna(start_char)
        or pd.isna(end_char)
    ):
        return pd.NA

    start_char = int(start_char)
    end_char = int(end_char)

    context_start = max(
        0,
        start_char - width,
    )

    context_end = min(
        len(text),
        end_char + width,
    )

    before = text[
        context_start:start_char
    ]

    mention = text[
        start_char:end_char
    ]

    after = text[
        end_char:context_end
    ]

    return (
        f"[context_start={context_start}] "
        f"{before}"
        f"<<<{mention}>>>"
        f"{after}"
        f" [context_end={context_end}]"
    )


collision_diagnostic_df[
    "assigned_span_context"
] = collision_diagnostic_df.apply(
    lambda row: extract_numbered_context(
        row["original_note_text"],
        row["reconstructed_start_char"],
        row["reconstructed_end_char"],
    ),
    axis=1,
)


# 3E.5 Enumerating every plausible occurrence for each
#      collision candidate

occurrence_diagnostic_rows = []

for _, candidate_row in collision_diagnostic_df.iterrows():

    candidate_id = candidate_row[
        "candidate_id"
    ]

    note_text = candidate_row[
        "original_note_text"
    ]

    options = all_occurrence_options.get(
        candidate_id,
        [],
    )

    for option_number, option in enumerate(
        options,
        start=1,
    ):

        candidate_start = int(
            option["candidate_start"]
        )

        candidate_end = int(
            option["candidate_end"]
        )

        estimated_start = candidate_row.get(
            "anchored_estimated_start",
            pd.NA,
        )

        distance_from_estimate = (
            abs(
                candidate_start
                - float(estimated_start)
            )
            if pd.notna(estimated_start)
            else pd.NA
        )

        occurrence_diagnostic_rows.append(
            {
                "candidate_id":
                    candidate_id,

                "note_id":
                    candidate_row["note_id"],

                "window_number":
                    candidate_row.get(
                        "window_number",
                        pd.NA,
                    ),

                "entity_text":
                    candidate_row["entity_text"],

                "original_start_char":
                    candidate_row[
                        "original_start_char"
                    ],

                "anchored_estimated_start":
                    estimated_start,

                "currently_assigned_start":
                    candidate_row[
                        "reconstructed_start_char"
                    ],

                "option_number":
                    option_number,

                "option_start":
                    candidate_start,

                "option_end":
                    candidate_end,

                "option_source":
                    option["source"],

                "distance_from_estimate":
                    distance_from_estimate,

                "is_current_assignment":
                    (
                        candidate_start
                        ==
                        candidate_row[
                            "reconstructed_start_char"
                        ]
                        and
                        candidate_end
                        ==
                        candidate_row[
                            "reconstructed_end_char"
                        ]
                    ),

                "option_context":
                    extract_numbered_context(
                        note_text,
                        candidate_start,
                        candidate_end,
                    ),
            }
        )


collision_occurrence_options_df = pd.DataFrame(
    occurrence_diagnostic_rows
)

if not collision_occurrence_options_df.empty:

    collision_occurrence_options_df = (
        collision_occurrence_options_df.sort_values(
            [
                "note_id",
                "candidate_id",
                "option_start",
            ]
        )
        .reset_index(drop=True)
    )


# 3E.6 Showing neighbouring candidates in extraction order

neighbourhood_rows = []

collision_note_ids = (
    same_span_collision_df[
        "note_id"
    ]
    .dropna()
    .unique()
)

for note_id in collision_note_ids:

    note_candidate_df = (
        anchored_resolution_df.loc[
            anchored_resolution_df[
                "note_id"
            ] == note_id
        ]
        .sort_values(
            [
                "original_start_char",
                "original_end_char",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    target_positions = note_candidate_df.index[
        note_candidate_df[
            "candidate_id"
        ].astype(str).isin(
            collision_candidate_ids
        )
    ].tolist()

    included_positions = set()

    for target_position in target_positions:

        for position in range(
            max(0, target_position - 2),
            min(
                len(note_candidate_df),
                target_position + 3,
            ),
        ):
            included_positions.add(position)

    for position in sorted(
        included_positions
    ):

        row = note_candidate_df.iloc[
            position
        ]

        neighbourhood_rows.append(
            {
                "note_id":
                    note_id,

                "sequence_position":
                    position,

                "candidate_id":
                    row["candidate_id"],

                "is_collision_candidate":
                    str(row["candidate_id"])
                    in collision_candidate_ids,

                "entity_text":
                    row["entity_text"],

                "window_number":
                    row.get(
                        "window_number",
                        pd.NA,
                    ),

                "original_start_char":
                    row[
                        "original_start_char"
                    ],

                "reconstructed_start_char":
                    row[
                        "reconstructed_start_char"
                    ],

                "reconstructed_end_char":
                    row[
                        "reconstructed_end_char"
                    ],

                "sentence_text":
                    row.get(
                        "sentence_text",
                        pd.NA,
                    ),
            }
        )


collision_neighbourhood_df = pd.DataFrame(
    neighbourhood_rows
)


# 3E.7 Comparing candidates within each collision pair

collision_pair_rows = []

for span_key, span_df in (
    same_span_collision_df.groupby(
        [
            "note_id",
            "reconstructed_start_char",
            "reconstructed_end_char",
        ],
        dropna=False,
    )
):

    span_df = span_df.sort_values(
        [
            "original_start_char",
            "candidate_id",
        ]
    )

    candidate_ids = (
        span_df[
            "candidate_id"
        ].astype(str).tolist()
    )

    candidate_records = (
        collision_diagnostic_df.loc[
            collision_diagnostic_df[
                "candidate_id"
            ].astype(str).isin(
                candidate_ids
            )
        ]
        .set_index("candidate_id")
    )

    if len(candidate_ids) != 2:
        continue

    first_id = candidate_ids[0]
    second_id = candidate_ids[1]

    first_row = candidate_records.loc[
        first_id
    ]

    second_row = candidate_records.loc[
        second_id
    ]

    first_sentence = str(
        first_row.get(
            "sentence_text",
            "",
        )
    )

    second_sentence = str(
        second_row.get(
            "sentence_text",
            "",
        )
    )

    collision_pair_rows.append(
        {
            "note_id":
                span_key[0],

            "assigned_start":
                span_key[1],

            "assigned_end":
                span_key[2],

            "first_candidate_id":
                first_id,

            "second_candidate_id":
                second_id,

            "first_window_number":
                first_row.get(
                    "window_number",
                    pd.NA,
                ),

            "second_window_number":
                second_row.get(
                    "window_number",
                    pd.NA,
                ),

            "original_offset_gap":
                int(
                    second_row[
                        "original_start_char"
                    ]
                    -
                    first_row[
                        "original_start_char"
                    ]
                ),

            "same_sentence_text":
                (
                    first_sentence
                    ==
                    second_sentence
                ),

            "first_sentence_text":
                first_sentence,

            "second_sentence_text":
                second_sentence,

            "first_reviewer_decision":
                first_row.get(
                    "reviewer_decision",
                    pd.NA,
                ),

            "second_reviewer_decision":
                second_row.get(
                    "reviewer_decision",
                    pd.NA,
                ),

            "first_plausible_occurrences":
                first_row.get(
                    "plausible_occurrence_count",
                    pd.NA,
                ),

            "second_plausible_occurrences":
                second_row.get(
                    "plausible_occurrence_count",
                    pd.NA,
                ),
        }
    )


collision_pair_comparison_df = pd.DataFrame(
    collision_pair_rows
)



# 3E.8 Displaying diagnostic outputs


print("\nCollision-pair comparison:")

display(
    collision_pair_comparison_df
)


print("\nAll plausible occurrence options:")

if collision_occurrence_options_df.empty:

    print("No occurrence options were generated.")

else:

    display(
        collision_occurrence_options_df[
            [
                "note_id",
                "candidate_id",
                "window_number",
                "entity_text",
                "original_start_char",
                "anchored_estimated_start",
                "currently_assigned_start",
                "option_number",
                "option_start",
                "option_end",
                "option_source",
                "distance_from_estimate",
                "is_current_assignment",
                "option_context",
            ]
        ]
    )


print("\nNeighbouring candidates in extraction order:")

display(
    collision_neighbourhood_df
)


print("\nAssigned-span contexts:")

display(
    collision_diagnostic_df[
        [
            column
            for column in [
                "note_id",
                "candidate_id",
                "window_number",
                "entity_text",
                "original_start_char",
                "anchored_estimated_start",
                "reconstructed_start_char",
                "reconstructed_end_char",
                "sentence_text",
                "assigned_span_context",
                "reviewer_decision",
                "gold_mention_text",
            ]
            if column
            in collision_diagnostic_df.columns
        ]
    ].sort_values(
        [
            "note_id",
            "original_start_char",
        ]
    )
)



# 3E.9 Saving diagnostic outputs


COLLISION_PAIR_COMPARISON_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_pair_comparison.csv"
)

COLLISION_OCCURRENCE_OPTIONS_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_occurrence_options.csv"
)

COLLISION_NEIGHBOURHOOD_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_neighbourhood.csv"
)

COLLISION_DIAGNOSTIC_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_diagnostic.csv"
)

collision_pair_comparison_df.to_csv(
    COLLISION_PAIR_COMPARISON_FILE,
    index=False,
)

collision_occurrence_options_df.to_csv(
    COLLISION_OCCURRENCE_OPTIONS_FILE,
    index=False,
)

collision_neighbourhood_df.to_csv(
    COLLISION_NEIGHBOURHOOD_FILE,
    index=False,
)

collision_diagnostic_df.drop(
    columns=["original_note_text"],
    errors="ignore",
).to_csv(
    COLLISION_DIAGNOSTIC_FILE,
    index=False,
)



# 3E.10 Conclusion


print("\n" + "-" * 80)
print("STEP 3E — COLLISION DIAGNOSTIC CONCLUSION")
print("-" * 80)

print(
    f"\nCollision groups reviewed: "
    f"{same_span_group_count}"
)

print(
    f"Candidates involved: "
    f"{same_span_candidate_count}"
)

print(
    "\nNo prediction has been removed or reassigned "
    "in this diagnostic step."
)

print(
    "Use the occurrence options, supporting sentences and "
    "neighbouring candidate order to determine whether each "
    "pair represents:"
)

print(
    "1. duplicate extraction of the same mention, or"
)

print(
    "2. two separate mentions incorrectly mapped to one span."
)

print(
    f"\nCollision-pair comparison saved to:\n"
    f"{COLLISION_PAIR_COMPARISON_FILE}"
)

print(
    f"\nOccurrence options saved to:\n"
    f"{COLLISION_OCCURRENCE_OPTIONS_FILE}"
)

print(
    f"\nNeighbourhood table saved to:\n"
    f"{COLLISION_NEIGHBOURHOOD_FILE}"
)

In [ ]:
print(anchored_resolution_df.columns.tolist())

In [ ]:
print("Raw predictions:", len(anchored_resolution_df))
print(
    "Unique candidates:",
    anchored_resolution_df["candidate_id"].nunique(),
)

possible_category_columns = [
    "category",
    "predicted_category",
    "system_category",
    "complication_category",
]

category_column = next(
    (
        column
        for column in possible_category_columns
        if column in anchored_resolution_df.columns
    ),
    None,
)

print("Detected category column:", category_column)

collision_columns = [
    "note_id",
    "reconstructed_start_char",
    "reconstructed_end_char",
]

aggregation_spec = {
    "collision_count": ("candidate_id", "size"),
    "candidate_ids": (
        "candidate_id",
        lambda values: list(values),
    ),
    "entity_texts": (
        "entity_text",
        lambda values: list(values),
    ),
}

if category_column is not None:
    aggregation_spec["categories"] = (
        category_column,
        lambda values: list(values),
    )

collision_audit_df = (
    anchored_resolution_df
    .groupby(
        collision_columns,
        dropna=False,
    )
    .agg(**aggregation_spec)
    .reset_index()
)

actual_collisions_df = collision_audit_df.loc[
    collision_audit_df["collision_count"] > 1
].copy()

print("Collision groups:", len(actual_collisions_df))

display(actual_collisions_df)

In [ ]:
old_manual_ids = [
    "CAND-00039",
    "CAND-00040",
    "CAND-00233",
    "CAND-00235",
    "CAND-00269",
    "CAND-00270",
]

display_columns = [
    "candidate_id",
    "note_id",
    "entity_text",
]

if category_column is not None:
    display_columns.append(category_column)

for column in [
    "original_start_char",
    "original_end_char",
    "reconstructed_start_char",
    "reconstructed_end_char",
    "sentence_text",
]:
    if column in anchored_resolution_df.columns:
        display_columns.append(column)

display(
    anchored_resolution_df.loc[
        anchored_resolution_df[
            "candidate_id"
        ].isin(old_manual_ids),
        display_columns,
    ]
)

In [ ]:
collision_candidate_ids = [
    "CAND-00046",
    "CAND-00047",
    "CAND-00076",
    "CAND-00077",
    "CAND-00322",
    "CAND-00324",
    "CAND-00497",
    "CAND-00498",
]

inspection_columns = [
    column
    for column in [
        "candidate_id",
        "annotation_note_number",
        "note_id",
        "entity_text",
        "window_number",
        "original_start_char",
        "original_end_char",
        "anchored_estimated_start",
        "reconstructed_start_char",
        "reconstructed_end_char",
        "predicted_assertion",
        "predicted_temporality",
        "sentence_text",
    ]
    if column in anchored_resolution_df.columns
]

collision_detail_df = (
    anchored_resolution_df.loc[
        anchored_resolution_df["candidate_id"].isin(
            collision_candidate_ids
        ),
        inspection_columns,
    ]
    .sort_values(
        [
            "note_id",
            "original_start_char",
            "candidate_id",
        ]
    )
    .reset_index(drop=True)
)

display(collision_detail_df)

In [ ]:
for note_id, group_df in collision_detail_df.groupby("note_id"):

    print("\n" + "=" * 100)
    print("NOTE:", note_id)
    print("=" * 100)

    for _, row in group_df.iterrows():

        print("\nCandidate:", row["candidate_id"])
        print("Entity:", repr(row["entity_text"]))

        if "window_number" in row.index:
            print("Window:", row["window_number"])

        print(
            "Original offsets:",
            row.get("original_start_char"),
            row.get("original_end_char"),
        )

        print(
            "Reconstructed offsets:",
            row.get("reconstructed_start_char"),
            row.get("reconstructed_end_char"),
        )

        print("Sentence:")
        print(row.get("sentence_text"))


### 3F. Resolve reconstructed-span collisions

Three reconstructed-span collisions were examined using candidate order, source
window, supporting sentence, positional estimates and all plausible occurrences.

Two collisions represented separate mentions that had been assigned to the same
document position. Their offsets were corrected to preserve extraction order and
supporting context.

The remaining collision represented duplicate extraction of one clinical mention.
The raw candidate record is preserved for provenance, but one representative is
retained in the canonical mention-level prediction set. No gold-standard annotation
was used to determine whether a system prediction was correct.

In [ ]:
# 3F. Resolving Collisions and Create Canonical Predictions

print("-" * 80)
print("STEP 3F — RESOLVE OFFSET COLLISIONS")
print("-" * 80)


# 3F.1 Preserving the complete raw prediction table

collision_resolved_df = anchored_resolution_df.copy()

collision_resolved_df[
    "reconstructed_start_char"
] = safe_integer_series(
    collision_resolved_df[
        "reconstructed_start_char"
    ]
)

collision_resolved_df[
    "reconstructed_end_char"
] = safe_integer_series(
    collision_resolved_df[
        "reconstructed_end_char"
    ]
)

collision_resolved_df[
    "collision_resolution_action"
] = "UNCHANGED"

collision_resolved_df[
    "collision_resolution_reason"
] = pd.NA

collision_resolved_df[
    "duplicate_of_candidate_id"
] = pd.NA

collision_resolved_df[
    "include_in_canonical_evaluation"
] = True


# 3F.2 Helper for assigning a validated document span

def assign_reconstructed_span(
    dataframe,
    candidate_id,
    expected_note_id,
    new_start,
    new_end,
    reason,
):
    """
    Assign a manually audited document-level span.

    The span is accepted only when:
    - the candidate exists exactly once;
    - it belongs to the expected note; and
    - the extracted note text matches the candidate entity.
    """

    candidate_mask = (
        dataframe["candidate_id"].astype(str)
        == str(candidate_id)
    )

    matching_count = int(candidate_mask.sum())

    if matching_count != 1:
        raise ValueError(
            f"{candidate_id}: expected exactly one row, "
            f"found {matching_count}."
        )

    row_index = dataframe.index[
        candidate_mask
    ][0]

    actual_note_id = str(
        dataframe.loc[
            row_index,
            "note_id",
        ]
    )

    if actual_note_id != str(expected_note_id):
        raise ValueError(
            f"{candidate_id}: expected note "
            f"{expected_note_id}, found {actual_note_id}."
        )

    note_text = str(
        dataframe.loc[
            row_index,
            "original_note_text",
        ]
    )

    entity_text = str(
        dataframe.loc[
            row_index,
            "entity_text",
        ]
    )

    extracted_text = note_text[
        int(new_start):int(new_end)
    ]

    extracted_normalised, _ = (
        normalise_with_index_map(
            extracted_text
        )
    )

    entity_normalised, _ = (
        normalise_with_index_map(
            entity_text
        )
    )

    if (
        extracted_normalised.strip()
        != entity_normalised.strip()
    ):
        raise ValueError(
            f"{candidate_id}: proposed span "
            f"{new_start}:{new_end} gives "
            f"{extracted_text!r}, not {entity_text!r}."
        )

    dataframe.loc[
        row_index,
        "reconstructed_start_char",
    ] = int(new_start)

    dataframe.loc[
        row_index,
        "reconstructed_end_char",
    ] = int(new_end)

    dataframe.loc[
        row_index,
        "reconstructed_text",
    ] = extracted_text

    dataframe.loc[
        row_index,
        "reconstructed_text_matches",
    ] = True

    dataframe.loc[
        row_index,
        "final_span_text_match",
    ] = True

    dataframe.loc[
        row_index,
        "collision_resolution_action",
    ] = "OFFSET_REASSIGNED"

    dataframe.loc[
        row_index,
        "collision_resolution_reason",
    ] = reason


# 3F.3 Resolving separate mentions incorrectly mapped to the same span

import re


def find_case_insensitive_occurrences(
    text,
    entity_text,
):
    """
    Return every exact case-insensitive occurrence of entity_text
    in the original note.
    """

    pattern = re.compile(
        re.escape(str(entity_text)),
        flags=re.IGNORECASE,
    )

    return [
        (match.start(), match.end(), match.group())
        for match in pattern.finditer(str(text))
    ]


def resolve_collision_group_to_unique_occurrences(
    dataframe,
    candidate_ids,
    reason,
):
    """
    Assign each colliding candidate to a distinct textual occurrence.

    Candidates are processed according to their anchored estimated
    position. Each candidate receives the closest unused exact
    case-insensitive occurrence in the original note.
    """

    candidate_mask = (
        dataframe["candidate_id"]
        .astype(str)
        .isin([str(value) for value in candidate_ids])
    )

    group_df = dataframe.loc[candidate_mask].copy()

    if len(group_df) != len(candidate_ids):
        raise ValueError(
            "Expected "
            f"{len(candidate_ids)} candidates, found {len(group_df)} "
            f"for {candidate_ids}."
        )

    if group_df["note_id"].nunique() != 1:
        raise ValueError(
            f"Candidates do not belong to one note: {candidate_ids}"
        )

    note_id = str(group_df["note_id"].iloc[0])
    note_text = str(group_df["original_note_text"].iloc[0])

    group_df = group_df.sort_values(
        [
            "anchored_estimated_start",
            "original_start_char",
            "candidate_id",
        ],
        na_position="last",
    )

    # All candidates in these collision groups represent the same
    # lexical expression, differing only in case.
    search_entity = str(group_df["entity_text"].iloc[0])

    occurrences = find_case_insensitive_occurrences(
        text=note_text,
        entity_text=search_entity,
    )

    if len(occurrences) < len(group_df):
        raise ValueError(
            f"{note_id}: found only {len(occurrences)} occurrences "
            f"of {search_entity!r} for {len(group_df)} candidates."
        )

    unused_occurrences = list(occurrences)
    assignments = []

    for _, row in group_df.iterrows():

        candidate_id = str(row["candidate_id"])

        anchor_value = row.get(
            "anchored_estimated_start",
            pd.NA,
        )

        if pd.isna(anchor_value):
            anchor_value = row["reconstructed_start_char"]

        anchor_value = float(anchor_value)

        best_occurrence = min(
            unused_occurrences,
            key=lambda occurrence: abs(
                occurrence[0] - anchor_value
            ),
        )

        new_start, new_end, matched_text = best_occurrence

        assign_reconstructed_span(
            dataframe=dataframe,
            candidate_id=candidate_id,
            expected_note_id=note_id,
            new_start=new_start,
            new_end=new_end,
            reason=reason,
        )

        unused_occurrences.remove(best_occurrence)

        assignments.append(
            {
                "candidate_id": candidate_id,
                "note_id": note_id,
                "entity_text": row["entity_text"],
                "anchored_estimated_start": anchor_value,
                "assigned_start": new_start,
                "assigned_end": new_end,
                "assigned_text": matched_text,
            }
        )

    return pd.DataFrame(assignments)


collision_assignment_tables = []


# GI bleed occurs twice in the same medication paragraph.

collision_assignment_tables.append(
    resolve_collision_group_to_unique_occurrences(
        dataframe=collision_resolved_df,
        candidate_ids=[
            "CAND-00046",
            "CAND-00047",
        ],
        reason=(
            "Distinct GI bleed mentions in the same medication "
            "management paragraph; assigned to separate exact "
            "document occurrences."
        ),
    )
)


# Bacteremia occurs twice in the same sentence:
# heading mention and S. aureus bacteremia mention.

collision_assignment_tables.append(
    resolve_collision_group_to_unique_occurrences(
        dataframe=collision_resolved_df,
        candidate_ids=[
            "CAND-00076",
            "CAND-00077",
        ],
        reason=(
            "Distinct bacteremia mentions in the same sentence; "
            "assigned to separate exact document occurrences."
        ),
    )
)


# Sepsis occurs twice in the discharge-diagnosis sequence.

collision_assignment_tables.append(
    resolve_collision_group_to_unique_occurrences(
        dataframe=collision_resolved_df,
        candidate_ids=[
            "CAND-00322",
            "CAND-00324",
        ],
        reason=(
            "Distinct sepsis mentions in the discharge-diagnosis "
            "sequence; assigned to separate exact document "
            "occurrences."
        ),
    )
)


# Melena occurs in two different clinical sections.

collision_assignment_tables.append(
    resolve_collision_group_to_unique_occurrences(
        dataframe=collision_resolved_df,
        candidate_ids=[
            "CAND-00497",
            "CAND-00498",
        ],
        reason=(
            "Distinct melena mentions in separate clinical "
            "sections; assigned to separate exact document "
            "occurrences."
        ),
    )
)


collision_occurrence_assignments_df = pd.concat(
    collision_assignment_tables,
    ignore_index=True,
)

print("\nCollision occurrence assignments:")

display(
    collision_occurrence_assignments_df.sort_values(
        [
            "note_id",
            "assigned_start",
        ]
    )
)


# No duplicate candidates were identified in the 546-candidate dataset.

collision_resolved_df[
    "duplicate_of_candidate_id"
] = pd.NA

collision_resolved_df[
    "include_in_canonical_evaluation"
] = True


# 3F.5 Revalidating all 546 raw candidate records

def validate_collision_resolved_span(row):

    start_char = row[
        "reconstructed_start_char"
    ]

    end_char = row[
        "reconstructed_end_char"
    ]

    if (
        pd.isna(start_char)
        or pd.isna(end_char)
    ):
        return False

    note_text = str(
        row["original_note_text"]
    )

    entity_text = str(
        row["entity_text"]
    )

    extracted_text = note_text[
        int(start_char):int(end_char)
    ]

    extracted_normalised, _ = (
        normalise_with_index_map(
            extracted_text
        )
    )

    entity_normalised, _ = (
        normalise_with_index_map(
            entity_text
        )
    )

    return (
        extracted_normalised.strip()
        ==
        entity_normalised.strip()
    )


collision_resolved_df[
    "post_collision_text_match"
] = (
    collision_resolved_df.apply(
        validate_collision_resolved_span,
        axis=1,
    )
)

invalid_post_collision_df = (
    collision_resolved_df.loc[
        ~collision_resolved_df[
            "post_collision_text_match"
        ]
    ]
    .copy()
)

if not invalid_post_collision_df.empty:
    raise ValueError(
        "At least one collision-resolved span failed "
        "text validation."
    )


# 3F.6 Creating canonical mention-level prediction table

canonical_system_predictions_df = (
    collision_resolved_df.loc[
        collision_resolved_df[
            "include_in_canonical_evaluation"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

expected_raw_prediction_count = len(
    anchored_resolution_df
)

expected_canonical_prediction_count = (
    expected_raw_prediction_count
)

if len(collision_resolved_df) != expected_raw_prediction_count:
    raise ValueError(
        "Unexpected raw prediction count: "
        f"{len(collision_resolved_df)}."
    )

if (
    len(canonical_system_predictions_df)
    != expected_canonical_prediction_count
):
    raise ValueError(
        "Unexpected canonical prediction count: "
        f"{len(canonical_system_predictions_df)}."
    )


# 3F.7 Repeating collision audit on canonical predictions

canonical_span_columns = [
    "note_id",
    "reconstructed_start_char",
    "reconstructed_end_char",
]

canonical_system_predictions_df[
    "canonical_same_span_count"
] = (
    canonical_system_predictions_df.groupby(
        canonical_span_columns,
        dropna=False,
    )["candidate_id"]
    .transform("count")
)

remaining_canonical_collisions_df = (
    canonical_system_predictions_df.loc[
        canonical_system_predictions_df[
            "canonical_same_span_count"
        ] > 1
    ]
    .copy()
)


# 3F.8 Repeating extraction-order audit

post_resolution_order_rows = []

for note_id, note_df in collision_resolved_df.groupby(
    "note_id"
):

    ordered_note_df = (
        note_df.sort_values(
            [
                "original_start_char",
                "original_end_char",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    previous_start = None
    previous_candidate_id = None

    for _, row in ordered_note_df.iterrows():

        current_start = int(
            row[
                "reconstructed_start_char"
            ]
        )

        is_collapsed_duplicate = (
            row[
                "collision_resolution_action"
            ]
            == "DUPLICATE_COLLAPSED"
        )

        order_violation = (
            previous_start is not None
            and current_start < previous_start
            and not is_collapsed_duplicate
        )

        post_resolution_order_rows.append(
            {
                "note_id":
                    note_id,

                "candidate_id":
                    row["candidate_id"],

                "previous_candidate_id":
                    previous_candidate_id,

                "original_start_char":
                    row["original_start_char"],

                "reconstructed_start_char":
                    current_start,

                "previous_reconstructed_start":
                    previous_start,

                "is_collapsed_duplicate":
                    is_collapsed_duplicate,

                "order_violation":
                    order_violation,
            }
        )

        if not is_collapsed_duplicate:
            previous_start = current_start
            previous_candidate_id = (
                row["candidate_id"]
            )


post_resolution_order_audit_df = (
    pd.DataFrame(
        post_resolution_order_rows
    )
)

remaining_order_violations_df = (
    post_resolution_order_audit_df.loc[
        post_resolution_order_audit_df[
            "order_violation"
        ]
    ]
    .copy()
)


# 3F.9 Displaying resolution records

resolution_candidate_ids = [
    "CAND-00046",
    "CAND-00047",
    "CAND-00076",
    "CAND-00077",
    "CAND-00322",
    "CAND-00324",
    "CAND-00497",
    "CAND-00498",
]

collision_resolution_records_df = (
    collision_resolved_df.loc[
        collision_resolved_df[
            "candidate_id"
        ].isin(
            resolution_candidate_ids
        ),
        [
            "candidate_id",
            "note_id",
            "entity_text",
            "window_number",
            "original_start_char",
            "anchored_estimated_start",
            "reconstructed_start_char",
            "reconstructed_end_char",
            "collision_resolution_action",
            "duplicate_of_candidate_id",
            "include_in_canonical_evaluation",
            "post_collision_text_match",
            "collision_resolution_reason",
        ],
    ]
    .sort_values(
        [
            "note_id",
            "original_start_char",
        ]
    )
)

print("\nCollision-resolution records:")

display(
    collision_resolution_records_df
)


# 3F.10 Final summary


collision_resolution_summary_df = pd.DataFrame(
    [
        {
            "Measure":
                "Raw candidate records preserved",
            "Value":
                len(collision_resolved_df),
        },
        {
            "Measure":
                "Offsets reassigned",
            "Value":
                int(
                    (
                        collision_resolved_df[
                            "collision_resolution_action"
                        ]
                        == "OFFSET_REASSIGNED"
                    ).sum()
                ),
        },
        {
            "Measure":
                "Confirmed duplicate records",
            "Value":
                int(
                    (
                        collision_resolved_df[
                            "collision_resolution_action"
                        ]
                        == "DUPLICATE_COLLAPSED"
                    ).sum()
                ),
        },
        {
            "Measure":
                "Canonical mention-level predictions",
            "Value":
                len(
                    canonical_system_predictions_df
                ),
        },
        {
            "Measure":
                "Remaining canonical span collisions",
            "Value":
                int(
                    remaining_canonical_collisions_df[
                        canonical_span_columns
                    ]
                    .drop_duplicates()
                    .shape[0]
                ),
        },
        {
            "Measure":
                "Remaining extraction-order violations",
            "Value":
                len(
                    remaining_order_violations_df
                ),
        },
        {
            "Measure":
                "Raw spans passing text validation",
            "Value":
                int(
                    collision_resolved_df[
                        "post_collision_text_match"
                    ].sum()
                ),
        },
    ]
)

print("\nCollision-resolution summary:")

display(
    collision_resolution_summary_df
)

print("\nRemaining canonical span collisions:")

if remaining_canonical_collisions_df.empty:
    print("None.")
else:
    display(
        remaining_canonical_collisions_df[
            [
                "candidate_id",
                "note_id",
                "entity_text",
                "reconstructed_start_char",
                "reconstructed_end_char",
            ]
        ]
    )

print("\nRemaining extraction-order violations:")

if remaining_order_violations_df.empty:
    print("None.")
else:
    display(
        remaining_order_violations_df
    )


# 3F.11 Saving raw and canonical outputs

COLLISION_RESOLVED_RAW_FILE = (
    EVALUATION_DIRECTORY
    / "system_predictions_collision_resolved_raw_546.csv"
)

CANONICAL_SYSTEM_PREDICTIONS_FILE = (
    EVALUATION_DIRECTORY
    / "system_predictions_canonical_mentions_546.csv"
)

COLLISION_RESOLUTION_RECORDS_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_resolution_records.csv"
)

COLLISION_RESOLUTION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_collision_resolution_report.json"
)

collision_resolved_df.to_csv(
    COLLISION_RESOLVED_RAW_FILE,
    index=False,
)

canonical_system_predictions_df.to_csv(
    CANONICAL_SYSTEM_PREDICTIONS_FILE,
    index=False,
)

collision_resolution_records_df.to_csv(
    COLLISION_RESOLUTION_RECORDS_FILE,
    index=False,
)


collision_resolution_report = {
    "raw_candidate_count":
        int(len(collision_resolved_df)),

    "canonical_prediction_count":
        int(
            len(
                canonical_system_predictions_df
            )
        ),

    "offset_reassignment_count":
        int(
            (
                collision_resolved_df[
                    "collision_resolution_action"
                ]
                == "OFFSET_REASSIGNED"
            ).sum()
        ),

    "collapsed_duplicate_count":
        int(
            (
                collision_resolved_df[
                    "collision_resolution_action"
                ]
                == "DUPLICATE_COLLAPSED"
            ).sum()
        ),

    "remaining_canonical_collision_groups":
        int(
            remaining_canonical_collisions_df[
                canonical_span_columns
            ]
            .drop_duplicates()
            .shape[0]
        ),

    "remaining_order_violations":
        int(
            len(
                remaining_order_violations_df
            )
        ),

    "raw_text_validated_count":
        int(
            collision_resolved_df[
                "post_collision_text_match"
            ].sum()
        ),

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    COLLISION_RESOLUTION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        collision_resolution_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# 3F.12 Final conclusion

print("\n" + "-" * 80)
print("STEP 3F — COLLISION RESOLUTION CONCLUSION")
print("-" * 80)

ready_for_matching = (
    remaining_canonical_collisions_df.empty
    and remaining_order_violations_df.empty
    and collision_resolved_df[
        "post_collision_text_match"
    ].all()
    and len(
        canonical_system_predictions_df
    ) == expected_canonical_prediction_count
)

print(
    f"\nRaw candidate records preserved: "
    f"{len(collision_resolved_df):,}"
)

print(
    f"Canonical mention predictions: "
    f"{len(canonical_system_predictions_df):,}"
)

if ready_for_matching:

    print(
        "\nAll reconstructed offsets now pass text, "
        "collision and extraction-order validation."
    )

    print(
        f"The canonical {len(canonical_system_predictions_df):,}-prediction table is ready for "
        "one-to-one mention matching against the gold standard."
    )

else:

    print(
        "\nOne or more validation checks still failed."
    )

    print(
        "Do not calculate mention-level metrics until the "
        "displayed issue has been resolved."
    )

print(
    f"\nRaw collision-resolved table saved to:\n"
    f"{COLLISION_RESOLVED_RAW_FILE}"
)

print(
    f"\nCanonical prediction table saved to:\n"
    f"{CANONICAL_SYSTEM_PREDICTIONS_FILE}"
)

print(
    f"\nResolution report saved to:\n"
    f"{COLLISION_RESOLUTION_REPORT_FILE}"
)

## 4. One-to-one mention matching and evaluation

The reconstructed canonical prediction table is now compared against the final
gold-standard mention annotations.

Evaluation will be performed at several levels:

1. **Strict span matching** — prediction and gold mention have identical document
   offsets.
2. **Relaxed overlap matching** — prediction and gold mention overlap in the same
   note.
3. **Category-aware matching** — matched spans must also have the same normalized
   complication category.
4. **Attribute evaluation** — assertion and temporality are evaluated only among
   appropriately matched mentions.

All matching will be one-to-one. A gold mention can match at most one prediction,
and a prediction can match at most one gold mention.

In [ ]:
# 4A. Confirm Gold and Prediction Evaluation Schemas

print("-" * 80)
print("STEP 4A — CONFIRM EVALUATION TABLE SCHEMAS")
print("-" * 80)


# 4A.1 Selecting final evaluation tables

gold_evaluation_df = gold_standard_df.copy()

prediction_evaluation_df = (
    canonical_system_predictions_df.copy()
)


# 4A.2 Basic record counts

evaluation_table_summary_df = pd.DataFrame(
    [
        {
            "Table":
                "Gold-standard mentions",
            "Rows":
                len(gold_evaluation_df),
            "Unique notes":
                gold_evaluation_df[
                    "note_id"
                ].nunique(),
        },
        {
            "Table":
                "Canonical system predictions",
            "Rows":
                len(prediction_evaluation_df),
            "Unique notes":
                prediction_evaluation_df[
                    "note_id"
                ].nunique(),
        },
    ]
)

print("\nEvaluation-table summary:")

display(
    evaluation_table_summary_df
)


# 4A.3 Displaying complete column inventories

gold_column_inventory_df = pd.DataFrame(
    {
        "column_position":
            range(
                len(
                    gold_evaluation_df.columns
                )
            ),

        "column_name":
            gold_evaluation_df.columns,

        "dtype": [
            str(dtype)
            for dtype in
            gold_evaluation_df.dtypes
        ],

        "non_missing_count": [
            int(
                gold_evaluation_df[
                    column
                ].notna().sum()
            )
            for column in
            gold_evaluation_df.columns
        ],

        "unique_value_count": [
            int(
                gold_evaluation_df[
                    column
                ].nunique(
                    dropna=True
                )
            )
            for column in
            gold_evaluation_df.columns
        ],
    }
)

prediction_column_inventory_df = pd.DataFrame(
    {
        "column_position":
            range(
                len(
                    prediction_evaluation_df.columns
                )
            ),

        "column_name":
            prediction_evaluation_df.columns,

        "dtype": [
            str(dtype)
            for dtype in
            prediction_evaluation_df.dtypes
        ],

        "non_missing_count": [
            int(
                prediction_evaluation_df[
                    column
                ].notna().sum()
            )
            for column in
            prediction_evaluation_df.columns
        ],

        "unique_value_count": [
            int(
                prediction_evaluation_df[
                    column
                ].nunique(
                    dropna=True
                )
            )
            for column in
            prediction_evaluation_df.columns
        ],
    }
)

print("\nGold-standard column inventory:")

display(
    gold_column_inventory_df
)

print("\nPrediction column inventory:")

display(
    prediction_column_inventory_df
)


# 4A.4 Locating likely evaluation fields

def find_likely_columns(
    dataframe,
    search_terms,
):
    """
    Return columns whose names contain any supplied term.
    """

    matching_columns = []

    for column in dataframe.columns:

        normalized_column = (
            str(column)
            .strip()
            .lower()
        )

        if any(
            term.lower()
            in normalized_column
            for term in search_terms
        ):
            matching_columns.append(
                column
            )

    return matching_columns


field_search_groups = {
    "identifier": [
        "id",
        "mention",
        "candidate",
    ],

    "note": [
        "note",
    ],

    "start offset": [
        "start",
    ],

    "end offset": [
        "end",
    ],

    "mention text": [
        "mention_text",
        "entity_text",
        "span_text",
        "reconstructed_text",
    ],

    "category": [
        "category",
        "complication",
        "label",
    ],

    "assertion": [
        "assertion",
        "negation",
    ],

    "temporality": [
        "temporality",
        "temporal",
        "historical",
    ],
}


likely_field_rows = []

for field_group, search_terms in (
    field_search_groups.items()
):

    likely_field_rows.append(
        {
            "field_group":
                field_group,

            "gold_candidates":
                " | ".join(
                    find_likely_columns(
                        gold_evaluation_df,
                        search_terms,
                    )
                ),

            "prediction_candidates":
                " | ".join(
                    find_likely_columns(
                        prediction_evaluation_df,
                        search_terms,
                    )
                ),
        }
    )


likely_evaluation_fields_df = pd.DataFrame(
    likely_field_rows
)

print("\nLikely evaluation fields:")

display(
    likely_evaluation_fields_df
)


# 4A.5 Showing representative records

print("\nFirst five gold-standard records:")

display(
    gold_evaluation_df.head()
)

print("\nFirst five canonical prediction records:")

display(
    prediction_evaluation_df.head()
)


# 4A.6 Inspecting likely category and attribute values

gold_category_candidates = (
    find_likely_columns(
        gold_evaluation_df,
        [
            "category",
            "complication",
            "label",
        ],
    )
)

prediction_category_candidates = (
    find_likely_columns(
        prediction_evaluation_df,
        [
            "category",
            "complication",
            "label",
        ],
    )
)

gold_assertion_candidates = (
    find_likely_columns(
        gold_evaluation_df,
        [
            "assertion",
            "negation",
        ],
    )
)

prediction_assertion_candidates = (
    find_likely_columns(
        prediction_evaluation_df,
        [
            "assertion",
            "negation",
        ],
    )
)

gold_temporality_candidates = (
    find_likely_columns(
        gold_evaluation_df,
        [
            "temporality",
            "temporal",
            "historical",
        ],
    )
)

prediction_temporality_candidates = (
    find_likely_columns(
        prediction_evaluation_df,
        [
            "temporality",
            "temporal",
            "historical",
        ],
    )
)


def display_candidate_value_counts(
    dataframe,
    columns,
    table_name,
    maximum_columns=10,
):
    """
    Display value counts for likely evaluation columns.
    """

    if not columns:
        print(
            f"\nNo candidate columns found for "
            f"{table_name}."
        )
        return

    for column in columns[
        :maximum_columns
    ]:

        print(
            f"\n{table_name} — {column}:"
        )

        display(
            dataframe[
                column
            ]
            .value_counts(
                dropna=False
            )
            .rename_axis(
                column
            )
            .reset_index(
                name="count"
            )
            .head(30)
        )


display_candidate_value_counts(
    dataframe=gold_evaluation_df,
    columns=gold_category_candidates,
    table_name="Gold category candidate",
)

display_candidate_value_counts(
    dataframe=prediction_evaluation_df,
    columns=prediction_category_candidates,
    table_name="Prediction category candidate",
)

display_candidate_value_counts(
    dataframe=gold_evaluation_df,
    columns=gold_assertion_candidates,
    table_name="Gold assertion candidate",
)

display_candidate_value_counts(
    dataframe=prediction_evaluation_df,
    columns=prediction_assertion_candidates,
    table_name="Prediction assertion candidate",
)

display_candidate_value_counts(
    dataframe=gold_evaluation_df,
    columns=gold_temporality_candidates,
    table_name="Gold temporality candidate",
)

display_candidate_value_counts(
    dataframe=prediction_evaluation_df,
    columns=prediction_temporality_candidates,
    table_name="Prediction temporality candidate",
)


# 4A.7 Verifying mandatory structural fields

gold_structural_candidates = {
    "note_id":
        find_likely_columns(
            gold_evaluation_df,
            ["note_id"],
        ),

    "start":
        find_likely_columns(
            gold_evaluation_df,
            ["start"],
        ),

    "end":
        find_likely_columns(
            gold_evaluation_df,
            ["end"],
        ),
}

prediction_structural_candidates = {
    "note_id":
        find_likely_columns(
            prediction_evaluation_df,
            ["note_id"],
        ),

    "start":
        find_likely_columns(
            prediction_evaluation_df,
            ["reconstructed_start"],
        ),

    "end":
        find_likely_columns(
            prediction_evaluation_df,
            ["reconstructed_end"],
        ),
}


schema_check_rows = []

for structural_field in [
    "note_id",
    "start",
    "end",
]:

    schema_check_rows.append(
        {
            "structural_field":
                structural_field,

            "gold_candidates":
                " | ".join(
                    gold_structural_candidates[
                        structural_field
                    ]
                ),

            "prediction_candidates":
                " | ".join(
                    prediction_structural_candidates[
                        structural_field
                    ]
                ),

            "gold_found":
                bool(
                    gold_structural_candidates[
                        structural_field
                    ]
                ),

            "prediction_found":
                bool(
                    prediction_structural_candidates[
                        structural_field
                    ]
                ),
        }
    )


schema_check_df = pd.DataFrame(
    schema_check_rows
)

print("\nStructural schema check:")

display(
    schema_check_df
)


# 4A.8 Save schema audit

GOLD_SCHEMA_INVENTORY_FILE = (
    EVALUATION_DIRECTORY
    / "gold_evaluation_schema_inventory.csv"
)

PREDICTION_SCHEMA_INVENTORY_FILE = (
    EVALUATION_DIRECTORY
    / "prediction_evaluation_schema_inventory.csv"
)

LIKELY_EVALUATION_FIELDS_FILE = (
    EVALUATION_DIRECTORY
    / "likely_evaluation_fields.csv"
)

gold_column_inventory_df.to_csv(
    GOLD_SCHEMA_INVENTORY_FILE,
    index=False,
)

prediction_column_inventory_df.to_csv(
    PREDICTION_SCHEMA_INVENTORY_FILE,
    index=False,
)

likely_evaluation_fields_df.to_csv(
    LIKELY_EVALUATION_FIELDS_FILE,
    index=False,
)


# 4A.9 Conclusion

structural_schema_valid = bool(
    schema_check_df[
        [
            "gold_found",
            "prediction_found",
        ]
    ].all().all()
)

print("\n" + "-" * 80)
print("STEP 4A — SCHEMA CONFIRMATION CONCLUSION")
print("-" * 80)

if structural_schema_valid:

    print(
        "\nThe required note and offset fields were located."
    )

    print(
        "Confirm the exact category, assertion and temporality "
        "columns from the displayed inventories before "
        "constructing the one-to-one matcher."
    )

else:

    print(
        "\nOne or more required structural fields were not "
        "identified automatically."
    )

    print(
        "Inspect the displayed inventories before proceeding."
    )

print(
    f"\nGold schema inventory saved to:\n"
    f"{GOLD_SCHEMA_INVENTORY_FILE}"
)

print(
    f"\nPrediction schema inventory saved to:\n"
    f"{PREDICTION_SCHEMA_INVENTORY_FILE}"
)

In [ ]:
# 4B. Audit Upstream Prediction Attribute Availability

print("-" * 80)
print("STEP 4B — AUDIT UPSTREAM PREDICTION ATTRIBUTES")
print("-" * 80)


# 4B.1 Searching all current DataFrames containing candidate_id

candidate_dataframe_rows = []
candidate_dataframes = {}

for variable_name, variable_value in list(globals().items()):

    if not isinstance(variable_value, pd.DataFrame):
        continue

    if "candidate_id" not in variable_value.columns:
        continue

    candidate_dataframes[
        variable_name
    ] = variable_value

    candidate_dataframe_rows.append(
        {
            "dataframe_name":
                variable_name,

            "row_count":
                len(variable_value),

            "column_count":
                len(variable_value.columns),

            "unique_candidate_ids":
                variable_value[
                    "candidate_id"
                ].nunique(
                    dropna=True
                ),

            "duplicate_candidate_ids":
                int(
                    variable_value[
                        "candidate_id"
                    ].duplicated(
                        keep=False
                    ).sum()
                ),
        }
    )


candidate_dataframe_inventory_df = pd.DataFrame(
    candidate_dataframe_rows
)

if not candidate_dataframe_inventory_df.empty:

    candidate_dataframe_inventory_df = (
        candidate_dataframe_inventory_df.sort_values(
            [
                "unique_candidate_ids",
                "row_count",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


print("\nDataFrames containing candidate_id:")

if candidate_dataframe_inventory_df.empty:

    print("None found.")

else:

    display(
        candidate_dataframe_inventory_df
    )


# 4B.2 Identifying possible system attribute columns

attribute_search_terms = {
    "category": [
        "category",
        "complication",
        "entity_type",
        "concept",
        "label",
        "rule",
        "pattern",
    ],

    "assertion": [
        "assertion",
        "negation",
        "negated",
        "uncertain",
        "certainty",
        "status",
        "context",
    ],

    "temporality": [
        "temporality",
        "temporal",
        "historical",
        "history",
        "current",
        "recent",
    ],
}


def identify_attribute_columns(
    dataframe,
    terms,
):
    matching_columns = []

    for column in dataframe.columns:

        normalized_column = (
            str(column)
            .strip()
            .lower()
        )

        if any(
            term in normalized_column
            for term in terms
        ):
            matching_columns.append(
                column
            )

    return matching_columns


attribute_inventory_rows = []

for dataframe_name, dataframe in (
    candidate_dataframes.items()
):

    category_columns = identify_attribute_columns(
        dataframe,
        attribute_search_terms["category"],
    )

    assertion_columns = identify_attribute_columns(
        dataframe,
        attribute_search_terms["assertion"],
    )

    temporality_columns = identify_attribute_columns(
        dataframe,
        attribute_search_terms["temporality"],
    )

    attribute_inventory_rows.append(
        {
            "dataframe_name":
                dataframe_name,

            "row_count":
                len(dataframe),

            "category_candidates":
                " | ".join(
                    category_columns
                ),

            "assertion_candidates":
                " | ".join(
                    assertion_columns
                ),

            "temporality_candidates":
                " | ".join(
                    temporality_columns
                ),

            "contains_possible_category":
                bool(category_columns),

            "contains_possible_assertion":
                bool(assertion_columns),

            "contains_possible_temporality":
                bool(temporality_columns),
        }
    )


candidate_attribute_inventory_df = pd.DataFrame(
    attribute_inventory_rows
)

print("\nPossible system-generated attribute fields:")

if candidate_attribute_inventory_df.empty:

    print("No candidate DataFrames were available.")

else:

    display(
        candidate_attribute_inventory_df
    )


# 4B.3 Excluding manually reviewed or gold-derived columns

manual_or_gold_terms = [
    "gold",
    "reviewer",
    "manual",
    "annotation",
    "corrected",
    "final",
]

safe_attribute_rows = []

for dataframe_name, dataframe in (
    candidate_dataframes.items()
):

    for attribute_type, terms in (
        attribute_search_terms.items()
    ):

        possible_columns = (
            identify_attribute_columns(
                dataframe,
                terms,
            )
        )

        for column in possible_columns:

            normalized_column = (
                str(column)
                .strip()
                .lower()
            )

            appears_manual_or_gold = any(
                term in normalized_column
                for term in manual_or_gold_terms
            )

            safe_attribute_rows.append(
                {
                    "dataframe_name":
                        dataframe_name,

                    "attribute_type":
                        attribute_type,

                    "column_name":
                        column,

                    "dtype":
                        str(
                            dataframe[
                                column
                            ].dtype
                        ),

                    "non_missing_count":
                        int(
                            dataframe[
                                column
                            ].notna().sum()
                        ),

                    "unique_value_count":
                        int(
                            dataframe[
                                column
                            ].nunique(
                                dropna=True
                            )
                        ),

                    "appears_manual_or_gold":
                        appears_manual_or_gold,

                    "potentially_usable_system_field":
                        not appears_manual_or_gold,
                }
            )


safe_attribute_audit_df = pd.DataFrame(
    safe_attribute_rows
)

print("\nCandidate attribute safety audit:")

if safe_attribute_audit_df.empty:

    print("No possible attribute fields were found.")

else:

    display(
        safe_attribute_audit_df.sort_values(
            [
                "attribute_type",
                "potentially_usable_system_field",
                "dataframe_name",
                "column_name",
            ],
            ascending=[
                True,
                False,
                True,
                True,
            ],
        )
    )


# 4B.4 Displaying value counts for potentially usable fields

usable_attribute_rows_df = (
    safe_attribute_audit_df.loc[
        safe_attribute_audit_df[
            "potentially_usable_system_field"
        ]
    ]
    .copy()
    if not safe_attribute_audit_df.empty
    else pd.DataFrame()
)


if usable_attribute_rows_df.empty:

    print(
        "\nNo clearly system-generated category, assertion "
        "or temporality fields were found."
    )

else:

    print(
        "\nValue counts for potentially usable "
        "system-generated fields:"
    )

    for _, attribute_row in (
        usable_attribute_rows_df.iterrows()
    ):

        dataframe_name = attribute_row[
            "dataframe_name"
        ]

        column_name = attribute_row[
            "column_name"
        ]

        dataframe = candidate_dataframes[
            dataframe_name
        ]

        print(
            f"\n{dataframe_name} — "
            f"{attribute_row['attribute_type']} — "
            f"{column_name}"
        )

        display(
            dataframe[
                column_name
            ]
            .value_counts(
                dropna=False
            )
            .rename_axis(
                column_name
            )
            .reset_index(
                name="count"
            )
            .head(50)
        )


# 4B.5 Checking one-to-one join coverage with canonical predictions

join_coverage_rows = []

for dataframe_name, dataframe in (
    candidate_dataframes.items()
):

    unique_source = (
        dataframe[
            ["candidate_id"]
        ]
        .drop_duplicates()
    )

    canonical_ids = set(
        prediction_evaluation_df[
            "candidate_id"
        ].astype(str)
    )

    source_ids = set(
        unique_source[
            "candidate_id"
        ].astype(str)
    )

    join_coverage_rows.append(
        {
            "dataframe_name":
                dataframe_name,

            "canonical_prediction_count":
                len(canonical_ids),

            "canonical_ids_found":
                len(
                    canonical_ids
                    & source_ids
                ),

            "canonical_ids_missing":
                len(
                    canonical_ids
                    - source_ids
                ),

            "join_coverage_percentage":
                round(
                    100
                    * len(
                        canonical_ids
                        & source_ids
                    )
                    / len(canonical_ids),
                    2,
                ),
        }
    )


candidate_join_coverage_df = pd.DataFrame(
    join_coverage_rows
)

print("\nCandidate-ID join coverage:")

if candidate_join_coverage_df.empty:

    print("No candidate source tables available.")

else:

    display(
        candidate_join_coverage_df.sort_values(
            "join_coverage_percentage",
            ascending=False,
        )
    )


# 4B.6 Saving audit outputs


CANDIDATE_DATAFRAME_INVENTORY_FILE = (
    EVALUATION_DIRECTORY
    / "candidate_dataframe_inventory.csv"
)

CANDIDATE_ATTRIBUTE_INVENTORY_FILE = (
    EVALUATION_DIRECTORY
    / "candidate_attribute_inventory.csv"
)

CANDIDATE_ATTRIBUTE_SAFETY_FILE = (
    EVALUATION_DIRECTORY
    / "candidate_attribute_safety_audit.csv"
)

CANDIDATE_JOIN_COVERAGE_FILE = (
    EVALUATION_DIRECTORY
    / "candidate_attribute_join_coverage.csv"
)

candidate_dataframe_inventory_df.to_csv(
    CANDIDATE_DATAFRAME_INVENTORY_FILE,
    index=False,
)

candidate_attribute_inventory_df.to_csv(
    CANDIDATE_ATTRIBUTE_INVENTORY_FILE,
    index=False,
)

safe_attribute_audit_df.to_csv(
    CANDIDATE_ATTRIBUTE_SAFETY_FILE,
    index=False,
)

candidate_join_coverage_df.to_csv(
    CANDIDATE_JOIN_COVERAGE_FILE,
    index=False,
)


# 4B.7 Conclusion

print("\n" + "-" * 80)
print("STEP 4B — ATTRIBUTE AVAILABILITY CONCLUSION")
print("-" * 80)

if usable_attribute_rows_df.empty:

    print(
        "\nThe available prediction tables contain no clearly "
        "system-generated category, assertion or temporality "
        "outputs."
    )

    print(
        "Proceed with span-level mention detection evaluation only."
    )

    print(
        "Do not calculate category, assertion or temporality "
        "accuracy from gold or reviewer-derived fields."
    )

else:

    print(
        "\nPotential system-generated attribute fields were found."
    )

    print(
        "Their provenance and candidate-ID coverage must be "
        "confirmed before joining them to the canonical "
        "prediction table."
    )

## 4C. Enrichment of canonical predictions with native system attributes

The canonical prediction table contains the validated document-level spans used for
evaluation. The original extraction pipeline also produced native system predictions
for complication category, assertion, and temporality.

These prediction attributes are joined back into the canonical prediction table
using the immutable `candidate_id`. This preserves the original system outputs
without introducing any information from manual annotation or reviewer decisions.

In [ ]:
# 4C. Enriching Canonical Predictions with Native System Attributes

print("-" * 80)
print("STEP 4C — ENRICH CANONICAL PREDICTIONS")
print("-" * 80)


# 4C.1 Columns required from original prediction table

required_prediction_columns = [
    "candidate_id",
    "complication_category",
    "predicted_assertion",
    "predicted_temporality",
    "is_negated",
    "is_uncertain",
    "is_historical",
    "is_current_affirmed",
]

missing_columns = [
    c for c in required_prediction_columns
    if c not in system_predictions_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required prediction columns:\n{missing_columns}"
    )


# 4C.2 Building attribute table

prediction_attribute_df = (
    system_predictions_df[
        required_prediction_columns
    ]
    .copy()
    .drop_duplicates(subset=["candidate_id"])
)

if prediction_attribute_df["candidate_id"].duplicated().any():
    raise ValueError(
        "Duplicate candidate_id values found in attribute table."
    )


# 4C.3 Preserving canonical table

canonical_prediction_attributes_df = (
    canonical_system_predictions_df.copy()
)

expected_canonical_prediction_count = len(
    canonical_system_predictions_df
)


# 4C.4 Merge

canonical_prediction_attributes_df = (
    canonical_prediction_attributes_df.merge(
        prediction_attribute_df,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )
)


# 4C.5 Join validation

joined_attribute_columns = [
    "complication_category",
    "predicted_assertion",
    "predicted_temporality",
]

missing_attribute_rows = (
    canonical_prediction_attributes_df[
        joined_attribute_columns
    ]
    .isna()
    .any(axis=1)
)

missing_attribute_df = (
    canonical_prediction_attributes_df.loc[
        missing_attribute_rows
    ].copy()
)

join_summary_df = pd.DataFrame(
    [
        {
            "Measure": "Canonical predictions",
            "Value": len(canonical_prediction_attributes_df),
        },
        {
            "Measure": "Successfully enriched",
            "Value": int((~missing_attribute_rows).sum()),
        },
        {
            "Measure": "Missing joins",
            "Value": int(missing_attribute_rows.sum()),
        },
    ]
)

print("\nJoin summary:")

display(join_summary_df)


# 4C.6 Distribution checks

print("\nPredicted complication categories:")

display(
    canonical_prediction_attributes_df[
        "complication_category"
    ]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="count")
)

print("\nPredicted assertions:")

display(
    canonical_prediction_attributes_df[
        "predicted_assertion"
    ]
    .value_counts(dropna=False)
    .rename_axis("assertion")
    .reset_index(name="count")
)

print("\nPredicted temporality:")

display(
    canonical_prediction_attributes_df[
        "predicted_temporality"
    ]
    .value_counts(dropna=False)
    .rename_axis("temporality")
    .reset_index(name="count")
)


# 4C.7 Saving enriched prediction table

CANONICAL_ENRICHED_FILE = (
    EVALUATION_DIRECTORY
    / "canonical_predictions_enriched_546.csv"
)

canonical_prediction_attributes_df.to_csv(
    CANONICAL_ENRICHED_FILE,
    index=False,
)


# 4C.8 Final checks

if (
    len(canonical_prediction_attributes_df)
    != expected_canonical_prediction_count
):
    raise ValueError(
        "Unexpected canonical prediction count: "
        f"{len(canonical_prediction_attributes_df)} "
        f"instead of "
        f"{expected_canonical_prediction_count}."
    )

if missing_attribute_rows.any():
    raise ValueError(
        "Some canonical predictions were not enriched. "
        f"Missing joins: {int(missing_attribute_rows.sum())}."
    )


print("\n" + "-" * 80)
print("STEP 4C — ENRICHMENT CONCLUSION")
print("-" * 80)

print(
    f"\nCanonical predictions : "
    f"{len(canonical_prediction_attributes_df):,}"
)

print(
    f"Successfully enriched : "
    f"{int((~missing_attribute_rows).sum()):,}"
)

print(
    f"Missing joins         : "
    f"{int(missing_attribute_rows.sum()):,}"
)

print(
    "\nThe enriched canonical prediction table is now the "
    "official evaluation dataset."
)

print(
    f"\nSaved to:\n{CANONICAL_ENRICHED_FILE}"
)

## 5. Exact span mention matching

The enriched canonical prediction table is compared with the gold-standard
annotations using strict document-level span matching.

A prediction is considered a true positive only if:

- it belongs to the same note,
- its document start offset exactly matches the gold start offset,
- its document end offset exactly matches the gold end offset.

Each prediction and each gold annotation may participate in at most one match,
ensuring a one-to-one evaluation.

In [ ]:
# 5A. Exact One-to-One Mention Matching

print("-" * 80)
print("STEP 5A — EXACT SPAN MATCHING")
print("-" * 80)

import pandas as pd

# Gold evaluation table

gold_eval = gold_standard_df.copy()

gold_eval = gold_eval.rename(
    columns={
        "start_char": "gold_start",
        "end_char": "gold_end",
        "mention_text": "gold_text",
        "complication_category": "gold_category",
    }
)

# Prediction evaluation table

pred_eval = canonical_prediction_attributes_df.copy()

pred_eval = pred_eval.rename(
    columns={
        "reconstructed_start_char": "pred_start",
        "reconstructed_end_char": "pred_end",
        "entity_text": "pred_text",
    }
)


# Candidate exact matches

candidate_matches = gold_eval.merge(
    pred_eval,
    left_on=["note_id", "gold_start", "gold_end"],
    right_on=["note_id", "pred_start", "pred_end"],
    how="inner",
    suffixes=("_gold", "_pred"),
)

print(f"\nCandidate exact matches: {len(candidate_matches)}")


# One-to-one enforcement

candidate_matches = (
    candidate_matches
    .sort_values(
        ["note_id", "gold_start"]
    )
    .drop_duplicates(
        subset=["annotation_id"],
        keep="first"
    )
    .drop_duplicates(
        subset=["candidate_id"],
        keep="first"
    )
)

exact_match_df = candidate_matches.copy()

# True positives

TP = len(exact_match_df)

print(f"\nTrue positives: {TP}")

# False positives

matched_prediction_ids = set(
    exact_match_df["candidate_id"]
)

false_positive_df = pred_eval[
    ~pred_eval["candidate_id"].isin(
        matched_prediction_ids
    )
].copy()

FP = len(false_positive_df)

# False negatives

matched_gold_ids = set(
    exact_match_df["annotation_id"]
)

false_negative_df = gold_eval[
    ~gold_eval["annotation_id"].isin(
        matched_gold_ids
    )
].copy()

FN = len(false_negative_df)

# Metrics

precision = TP / (TP + FP) if TP + FP else 0

recall = TP / (TP + FN) if TP + FN else 0

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall
    else 0
)

metrics_df = pd.DataFrame(
    [
        {
            "Metric": "True Positives",
            "Value": TP,
        },
        {
            "Metric": "False Positives",
            "Value": FP,
        },
        {
            "Metric": "False Negatives",
            "Value": FN,
        },
        {
            "Metric": "Precision",
            "Value": round(precision,4),
        },
        {
            "Metric": "Recall",
            "Value": round(recall,4),
        },
        {
            "Metric": "F1-score",
            "Value": round(f1,4),
        },
    ]
)

print("\nExact-span evaluation:")

display(metrics_df)

# Integrity checks

assert TP + FN == len(gold_eval)

assert TP + FP == len(pred_eval)

assert exact_match_df["annotation_id"].duplicated().sum()==0

assert exact_match_df["candidate_id"].duplicated().sum()==0

# Save outputs

EXACT_MATCH_FILE = (
    EVALUATION_DIRECTORY /
    "exact_span_matches.csv"
)

FALSE_POSITIVE_FILE = (
    EVALUATION_DIRECTORY /
    "exact_span_false_positives.csv"
)

FALSE_NEGATIVE_FILE = (
    EVALUATION_DIRECTORY /
    "exact_span_false_negatives.csv"
)

METRICS_FILE = (
    EVALUATION_DIRECTORY /
    "exact_span_metrics.csv"
)

exact_match_df.to_csv(
    EXACT_MATCH_FILE,
    index=False,
)

false_positive_df.to_csv(
    FALSE_POSITIVE_FILE,
    index=False,
)

false_negative_df.to_csv(
    FALSE_NEGATIVE_FILE,
    index=False,
)

metrics_df.to_csv(
    METRICS_FILE,
    index=False,
)

# Conclusion

print("\n" + "-"*80)
print("STEP 5A — EXACT SPAN MATCHING COMPLETE")
print("-"*80)

print(f"\nGold mentions      : {len(gold_eval)}")
print(f"Predicted mentions : {len(pred_eval)}")
print(f"True positives     : {TP}")
print(f"False positives    : {FP}")
print(f"False negatives    : {FN}")

print("\nExact span matching completed successfully.")

print("\nOutputs written:")
print(EXACT_MATCH_FILE)
print(FALSE_POSITIVE_FILE)
print(FALSE_NEGATIVE_FILE)
print(METRICS_FILE)

## 5A.1 Boundary mismatch diagnostic

Exact span evaluation is intentionally strict and considers a prediction
correct only when its document boundaries exactly match the gold-standard
annotation.

This diagnostic compares false-positive predictions with false-negative
gold annotations within the same discharge summary to identify cases where
the predicted span overlaps the gold span. Such cases represent boundary
differences rather than complete extraction failures.

The diagnostic is used only for error analysis and does not modify the
exact-span evaluation results.

In [ ]:
# 5A.1 Boundary Mismatch Diagnostic

print("-" * 80)
print("STEP 5A.1 — BOUNDARY MISMATCH DIAGNOSTIC")
print("-" * 80)

import pandas as pd

boundary_records = []

for note_id in sorted(false_negative_df["note_id"].unique()):

    gold_note = false_negative_df[
        false_negative_df["note_id"] == note_id
    ]

    pred_note = false_positive_df[
        false_positive_df["note_id"] == note_id
    ]

    if gold_note.empty or pred_note.empty:
        continue

    for _, gold in gold_note.iterrows():

        for _, pred in pred_note.iterrows():

            # overlap test
            overlap = (
                pred["pred_start"] < gold["gold_end"]
                and
                gold["gold_start"] < pred["pred_end"]
            )

            if not overlap:
                continue

            overlap_start = max(
                pred["pred_start"],
                gold["gold_start"]
            )

            overlap_end = min(
                pred["pred_end"],
                gold["gold_end"]
            )

            overlap_length = overlap_end - overlap_start

            union_length = (
                max(pred["pred_end"], gold["gold_end"])
                -
                min(pred["pred_start"], gold["gold_start"])
            )

            iou = overlap_length / union_length

            boundary_records.append({

                "note_id": note_id,

                "annotation_id":
                    gold["annotation_id"],

                "candidate_id":
                    pred["candidate_id"],

                "gold_text":
                    gold["gold_text"],

                "prediction_text":
                    pred["pred_text"],

                "gold_start":
                    gold["gold_start"],

                "gold_end":
                    gold["gold_end"],

                "pred_start":
                    pred["pred_start"],

                "pred_end":
                    pred["pred_end"],

                "overlap_length":
                    overlap_length,

                "intersection_over_union":
                    round(iou,4)

            })

boundary_df = pd.DataFrame(boundary_records)

print()

print(f"Boundary overlap pairs found: {len(boundary_df)}")

if len(boundary_df):

    print()

    display(
        boundary_df.sort_values(
            "intersection_over_union",
            ascending=False
        ).head(20)
    )

BOUNDARY_FILE = (
    EVALUATION_DIRECTORY /
    "boundary_overlap_diagnostic.csv"
)

boundary_df.to_csv(
    BOUNDARY_FILE,
    index=False
)

print()

print("Diagnostic saved:")
print(BOUNDARY_FILE)

## 5A.2 False negative root cause analysis

This diagnostic investigates why gold-standard mentions were not matched during
exact span evaluation.

Each false-negative mention is compared against all canonical system
predictions within the same discharge summary.

The diagnostic determines whether the corresponding complication category was
detected elsewhere in the note or whether no prediction of that complication
exists.

This analysis is used to identify systematic causes of missed recall before
modifying the extraction pipeline.

In [ ]:
# 5A.2 False Negative Root Cause Analysis

print("-" * 80)
print("STEP 5A.2 — FALSE NEGATIVE ROOT CAUSE ANALYSIS")
print("-" * 80)

analysis_rows = []

for _, gold in false_negative_df.iterrows():

    note_predictions = canonical_prediction_attributes_df[
        canonical_prediction_attributes_df["note_id"] == gold["note_id"]
    ]

    same_category = note_predictions[
        note_predictions["complication_category"]
        == gold["gold_category"]
    ]

    if len(same_category):

        cause = "CATEGORY_PRESENT_ELSEWHERE"

    else:

        cause = "CATEGORY_NOT_DETECTED"

    analysis_rows.append({

        "note_id": gold["note_id"],

        "annotation_id": gold["annotation_id"],

        "gold_text": gold["gold_text"],

        "gold_category": gold["gold_category"],

        "cause": cause,

        "predictions_in_note": len(note_predictions),

        "same_category_predictions": len(same_category)

    })

fn_rootcause_df = pd.DataFrame(analysis_rows)

summary = (
    fn_rootcause_df["cause"]
    .value_counts()
    .rename_axis("Cause")
    .reset_index(name="Count")
)

print()

display(summary)

print()

display(
    fn_rootcause_df.head(20)
)

OUTPUT_FILE = (
    EVALUATION_DIRECTORY /
    "false_negative_root_cause_analysis.csv"
)

SUMMARY_FILE = (
    EVALUATION_DIRECTORY /
    "false_negative_root_cause_summary.csv"
)

fn_rootcause_df.to_csv(
    OUTPUT_FILE,
    index=False
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print()

print("Outputs written:")
print(OUTPUT_FILE)
print(SUMMARY_FILE)

## 5A.3 Lexicon gap analysis

False-negative mentions classified as CATEGORY_NOT_DETECTED represent
complication categories that were completely missed by the extraction
pipeline.

This diagnostic summarises the missed clinical expressions to identify
terminology gaps in the rule-based lexicon. The results guide targeted
expansion of the TargetMatcher patterns while avoiding unnecessary
modifications to existing rules.

In [ ]:
# 5A.3 Lexicon Gap Analysis

print("-" * 80)
print("STEP 5A.3 — LEXICON GAP ANALYSIS")
print("-" * 80)

import pandas as pd

# Keep only genuine category misses

lexicon_gap_df = fn_rootcause_df[
    fn_rootcause_df["cause"] == "CATEGORY_NOT_DETECTED"
].copy()

print(f"\nTotal genuine category misses: {len(lexicon_gap_df)}")

# Missing expressions

expression_summary = (
    lexicon_gap_df
    .groupby(
        ["gold_category", "gold_text"],
        dropna=False
    )
    .size()
    .reset_index(name="Frequency")
    .sort_values(
        ["Frequency", "gold_category", "gold_text"],
        ascending=[False, True, True]
    )
)

print("\nMost frequently missed expressions:\n")

display(expression_summary.head(50))


# Missing categories

category_summary = (
    lexicon_gap_df
    .groupby("gold_category")
    .size()
    .reset_index(name="Missed Mentions")
    .sort_values(
        "Missed Mentions",
        ascending=False
    )
)

print("\nMissed complication categories:\n")

display(category_summary)


# Notes contributing to misses


note_summary = (
    lexicon_gap_df
    .groupby("note_id")
    .size()
    .reset_index(name="Missed Mentions")
    .sort_values(
        "Missed Mentions",
        ascending=False
    )
)

print("\nNotes with most missed terminology:\n")

display(note_summary.head(20))

# Save

EXPR_FILE = (
    EVALUATION_DIRECTORY /
    "lexicon_gap_expressions.csv"
)

CATEGORY_FILE = (
    EVALUATION_DIRECTORY /
    "lexicon_gap_categories.csv"
)

NOTE_FILE = (
    EVALUATION_DIRECTORY /
    "lexicon_gap_notes.csv"
)

expression_summary.to_csv(EXPR_FILE, index=False)
category_summary.to_csv(CATEGORY_FILE, index=False)
note_summary.to_csv(NOTE_FILE, index=False)

print("\nOutputs written:")

print(EXPR_FILE)
print(CATEGORY_FILE)
print(NOTE_FILE)

# 5B–5H. Complete Evaluation

The following sections complete Notebook 07 by evaluating:

- category classification on exact-span matched mentions;
- assertion and temporality classification;
- strict end-to-end extraction;
- relaxed one-to-one overlap matching;
- per-category precision, recall and F1;
- consolidated final outputs and an evaluation report.

In [ ]:
# 5B. Exact-Span Category Classification

print("-" * 80)
print("STEP 5B — EXACT-SPAN CATEGORY CLASSIFICATION")
print("-" * 80)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

required_exact_columns = {
    "annotation_id",
    "candidate_id",
    "gold_category",
    "complication_category",
}

missing_exact_columns = (
    required_exact_columns
    - set(exact_match_df.columns)
)

if missing_exact_columns:
    raise ValueError(
        "Exact-match table is missing category columns: "
        + ", ".join(sorted(missing_exact_columns))
    )

exact_category_df = exact_match_df[
    [
        "annotation_id",
        "candidate_id",
        "note_id",
        "gold_text",
        "pred_text",
        "gold_category",
        "complication_category",
    ]
].copy()

exact_category_df = exact_category_df.rename(
    columns={
        "complication_category": "pred_category",
    }
)

exact_category_df["category_correct"] = (
    exact_category_df["gold_category"]
    == exact_category_df["pred_category"]
)

category_accuracy = accuracy_score(
    exact_category_df["gold_category"],
    exact_category_df["pred_category"],
)

category_macro_precision, category_macro_recall, category_macro_f1, _ = (
    precision_recall_fscore_support(
        exact_category_df["gold_category"],
        exact_category_df["pred_category"],
        average="macro",
        zero_division=0,
    )
)

category_weighted_precision, category_weighted_recall, category_weighted_f1, _ = (
    precision_recall_fscore_support(
        exact_category_df["gold_category"],
        exact_category_df["pred_category"],
        average="weighted",
        zero_division=0,
    )
)

category_classification_summary_df = pd.DataFrame(
    [
        {"Metric": "Matched exact spans", "Value": len(exact_category_df)},
        {
            "Metric": "Correct category labels",
            "Value": int(exact_category_df["category_correct"].sum()),
        },
        {
            "Metric": "Incorrect category labels",
            "Value": int((~exact_category_df["category_correct"]).sum()),
        },
        {"Metric": "Category accuracy", "Value": round(category_accuracy, 4)},
        {
            "Metric": "Macro precision",
            "Value": round(category_macro_precision, 4),
        },
        {
            "Metric": "Macro recall",
            "Value": round(category_macro_recall, 4),
        },
        {
            "Metric": "Macro F1",
            "Value": round(category_macro_f1, 4),
        },
        {
            "Metric": "Weighted F1",
            "Value": round(category_weighted_f1, 4),
        },
    ]
)

category_report_df = (
    pd.DataFrame(
        classification_report(
            exact_category_df["gold_category"],
            exact_category_df["pred_category"],
            output_dict=True,
            zero_division=0,
        )
    )
    .transpose()
    .reset_index()
    .rename(columns={"index": "category"})
)

category_labels = sorted(
    set(exact_category_df["gold_category"].dropna())
    | set(exact_category_df["pred_category"].dropna())
)

category_confusion_df = pd.DataFrame(
    confusion_matrix(
        exact_category_df["gold_category"],
        exact_category_df["pred_category"],
        labels=category_labels,
    ),
    index=[f"GOLD::{label}" for label in category_labels],
    columns=[f"PRED::{label}" for label in category_labels],
)

category_error_df = exact_category_df.loc[
    ~exact_category_df["category_correct"]
].copy()

display(category_classification_summary_df)

print("\nPer-category classification report:")
display(category_report_df)

print("\nCategory errors:")
if category_error_df.empty:
    print("None.")
else:
    display(category_error_df.head(30))

EXACT_CATEGORY_MATCH_FILE = (
    EVALUATION_DIRECTORY
    / "exact_span_category_classification.csv"
)

EXACT_CATEGORY_SUMMARY_FILE = (
    EVALUATION_DIRECTORY
    / "exact_span_category_summary.csv"
)

EXACT_CATEGORY_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "exact_span_category_report.csv"
)

EXACT_CATEGORY_CONFUSION_FILE = (
    EVALUATION_DIRECTORY
    / "exact_span_category_confusion_matrix.csv"
)

EXACT_CATEGORY_ERRORS_FILE = (
    EVALUATION_DIRECTORY
    / "exact_span_category_errors.csv"
)

exact_category_df.to_csv(EXACT_CATEGORY_MATCH_FILE, index=False)
category_classification_summary_df.to_csv(
    EXACT_CATEGORY_SUMMARY_FILE,
    index=False,
)
category_report_df.to_csv(EXACT_CATEGORY_REPORT_FILE, index=False)
category_confusion_df.to_csv(EXACT_CATEGORY_CONFUSION_FILE)
category_error_df.to_csv(EXACT_CATEGORY_ERRORS_FILE, index=False)

print("\nSTEP 5B complete.")


In [ ]:
# 5C. Assertion and Temporality Classification

print("-" * 80)
print("STEP 5C — CONTEXT ATTRIBUTE CLASSIFICATION")
print("-" * 80)


def evaluate_attribute(
    dataframe,
    gold_column,
    prediction_column,
    attribute_name,
):
    """
    Evaluate one categorical attribute on exact-span matched mentions.
    """

    required_columns = {
        "annotation_id",
        "candidate_id",
        gold_column,
        prediction_column,
    }

    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{attribute_name}: missing columns: "
            + ", ".join(sorted(missing_columns))
        )

    evaluation_df = dataframe[
        [
            "annotation_id",
            "candidate_id",
            "note_id",
            "gold_text",
            "pred_text",
            gold_column,
            prediction_column,
        ]
    ].copy()

    evaluation_df = evaluation_df.dropna(
        subset=[gold_column, prediction_column]
    )

    evaluation_df = evaluation_df.rename(
        columns={
            gold_column: "gold_label",
            prediction_column: "predicted_label",
        }
    )

    evaluation_df["correct"] = (
        evaluation_df["gold_label"]
        == evaluation_df["predicted_label"]
    )

    accuracy = accuracy_score(
        evaluation_df["gold_label"],
        evaluation_df["predicted_label"],
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            evaluation_df["gold_label"],
            evaluation_df["predicted_label"],
            average="macro",
            zero_division=0,
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            evaluation_df["gold_label"],
            evaluation_df["predicted_label"],
            average="weighted",
            zero_division=0,
        )
    )

    summary_df = pd.DataFrame(
        [
            {"Attribute": attribute_name, "Metric": "Evaluated mentions",
             "Value": len(evaluation_df)},
            {"Attribute": attribute_name, "Metric": "Correct labels",
             "Value": int(evaluation_df["correct"].sum())},
            {"Attribute": attribute_name, "Metric": "Incorrect labels",
             "Value": int((~evaluation_df["correct"]).sum())},
            {"Attribute": attribute_name, "Metric": "Accuracy",
             "Value": round(accuracy, 4)},
            {"Attribute": attribute_name, "Metric": "Macro precision",
             "Value": round(macro_precision, 4)},
            {"Attribute": attribute_name, "Metric": "Macro recall",
             "Value": round(macro_recall, 4)},
            {"Attribute": attribute_name, "Metric": "Macro F1",
             "Value": round(macro_f1, 4)},
            {"Attribute": attribute_name, "Metric": "Weighted F1",
             "Value": round(weighted_f1, 4)},
        ]
    )

    report_df = (
        pd.DataFrame(
            classification_report(
                evaluation_df["gold_label"],
                evaluation_df["predicted_label"],
                output_dict=True,
                zero_division=0,
            )
        )
        .transpose()
        .reset_index()
        .rename(columns={"index": "label"})
    )

    labels = sorted(
        set(evaluation_df["gold_label"].dropna())
        | set(evaluation_df["predicted_label"].dropna())
    )

    confusion_df = pd.DataFrame(
        confusion_matrix(
            evaluation_df["gold_label"],
            evaluation_df["predicted_label"],
            labels=labels,
        ),
        index=[f"GOLD::{label}" for label in labels],
        columns=[f"PRED::{label}" for label in labels],
    )

    error_df = evaluation_df.loc[
        ~evaluation_df["correct"]
    ].copy()

    return {
        "evaluation": evaluation_df,
        "summary": summary_df,
        "report": report_df,
        "confusion": confusion_df,
        "errors": error_df,
    }


assertion_results = evaluate_attribute(
    dataframe=exact_match_df,
    gold_column="assertion",
    prediction_column="predicted_assertion",
    attribute_name="ASSERTION",
)

temporality_results = evaluate_attribute(
    dataframe=exact_match_df,
    gold_column="temporality",
    prediction_column="predicted_temporality",
    attribute_name="TEMPORALITY",
)

context_attribute_summary_df = pd.concat(
    [
        assertion_results["summary"],
        temporality_results["summary"],
    ],
    ignore_index=True,
)

print("\nContext-attribute summary:")
display(context_attribute_summary_df)

print("\nAssertion report:")
display(assertion_results["report"])

print("\nTemporality report:")
display(temporality_results["report"])

ASSERTION_EVALUATION_FILE = (
    EVALUATION_DIRECTORY
    / "assertion_evaluation_exact_spans.csv"
)

ASSERTION_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "assertion_classification_report.csv"
)

ASSERTION_CONFUSION_FILE = (
    EVALUATION_DIRECTORY
    / "assertion_confusion_matrix.csv"
)

ASSERTION_ERRORS_FILE = (
    EVALUATION_DIRECTORY
    / "assertion_errors.csv"
)

TEMPORALITY_EVALUATION_FILE = (
    EVALUATION_DIRECTORY
    / "temporality_evaluation_exact_spans.csv"
)

TEMPORALITY_REPORT_FILE = (
    EVALUATION_DIRECTORY
    / "temporality_classification_report.csv"
)

TEMPORALITY_CONFUSION_FILE = (
    EVALUATION_DIRECTORY
    / "temporality_confusion_matrix.csv"
)

TEMPORALITY_ERRORS_FILE = (
    EVALUATION_DIRECTORY
    / "temporality_errors.csv"
)

CONTEXT_ATTRIBUTE_SUMMARY_FILE = (
    EVALUATION_DIRECTORY
    / "context_attribute_summary.csv"
)

assertion_results["evaluation"].to_csv(
    ASSERTION_EVALUATION_FILE,
    index=False,
)
assertion_results["report"].to_csv(
    ASSERTION_REPORT_FILE,
    index=False,
)
assertion_results["confusion"].to_csv(
    ASSERTION_CONFUSION_FILE,
)
assertion_results["errors"].to_csv(
    ASSERTION_ERRORS_FILE,
    index=False,
)

temporality_results["evaluation"].to_csv(
    TEMPORALITY_EVALUATION_FILE,
    index=False,
)
temporality_results["report"].to_csv(
    TEMPORALITY_REPORT_FILE,
    index=False,
)
temporality_results["confusion"].to_csv(
    TEMPORALITY_CONFUSION_FILE,
)
temporality_results["errors"].to_csv(
    TEMPORALITY_ERRORS_FILE,
    index=False,
)

context_attribute_summary_df.to_csv(
    CONTEXT_ATTRIBUTE_SUMMARY_FILE,
    index=False,
)

print("\nSTEP 5C complete.")


In [ ]:
# 5D. Strict End-to-End Exact-Span Evaluation

print("-" * 80)
print("STEP 5D — STRICT END-TO-END EXACT-SPAN EVALUATION")
print("-" * 80)


def extraction_metrics_from_correct_matches(
    correct_match_count,
    gold_count,
    prediction_count,
):
    """
    Calculate extraction precision, recall and F1 where only fully
    correct matched records count as true positives.
    """

    tp_value = int(correct_match_count)
    fp_value = int(prediction_count - tp_value)
    fn_value = int(gold_count - tp_value)

    precision_value = (
        tp_value / prediction_count
        if prediction_count
        else 0.0
    )

    recall_value = (
        tp_value / gold_count
        if gold_count
        else 0.0
    )

    f1_value = (
        2 * precision_value * recall_value
        / (precision_value + recall_value)
        if precision_value + recall_value
        else 0.0
    )

    return {
        "TP": tp_value,
        "FP": fp_value,
        "FN": fn_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "F1": f1_value,
    }


strict_exact_df = exact_match_df.copy()

strict_exact_df["category_correct"] = (
    strict_exact_df["gold_category"]
    == strict_exact_df["complication_category"]
)

strict_exact_df["assertion_correct"] = (
    strict_exact_df["assertion"]
    == strict_exact_df["predicted_assertion"]
)

strict_exact_df["temporality_correct"] = (
    strict_exact_df["temporality"]
    == strict_exact_df["predicted_temporality"]
)

strict_exact_df["category_assertion_correct"] = (
    strict_exact_df["category_correct"]
    & strict_exact_df["assertion_correct"]
)

strict_exact_df["all_attributes_correct"] = (
    strict_exact_df["category_correct"]
    & strict_exact_df["assertion_correct"]
    & strict_exact_df["temporality_correct"]
)

strict_variants = {
    "Exact span only":
        len(strict_exact_df),

    "Exact span + category":
        int(strict_exact_df["category_correct"].sum()),

    "Exact span + category + assertion":
        int(strict_exact_df["category_assertion_correct"].sum()),

    "Exact span + category + assertion + temporality":
        int(strict_exact_df["all_attributes_correct"].sum()),
}

strict_rows = []

for evaluation_name, correct_count in strict_variants.items():

    result = extraction_metrics_from_correct_matches(
        correct_match_count=correct_count,
        gold_count=len(gold_eval),
        prediction_count=len(pred_eval),
    )

    strict_rows.append(
        {
            "Evaluation": evaluation_name,
            "True Positives": result["TP"],
            "False Positives": result["FP"],
            "False Negatives": result["FN"],
            "Precision": round(result["Precision"], 4),
            "Recall": round(result["Recall"], 4),
            "F1-score": round(result["F1"], 4),
        }
    )

strict_end_to_end_metrics_df = pd.DataFrame(strict_rows)

display(strict_end_to_end_metrics_df)

STRICT_EXACT_MATCH_DETAIL_FILE = (
    EVALUATION_DIRECTORY
    / "strict_exact_match_attribute_detail.csv"
)

STRICT_END_TO_END_METRICS_FILE = (
    EVALUATION_DIRECTORY
    / "strict_end_to_end_metrics.csv"
)

strict_exact_df.to_csv(
    STRICT_EXACT_MATCH_DETAIL_FILE,
    index=False,
)

strict_end_to_end_metrics_df.to_csv(
    STRICT_END_TO_END_METRICS_FILE,
    index=False,
)

print("\nSTEP 5D complete.")


In [ ]:
# 5E. Relaxed One-to-One Overlap Evaluation

print("-" * 80)
print("STEP 5E — RELAXED ONE-TO-ONE OVERLAP EVALUATION")
print("-" * 80)

RELAXED_IOU_THRESHOLD = 0.0


def create_overlap_candidates(
    gold_dataframe,
    prediction_dataframe,
    require_category_match=False,
):
    """
    Create all within-note overlapping gold-prediction pairs.
    """

    records = []

    for note_id, gold_note_df in gold_dataframe.groupby("note_id"):

        prediction_note_df = prediction_dataframe.loc[
            prediction_dataframe["note_id"] == note_id
        ]

        if prediction_note_df.empty:
            continue

        for _, gold_row in gold_note_df.iterrows():

            for _, prediction_row in prediction_note_df.iterrows():

                if (
                    require_category_match
                    and gold_row["gold_category"]
                    != prediction_row["complication_category"]
                ):
                    continue

                gold_start = int(gold_row["gold_start"])
                gold_end = int(gold_row["gold_end"])
                pred_start = int(prediction_row["pred_start"])
                pred_end = int(prediction_row["pred_end"])

                overlap_start = max(gold_start, pred_start)
                overlap_end = min(gold_end, pred_end)
                overlap_length = max(0, overlap_end - overlap_start)

                if overlap_length <= 0:
                    continue

                union_start = min(gold_start, pred_start)
                union_end = max(gold_end, pred_end)
                union_length = union_end - union_start

                iou = (
                    overlap_length / union_length
                    if union_length
                    else 0.0
                )

                if iou <= RELAXED_IOU_THRESHOLD:
                    continue

                records.append(
                    {
                        "note_id": note_id,
                        "annotation_id": gold_row["annotation_id"],
                        "candidate_id": prediction_row["candidate_id"],
                        "gold_text": gold_row["gold_text"],
                        "pred_text": prediction_row["pred_text"],
                        "gold_category": gold_row["gold_category"],
                        "pred_category":
                            prediction_row["complication_category"],
                        "gold_start": gold_start,
                        "gold_end": gold_end,
                        "pred_start": pred_start,
                        "pred_end": pred_end,
                        "overlap_length": overlap_length,
                        "intersection_over_union": iou,
                        "category_correct": (
                            gold_row["gold_category"]
                            == prediction_row["complication_category"]
                        ),
                    }
                )

    return pd.DataFrame(records)


def greedy_one_to_one_overlap(overlap_candidate_df):
    """
    Select non-conflicting overlap pairs by descending IoU and overlap.
    """

    if overlap_candidate_df.empty:
        return overlap_candidate_df.copy()

    ordered_df = overlap_candidate_df.sort_values(
        [
            "intersection_over_union",
            "overlap_length",
            "note_id",
            "gold_start",
            "pred_start",
        ],
        ascending=[False, False, True, True, True],
    )

    used_gold_ids = set()
    used_prediction_ids = set()
    selected_indices = []

    for index, row in ordered_df.iterrows():

        annotation_id = row["annotation_id"]
        candidate_id = row["candidate_id"]

        if annotation_id in used_gold_ids:
            continue

        if candidate_id in used_prediction_ids:
            continue

        selected_indices.append(index)
        used_gold_ids.add(annotation_id)
        used_prediction_ids.add(candidate_id)

    return (
        ordered_df.loc[selected_indices]
        .copy()
        .sort_values(
            ["note_id", "gold_start", "pred_start"]
        )
        .reset_index(drop=True)
    )


relaxed_span_candidates_df = create_overlap_candidates(
    gold_dataframe=gold_eval,
    prediction_dataframe=pred_eval,
    require_category_match=False,
)

relaxed_span_match_df = greedy_one_to_one_overlap(
    relaxed_span_candidates_df
)

relaxed_category_candidates_df = create_overlap_candidates(
    gold_dataframe=gold_eval,
    prediction_dataframe=pred_eval,
    require_category_match=True,
)

relaxed_category_match_df = greedy_one_to_one_overlap(
    relaxed_category_candidates_df
)

relaxed_span_result = extraction_metrics_from_correct_matches(
    correct_match_count=len(relaxed_span_match_df),
    gold_count=len(gold_eval),
    prediction_count=len(pred_eval),
)

relaxed_category_result = extraction_metrics_from_correct_matches(
    correct_match_count=len(relaxed_category_match_df),
    gold_count=len(gold_eval),
    prediction_count=len(pred_eval),
)

relaxed_metrics_df = pd.DataFrame(
    [
        {
            "Evaluation": "Relaxed overlap span only",
            "True Positives": relaxed_span_result["TP"],
            "False Positives": relaxed_span_result["FP"],
            "False Negatives": relaxed_span_result["FN"],
            "Precision": round(relaxed_span_result["Precision"], 4),
            "Recall": round(relaxed_span_result["Recall"], 4),
            "F1-score": round(relaxed_span_result["F1"], 4),
        },
        {
            "Evaluation": "Relaxed overlap + category",
            "True Positives": relaxed_category_result["TP"],
            "False Positives": relaxed_category_result["FP"],
            "False Negatives": relaxed_category_result["FN"],
            "Precision": round(relaxed_category_result["Precision"], 4),
            "Recall": round(relaxed_category_result["Recall"], 4),
            "F1-score": round(relaxed_category_result["F1"], 4),
        },
    ]
)

display(relaxed_metrics_df)

print("\nHighest-IoU relaxed category-aware matches:")
display(
    relaxed_category_match_df.sort_values(
        "intersection_over_union",
        ascending=False,
    ).head(30)
)

RELAXED_SPAN_MATCH_FILE = (
    EVALUATION_DIRECTORY
    / "relaxed_overlap_span_matches.csv"
)

RELAXED_CATEGORY_MATCH_FILE = (
    EVALUATION_DIRECTORY
    / "relaxed_overlap_category_matches.csv"
)

RELAXED_METRICS_FILE = (
    EVALUATION_DIRECTORY
    / "relaxed_overlap_metrics.csv"
)

relaxed_span_match_df.to_csv(
    RELAXED_SPAN_MATCH_FILE,
    index=False,
)

relaxed_category_match_df.to_csv(
    RELAXED_CATEGORY_MATCH_FILE,
    index=False,
)

relaxed_metrics_df.to_csv(
    RELAXED_METRICS_FILE,
    index=False,
)

print("\nSTEP 5E complete.")


In [ ]:
# 5F. Per-Category Exact End-to-End Metrics

print("-" * 80)
print("STEP 5F — PER-CATEGORY EXACT END-TO-END METRICS")
print("-" * 80)

correct_exact_category_df = strict_exact_df.loc[
    strict_exact_df["category_correct"]
].copy()

all_categories = sorted(
    set(gold_eval["gold_category"].dropna())
    | set(pred_eval["complication_category"].dropna())
)

per_category_rows = []

for category in all_categories:

    gold_count = int(
        (gold_eval["gold_category"] == category).sum()
    )

    prediction_count = int(
        (
            pred_eval["complication_category"]
            == category
        ).sum()
    )

    true_positive_count = int(
        (
            correct_exact_category_df["gold_category"]
            == category
        ).sum()
    )

    false_positive_count = (
        prediction_count - true_positive_count
    )

    false_negative_count = (
        gold_count - true_positive_count
    )

    category_precision = (
        true_positive_count / prediction_count
        if prediction_count
        else 0.0
    )

    category_recall = (
        true_positive_count / gold_count
        if gold_count
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall
        else 0.0
    )

    per_category_rows.append(
        {
            "Category": category,
            "Gold Mentions": gold_count,
            "Predicted Mentions": prediction_count,
            "True Positives": true_positive_count,
            "False Positives": false_positive_count,
            "False Negatives": false_negative_count,
            "Precision": round(category_precision, 4),
            "Recall": round(category_recall, 4),
            "F1-score": round(category_f1, 4),
        }
    )

per_category_exact_metrics_df = (
    pd.DataFrame(per_category_rows)
    .sort_values(
        ["Gold Mentions", "Category"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(per_category_exact_metrics_df)

supported_category_df = per_category_exact_metrics_df.loc[
    per_category_exact_metrics_df["Gold Mentions"] > 0
].copy()

macro_category_precision = (
    supported_category_df["Precision"].mean()
)

macro_category_recall = (
    supported_category_df["Recall"].mean()
)

macro_category_f1 = (
    supported_category_df["F1-score"].mean()
)

per_category_macro_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Number of gold-supported categories",
            "Value": len(supported_category_df),
        },
        {
            "Metric": "Macro exact end-to-end precision",
            "Value": round(macro_category_precision, 4),
        },
        {
            "Metric": "Macro exact end-to-end recall",
            "Value": round(macro_category_recall, 4),
        },
        {
            "Metric": "Macro exact end-to-end F1",
            "Value": round(macro_category_f1, 4),
        },
    ]
)

print("\nMacro category summary:")
display(per_category_macro_summary_df)

PER_CATEGORY_EXACT_METRICS_FILE = (
    EVALUATION_DIRECTORY
    / "per_category_exact_end_to_end_metrics.csv"
)

PER_CATEGORY_MACRO_SUMMARY_FILE = (
    EVALUATION_DIRECTORY
    / "per_category_exact_macro_summary.csv"
)

per_category_exact_metrics_df.to_csv(
    PER_CATEGORY_EXACT_METRICS_FILE,
    index=False,
)

per_category_macro_summary_df.to_csv(
    PER_CATEGORY_MACRO_SUMMARY_FILE,
    index=False,
)

print("\nSTEP 5F complete.")


In [ ]:
# 5G. Consolidated Evaluation Summary

print("-" * 80)
print("STEP 5G — CONSOLIDATED EVALUATION SUMMARY")
print("-" * 80)


def summary_value(summary_df, metric_name):
    matching_rows = summary_df.loc[
        summary_df["Metric"] == metric_name,
        "Value",
    ]

    if matching_rows.empty:
        return np.nan

    return matching_rows.iloc[0]


exact_span_row = strict_end_to_end_metrics_df.loc[
    strict_end_to_end_metrics_df["Evaluation"]
    == "Exact span only"
].iloc[0]

exact_category_row = strict_end_to_end_metrics_df.loc[
    strict_end_to_end_metrics_df["Evaluation"]
    == "Exact span + category"
].iloc[0]

strict_all_row = strict_end_to_end_metrics_df.loc[
    strict_end_to_end_metrics_df["Evaluation"]
    == "Exact span + category + assertion + temporality"
].iloc[0]

relaxed_category_row = relaxed_metrics_df.loc[
    relaxed_metrics_df["Evaluation"]
    == "Relaxed overlap + category"
].iloc[0]

final_evaluation_summary_df = pd.DataFrame(
    [
        {
            "Evaluation Component": "Corpus",
            "Metric": "Reviewed notes",
            "Value": int(
                note_register_df["note_id"].nunique()
            ),
        },
        {
            "Evaluation Component": "Corpus",
            "Metric": "Gold mentions",
            "Value": len(gold_eval),
        },
        {
            "Evaluation Component": "System",
            "Metric": "Canonical predictions",
            "Value": len(pred_eval),
        },
        {
            "Evaluation Component": "Exact span detection",
            "Metric": "Precision",
            "Value": exact_span_row["Precision"],
        },
        {
            "Evaluation Component": "Exact span detection",
            "Metric": "Recall",
            "Value": exact_span_row["Recall"],
        },
        {
            "Evaluation Component": "Exact span detection",
            "Metric": "F1-score",
            "Value": exact_span_row["F1-score"],
        },
        {
            "Evaluation Component": "Exact span + category",
            "Metric": "Precision",
            "Value": exact_category_row["Precision"],
        },
        {
            "Evaluation Component": "Exact span + category",
            "Metric": "Recall",
            "Value": exact_category_row["Recall"],
        },
        {
            "Evaluation Component": "Exact span + category",
            "Metric": "F1-score",
            "Value": exact_category_row["F1-score"],
        },
        {
            "Evaluation Component": "Relaxed overlap + category",
            "Metric": "Precision",
            "Value": relaxed_category_row["Precision"],
        },
        {
            "Evaluation Component": "Relaxed overlap + category",
            "Metric": "Recall",
            "Value": relaxed_category_row["Recall"],
        },
        {
            "Evaluation Component": "Relaxed overlap + category",
            "Metric": "F1-score",
            "Value": relaxed_category_row["F1-score"],
        },
        {
            "Evaluation Component": "Category classification",
            "Metric": "Accuracy on exact-span matches",
            "Value": round(category_accuracy, 4),
        },
        {
            "Evaluation Component": "Category classification",
            "Metric": "Macro F1 on exact-span matches",
            "Value": round(category_macro_f1, 4),
        },
        {
            "Evaluation Component": "Assertion classification",
            "Metric": "Accuracy on exact-span matches",
            "Value": summary_value(
                assertion_results["summary"],
                "Accuracy",
            ),
        },
        {
            "Evaluation Component": "Assertion classification",
            "Metric": "Macro F1 on exact-span matches",
            "Value": summary_value(
                assertion_results["summary"],
                "Macro F1",
            ),
        },
        {
            "Evaluation Component": "Temporality classification",
            "Metric": "Accuracy on exact-span matches",
            "Value": summary_value(
                temporality_results["summary"],
                "Accuracy",
            ),
        },
        {
            "Evaluation Component": "Temporality classification",
            "Metric": "Macro F1 on exact-span matches",
            "Value": summary_value(
                temporality_results["summary"],
                "Macro F1",
            ),
        },
        {
            "Evaluation Component": "Strict full extraction",
            "Metric": "F1-score",
            "Value": strict_all_row["F1-score"],
        },
    ]
)

display(final_evaluation_summary_df)

FINAL_EVALUATION_SUMMARY_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_final_evaluation_summary.csv"
)

final_evaluation_summary_df.to_csv(
    FINAL_EVALUATION_SUMMARY_FILE,
    index=False,
)

print("\nSTEP 5G complete.")


In [ ]:
# 5H. Final Validation and Evaluation Report

print("-" * 80)
print("STEP 5H — FINAL NOTEBOOK 07 VALIDATION")
print("-" * 80)

final_validation_checks = {
    "registered_notes_equal_70":
        int(note_register_df["note_id"].nunique()) == 70,

    "gold_offsets_all_valid":
        bool(gold_standard_df["valid_offsets"].all()),

    "gold_text_all_valid":
        bool(gold_standard_df["case_insensitive_match"].all()),

    "canonical_prediction_count_preserved":
        len(pred_eval)
        == len(canonical_system_predictions_df),

    "canonical_prediction_ids_unique":
        not pred_eval["candidate_id"].duplicated().any(),

    "prediction_offsets_all_valid":
        bool(
            collision_resolved_df[
                "post_collision_text_match"
            ].all()
        ),

    "no_remaining_span_collisions":
        remaining_canonical_collisions_df.empty,

    "no_remaining_order_violations":
        remaining_order_violations_df.empty,

    "attribute_join_complete":
        not missing_attribute_rows.any(),

    "exact_matching_integrity":
        (
            TP + FN == len(gold_eval)
            and TP + FP == len(pred_eval)
        ),

    "relaxed_span_matches_one_to_one":
        (
            not relaxed_span_match_df[
                "annotation_id"
            ].duplicated().any()
            and not relaxed_span_match_df[
                "candidate_id"
            ].duplicated().any()
        ),

    "relaxed_category_matches_one_to_one":
        (
            not relaxed_category_match_df[
                "annotation_id"
            ].duplicated().any()
            and not relaxed_category_match_df[
                "candidate_id"
            ].duplicated().any()
        ),
}

final_validation_df = pd.DataFrame(
    [
        {
            "Check": check_name,
            "Passed": bool(check_result),
        }
        for check_name, check_result
        in final_validation_checks.items()
    ]
)

display(final_validation_df)

failed_final_checks = final_validation_df.loc[
    ~final_validation_df["Passed"]
].copy()

if not failed_final_checks.empty:
    raise ValueError(
        "Notebook 07 final validation failed:\n"
        + "\n".join(
            failed_final_checks["Check"].tolist()
        )
    )

evaluation_report = {
    "notebook": "07",
    "title": "Clinical NLP System Evaluation",
    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "corpus": {
        "reviewed_notes":
            int(note_register_df["note_id"].nunique()),
        "gold_mentions":
            int(len(gold_eval)),
        "canonical_predictions":
            int(len(pred_eval)),
    },

    "exact_span_detection": {
        "true_positives": int(TP),
        "false_positives": int(FP),
        "false_negatives": int(FN),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },

    "exact_span_category": {
        key: (
            int(value)
            if key in {
                "True Positives",
                "False Positives",
                "False Negatives",
            }
            else float(value)
        )
        for key, value
        in exact_category_row.to_dict().items()
        if key != "Evaluation"
    },

    "relaxed_overlap_category": {
        key: (
            int(value)
            if key in {
                "True Positives",
                "False Positives",
                "False Negatives",
            }
            else float(value)
        )
        for key, value
        in relaxed_category_row.to_dict().items()
        if key != "Evaluation"
    },

    "classification_on_exact_span_matches": {
        "category_accuracy":
            float(category_accuracy),
        "category_macro_f1":
            float(category_macro_f1),

        "assertion_accuracy":
            float(
                summary_value(
                    assertion_results["summary"],
                    "Accuracy",
                )
            ),
        "assertion_macro_f1":
            float(
                summary_value(
                    assertion_results["summary"],
                    "Macro F1",
                )
            ),

        "temporality_accuracy":
            float(
                summary_value(
                    temporality_results["summary"],
                    "Accuracy",
                )
            ),
        "temporality_macro_f1":
            float(
                summary_value(
                    temporality_results["summary"],
                    "Macro F1",
                )
            ),
    },

    "strict_full_extraction": {
        key: (
            int(value)
            if key in {
                "True Positives",
                "False Positives",
                "False Negatives",
            }
            else float(value)
        )
        for key, value
        in strict_all_row.to_dict().items()
        if key != "Evaluation"
    },

    "boundary_diagnostic": {
        "overlap_pairs":
            int(len(boundary_df)),
    },

    "validation": {
        key: bool(value)
        for key, value
        in final_validation_checks.items()
    },

    "output_directory":
        str(EVALUATION_DIRECTORY),
}

with open(
    EVALUATION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:

    json.dump(
        evaluation_report,
        report_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

FINAL_VALIDATION_FILE = (
    EVALUATION_DIRECTORY
    / "notebook_07_final_validation.csv"
)

final_validation_df.to_csv(
    FINAL_VALIDATION_FILE,
    index=False,
)

print("\n" + "-" * 80)
print("EVALUATION COMPLETE")
print("-" * 80)

print(
    f"\nReviewed notes       : "
    f"{int(note_register_df['note_id'].nunique()):,}"
)

print(
    f"Gold mentions        : "
    f"{len(gold_eval):,}"
)

print(
    f"Canonical predictions: "
    f"{len(pred_eval):,}"
)

print(
    f"\nExact span F1        : "
    f"{float(exact_span_row['F1-score']):.4f}"
)

print(
    f"Exact span + category F1: "
    f"{float(exact_category_row['F1-score']):.4f}"
)

print(
    f"Relaxed + category F1: "
    f"{float(relaxed_category_row['F1-score']):.4f}"
)

print(
    f"Strict full extraction F1: "
    f"{float(strict_all_row['F1-score']):.4f}"
)

print(
    "\nAll final integrity checks passed."
)

print(
    "\nFinal summary saved to:\n"
    f"{FINAL_EVALUATION_SUMMARY_FILE}"
)

print(
    "\nFinal report saved to:\n"
    f"{EVALUATION_REPORT_FILE}"
)
